In [ ]:
!uv pip install magika langchain-core langchain-openai langchain-text-splitters tree-sitter-language-pack python-frontmatter pydantic langgraph ast-grep-cli &> /dev/null

In [ ]:
import os
import pathlib as plb
import subprocess
from collections.abc import Generator
from functools import cached_property

from magika import Magika
from pydantic import BaseModel
from tree_sitter_language_pack import DownloadError, get_parser


In [ ]:
# The repository to audit; AUDIT_ROOT points it at another one. The gold set and drift test are kmctl's.
root_path = plb.Path(os.environ.get("AUDIT_ROOT", "/Users/jasperginn/Documents/code_projects/kmctl"))
REPO = root_path.name
path = root_path / ".okf"

## File readers

In [84]:
class FileMetadataModel(BaseModel):
    lang_name: str
    description: str
    is_text: bool
    score: float


class FileWithMedatadataModel(BaseModel):
    content: bytes
    metadata: FileMetadataModel
    path: plb.Path

    @cached_property
    def content_str(self):
        return self.content.decode("utf-8")

In [85]:
class FileReader:
    def __init__(self):
        self._language_predictor = Magika()

    @staticmethod
    def _read_bytes(file_path: plb.Path) -> bytes:
        return file_path.read_bytes()

    def _get_file_metadata(self, content: bytes) -> FileMetadataModel:
        metadata = self._language_predictor.identify_bytes(content)
        return FileMetadataModel(
            lang_name=metadata.output.label,
            description=metadata.output.description,
            is_text=metadata.output.is_text,
            score=metadata.score,
        )

    def read_file(self, file_path: plb.Path) -> FileWithMedatadataModel:
        content = self._read_bytes(file_path)
        metadata = self._get_file_metadata(content)
        return FileWithMedatadataModel(
            content=content, metadata=metadata, path=file_path
        )

    def read_files(
        self, file_paths: Generator[plb.Path, None, None]
    ) -> Generator[FileWithMedatadataModel, None, None]:
        for file_path in file_paths:
            yield self.read_file(file_path)

In [86]:
class ExclusionFileReader:
    def __init__(
        self,
        reader: FileReader,
        excluded_directories: list[str],
        included_extensions: list[str],
    ):
        self._reader = reader
        self._excluded_directories = set(excluded_directories)
        self._included_extensions = set(included_extensions)

    def _create_git_cmd(self, path: plb.Path) -> list[str]:
        cmd = [
            "git",
            "-C",
            str(path),
            "ls-files",
            "--cached",
            "--others",
            "--exclude-standard",
        ]
        cmd.append("--")
        cmd.append(".")
        for dir in self._excluded_directories:
            cmd.append(f":(exclude,glob)**/{dir}/**")
        return cmd

    def _run_git_cmd(self, cmd: list[str]) -> subprocess.CompletedProcess[str]:
        return subprocess.run(cmd, capture_output=True, text=True, check=True)

    def _parse_git_cmd_stdout(
        self, subprocess_result: subprocess.CompletedProcess[str]
    ) -> list[str]:
        return subprocess_result.stdout.splitlines()

    def _filter_file_paths(
        self, root_path: plb.Path, file_paths: list[str]
    ) -> Generator[plb.Path, None, None]:
        for line in file_paths:
            path = root_path / line
            if path.suffix in self._included_extensions and path.is_file():
                yield path

    def read_files(
        self, path: plb.Path
    ) -> Generator[FileWithMedatadataModel, None, None]:
        file_paths_from_git = self._parse_git_cmd_stdout(
            self._run_git_cmd(self._create_git_cmd(path=path))
        )
        file_paths_filtered = self._filter_file_paths(
            root_path=path, file_paths=file_paths_from_git
        )
        return self._reader.read_files(file_paths_filtered)

Exclusion file reader ignores any paths also ignored by git, and user can then additionally ignore folders they don't need parsed. Included extensions should be set explicitly.

In [ ]:
_reader = FileReader()
included_extensions = [
    ".py",
    ".toml",
    ".yaml",
    ".yml",
    ".jinja",
    ".sh",
    ".bash",
    ".md",
    ".json",
    # Common in other repositories: infrastructure, templates and other languages.
    ".hcl", ".tf", ".nomad", ".tpl", ".j2", ".go", ".js", ".mjs", ".ts", ".tsx", ".rs", ".java", ".rb",
    ".ini", ".cfg", ".conf", ".lua", ".sql",
]
excluded_directories = [".loop", ".claude", ".vscode", ".okf", "docs", ".claude"]

reader = ExclusionFileReader(
    reader=_reader,
    excluded_directories=excluded_directories,
    included_extensions=included_extensions,
)

files = [f for f in reader.read_files(root_path)]

We can retrieve required parsers and re-use them instead of setting up new ones for each parsing operation.

In [88]:
extensions = list({f.metadata.lang_name for f in files})

parsers = {}
for ext in extensions:
    try:
        parsers[ext] = get_parser(ext)
    except DownloadError:
        print(f"No parser for {ext}")

No parser for txt
No parser for jinja


In [89]:
parsers

{'yaml': <tree_sitter.Parser at 0x1177245d0>,
 'shell': <tree_sitter.Parser at 0x11aecb750>,
 'json': <tree_sitter.Parser at 0x11ed03690>,
 'python': <tree_sitter.Parser at 0x11af237e0>,
 'toml': <tree_sitter.Parser at 0x11af23630>,
 'markdown': <tree_sitter.Parser at 0x11ee9b5d0>}

Parsing files then becomes quite straightforward

In [90]:
files_parsed = []
for f in files:
    lang_name = f.metadata.lang_name
    if lang_name not in parsers:
        continue
    else:
        _parser = parsers[lang_name]
        files_parsed.append(_parser.parse(f.content))

In [91]:
len(files_parsed)

138

## Chunking the bundle

Each concept becomes one chunk per header section. A chunk has:

- a stable id: the concept id plus its header path, so edits elsewhere in the file leave it alone;
- numbered sentences, which the model cites by number instead of quoting;
- a key: a hash of everything the model sees (header path, sentences, the record's `Correction` and `Amended` sections, concept type). An unchanged key means a cached result is still good.

In [ ]:
import asyncio
import hashlib
import json
import operator
import os
import random
import re
from collections import Counter
from typing import Annotated, Any, Literal, cast

import frontmatter
from langchain_core.messages import AnyMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from typing_extensions import TypedDict
from pydantic import ConfigDict, Field

In [93]:
SKIP_NAMES = {"index.md", "log.md"}
SKIP_DIRS = {"prototypes"}
HEADERS = [("#", "h1"), ("##", "h2"), ("###", "h3")]
CORRECTION_PREFIXES = ("Correction", "Amended")
_SENTENCE_END = re.compile(r"(?<=[.!?])\s+(?=[A-Z`\[(*\"'/])")
_FENCE = re.compile(r"(```.*?```)", re.DOTALL)
_BULLET = re.compile(r"^\s*(?:[-*+]|\d+[.)])\s+")
_BLANK_LINE = re.compile(r"\n\s*\n")
_NOT_SLUG = re.compile(r"[^a-z0-9]+")

In [ ]:
class Chunk(BaseModel):
    """One header section of one concept, as the extractor sees it.

    Attributes
    ----------
    id : str
        The concept id plus the slug of its header path.
    concept : str
        The concept id: its path in the bundle without ``.md``.
    type : str
        The concept's ``type`` front-matter field.
    status : str | None
        The concept's ``status`` front-matter field.
    title : str
        The concept's ``title`` front-matter field.
    section : str
        The header path, joined with `` > ``.
    sentences : list[str]
        The units a claim can cite, in order.
    corrections : str
        The record's correction sections, shown as context; empty for a
        correction section itself.
    key : str
        A hash of everything the extractor sees.
    earlier : str
        The document's earlier sections, each sentence labelled ``[s.n]``
        (section s, sentence n), so references can be resolved.
    earlier_ids : list[str]
        The chunk id of each earlier section s, from 1.
    """

    id: str
    concept: str
    type: str
    status: str | None
    title: str
    section: str
    sentences: list[str]
    corrections: str
    key: str
    earlier: str = ""
    earlier_ids: list[str] = []

    @property
    def cache_key(self) -> str:
        """The key plus a hash of the earlier sections: the extractor sees both, the gold set only the key."""
        return self.key + ("-" + hashlib.sha256(self.earlier.encode()).hexdigest()[:8] if self.earlier else "")

    def numbered_sentences(self) -> str:
        """Return the sentences one per line, each prefixed with ``[n]`` from 1."""
        lines = []
        for number, sentence in enumerate(self.sentences, 1):
            lines.append(f"[{number}] {sentence}")
        return "\n".join(lines)

In [95]:
class Section(BaseModel):
    """One header section of a concept's body.

    Attributes
    ----------
    name : str
        The header path, joined with `` > ``.
    body : str
        The text under that header.
    """

    name: str
    body: str

    @property
    def last_header(self) -> str:
        """The innermost header of the section."""
        return self.name.rsplit(" > ", 1)[-1]

    @property
    def is_correction(self) -> bool:
        """Whether the section is a correction or amendment to the record."""
        return self.last_header.startswith(CORRECTION_PREFIXES)

In [96]:
class SentenceSplitter:
    """Split a section's text into the units a claim can cite.

    A code block or table is one unit, a list item is split by sentence, and
    a fragment of two words or fewer (a list number, a bold label) joins the
    sentence after it.
    """

    def split(self, text: str) -> list[str]:
        """Split a section into units to cite.

        Parameters
        ----------
        text :
            The section's body.

        Returns
        -------
        list[str]
            The non-empty units, in order.
        """
        units: list[str] = []
        parts = _FENCE.split(text)
        for index, part in enumerate(parts):
            is_code_block = index % 2 == 1
            if is_code_block:
                units.append(part.strip())
            else:
                units.extend(self._split_prose(part))
        return self._drop_empty(units)

    def _split_prose(self, text: str) -> list[str]:
        """Split text outside code blocks into units, block by block."""
        units: list[str] = []
        for block in _BLANK_LINE.split(text.strip()):
            units.extend(self._split_block(block.splitlines()))
        return units

    def _split_block(self, lines: list[str]) -> list[str]:
        """Split one blank-line-separated block: its tables first, then its items' sentences."""
        tables, items = self._collect_tables_and_items(lines)
        units = list(tables)
        for item in items:
            units.extend(self._split_item(item))
        return units

    @staticmethod
    def _collect_tables_and_items(lines: list[str]) -> tuple[list[str], list[str]]:
        """Group a block's lines into tables and list items.

        A line that starts a list item, or the first non-table line, opens an
        item; any other line continues the last item.
        """
        tables: list[str] = []
        items: list[str] = []
        table: list[str] = []
        for line in lines:
            if line.lstrip().startswith("|"):
                table.append(line.strip())
                continue
            if table:
                tables.append("\n".join(table))
                table = []
            if _BULLET.match(line) or not items:
                items.append(line.strip())
            else:
                items[-1] += " " + line.strip()
        if table:
            tables.append("\n".join(table))
        return tables, items

    @staticmethod
    def _split_item(item: str) -> list[str]:
        """Split one list item into sentences, joining short fragments to the next one.

        An item that is not split and needs no stripping stays whole, however
        short it is.
        """
        units: list[str] = []
        pending = ""
        for part in _SENTENCE_END.split(item):
            if pending:
                sentence = (pending + " " + part).strip()
            else:
                sentence = part.strip()
            is_whole_item = sentence is item
            is_fragment = len(sentence.split()) <= 2
            if is_fragment and not is_whole_item:
                pending = sentence
                continue
            pending = ""
            if sentence:
                units.append(sentence)
        if pending:
            units.append(pending)
        return units

    @staticmethod
    def _drop_empty(units: list[str]) -> list[str]:
        """Remove empty units."""
        kept = []
        for unit in units:
            if unit:
                kept.append(unit)
        return kept

In [ ]:
class BundleChunker:
    """Turn every live concept in an OKF bundle into chunks.

    Skips the reserved ``index.md`` and ``log.md``, the ``prototypes``
    folder and deprecated concepts.

    Parameters
    ----------
    splitter :
        Splits each section's body into the sentences a claim can cite.
    """

    def __init__(self, splitter: SentenceSplitter) -> None:
        self._splitter = splitter
        self._header_splitter = MarkdownHeaderTextSplitter(HEADERS)

    def chunks(self, bundle: plb.Path) -> Generator[Chunk, None, None]:
        """Yield the chunks of every live concept, in path order.

        Parameters
        ----------
        bundle :
            The bundle's root folder.

        Yields
        ------
        Chunk
            One chunk per non-empty header section.
        """
        for path in self._concept_paths(bundle):
            post = frontmatter.loads(path.read_text())
            if post.get("status") == "deprecated":
                continue
            yield from self._concept_chunks(bundle, path, post)

    @staticmethod
    def _concept_paths(bundle: plb.Path) -> Generator[plb.Path, None, None]:
        """Yield the bundle's concept files in sorted order, minus reserved names and skipped folders."""
        for path in sorted(bundle.rglob("*.md")):
            relative = path.relative_to(bundle)
            if path.name in SKIP_NAMES:
                continue
            if SKIP_DIRS & set(relative.parts):
                continue
            yield path

    def _concept_chunks(
        self, bundle: plb.Path, path: plb.Path, post: frontmatter.Post
    ) -> Generator[Chunk, None, None]:
        """Yield one chunk per section of a concept that holds any sentence."""
        concept = path.relative_to(bundle).with_suffix("").as_posix()
        self._ids_so_far: list[str] = []
        sections = self._sections(post)
        corrections = self._corrections(sections)
        seen: dict[str, int] = {}
        earlier: list[tuple[str, list[str]]] = []
        for section in sections:
            sentences = self._splitter.split(section.body)
            if not sentences:
                continue
            if section.is_correction:
                context = ""
            else:
                context = corrections
            chunk_id = self._chunk_id(concept, section, seen)
            before = self._earlier(earlier)
            if not section.is_correction:
                earlier.append((section.name, sentences))
            yield Chunk(
                id=chunk_id,
                concept=concept,
                type=str(post.get("type", "")),
                status=cast(str | None, post.get("status")),
                title=str(post.get("title", "")),
                section=section.name,
                sentences=sentences,
                corrections=context,
                key=self._key(section, sentences, context, post),
                earlier=before,
                earlier_ids=list(self._ids_so_far),
            )
            if not section.is_correction:
                self._ids_so_far.append(chunk_id)

    EARLIER_SENTENCES = 40

    def _earlier(self, earlier: list[tuple[str, list[str]]]) -> str:
        """The earlier sections' sentences, labelled [s.n], keeping the last EARLIER_SENTENCES."""
        lines = []
        for s, (name, sentences) in enumerate(earlier, 1):
            lines.append(f"{name}:")
            lines.extend(f"[{s}.{n}] {sentence}" for n, sentence in enumerate(sentences, 1))
        kept = [l for l in lines if l.startswith("[")][-self.EARLIER_SENTENCES:]
        return "\n".join(l for l in lines if not l.startswith("[") or l in kept)

    def _sections(self, post: frontmatter.Post) -> list[Section]:
        """Split a concept's body by its headers."""
        sections = []
        for document in self._header_splitter.split_text(post.content):
            name = " > ".join(document.metadata.values())
            sections.append(Section(name=name, body=document.page_content))
        return sections

    @staticmethod
    def _corrections(sections: list[Section]) -> str:
        """Join a record's correction sections, each headed by its header path."""
        parts = []
        for section in sections:
            if section.is_correction:
                parts.append(f"{section.name}\n{section.body}")
        return "\n\n".join(parts)

    @staticmethod
    def _chunk_id(concept: str, section: Section, seen: dict[str, int]) -> str:
        """Build a chunk id from the concept and its header path below the top header.

        A repeated slug gets a ``-2``, ``-3`` suffix; ``seen`` counts each
        slug and is updated in place.
        """
        below_top = section.name.split(" > ", 1)[-1]
        slug = _NOT_SLUG.sub("-", below_top.lower()).strip("-") or "top"
        seen[slug] = seen.get(slug, 0) + 1
        if seen[slug] > 1:
            return f"{concept}#{slug}-{seen[slug]}"
        return f"{concept}#{slug}"

    @staticmethod
    def _key(
        section: Section, sentences: list[str], context: str, post: frontmatter.Post
    ) -> str:
        """Hash the header path, sentences, corrections and concept type into a 16-character key."""
        content = [section.name, sentences, context, str(post.get("type"))]
        digest = hashlib.sha256(json.dumps(content).encode())
        return digest.hexdigest()[:16]

In [98]:
def read_chunks(path: plb.Path) -> Generator[Chunk, None, None]:
    """Yield the chunks stored one JSON object per line in ``path``."""
    for line in path.read_text().splitlines():
        yield Chunk.model_validate_json(line)

In [ ]:
chunker = BundleChunker(splitter=SentenceSplitter())
all_chunks = [c.model_dump() for c in chunker.chunks(path)]
if REPO != "kmctl" and os.environ.get("AUDIT_SAMPLE"):  # a sample of concepts, to try another repository cheaply
    concepts = sorted({c["concept"] for c in all_chunks})
    keep = set(random.Random(0).sample(concepts, min(int(os.environ["AUDIT_SAMPLE"]), len(concepts))))
    all_chunks = [c for c in all_chunks if c["concept"] in keep]
chunks_by_id = {c["id"]: c for c in all_chunks}
if REPO == "kmctl":  # the gold set reads kmctl's chunks
    plb.Path("chunks.jsonl").write_text("".join(json.dumps(c) + "\n" for c in all_chunks))
len(all_chunks)

## Gold set

26 chunks sampled with a fixed seed across decision sections, corrections, the manifesto, the architecture page and the summary. `LABELS` gives each sentence the claim types it supports; a label that includes `none` is optional. The labels are claude-opus/5.5's and no person has checked them yet.

`gold.jsonl` stores the labels with the key each chunk had when it was labelled, so a chunk that changed since shows up as stale. Set `WRITE_GOLD` after relabelling.

In [100]:
class GoldRecord(BaseModel):
    """The hand labels of one gold chunk.

    Attributes
    ----------
    id : str
        The chunk id.
    key : str
        The chunk's key when it was labelled; a different key means it changed.
    labels : dict[str, list[str]]
        Per sentence number, every claim type a careful reader could defend.
        A sentence left out supports no claim; one whose types include
        ``none`` is optional.
    """

    id: str
    key: str
    labels: dict[str, list[str]]

    def sentence_labels(self) -> dict[int, set[str]]:
        """Return the allowed types per sentence number."""
        labels = {}
        for number, types in self.labels.items():
            labels[int(number)] = set(types)
        return labels

    def required_sentences(self) -> set[int]:
        """Return the numbers of the sentences a model must cite."""
        required = set()
        for number, types in self.sentence_labels().items():
            if "none" not in types:
                required.add(number)
        return required

In [101]:
SEED = 20261105
QUOTA = {
    "decision": 6,
    "context": 4,
    "consequences": 4,
    "correction": 3,
    "manifesto": 4,
    "architecture": 3,
    "summary": 2,
}

In [102]:
class GoldSampler:
    """Sample chunk ids per kind of section, with a fixed seed.

    Parameters
    ----------
    seed :
        Seeds the random sample, so the same chunks give the same ids.
    quota :
        How many chunks to take from each stratum, in sampling order.
    """

    def __init__(self, seed: int, quota: dict[str, int]) -> None:
        self._seed = seed
        self._quota = quota

    def sample(self, chunks: list[Chunk]) -> Generator[str, None, None]:
        """Yield the sampled chunk ids, stratum by stratum, sorted within each.

        Parameters
        ----------
        chunks :
            Every chunk to sample from.

        Yields
        ------
        str
            One sampled chunk id.
        """
        rng = random.Random(self._seed)
        for name, size in self._quota.items():
            pool = self._pool(chunks, name)
            picked = rng.sample(pool, min(size, len(pool)))
            yield from sorted(picked)

    def _pool(self, chunks: list[Chunk], name: str) -> list[str]:
        """Return the sorted ids of the chunks in one stratum."""
        ids = []
        for chunk in chunks:
            if self._stratum(chunk) == name:
                ids.append(chunk.id)
        return sorted(ids)

    @staticmethod
    def _stratum(chunk: Chunk) -> str:
        """Name the kind of section a chunk is, for sampling.

        A decision record's section is named by its last header, with every
        correction or amendment grouped as ``correction``; any other chunk is
        named by its concept's top folder.
        """
        last_header = chunk.section.rsplit(" > ", 1)[-1].lower()
        if not chunk.concept.startswith("decisions/"):
            return chunk.concept.split("/")[0]
        if last_header.startswith(("correction", "amended")):
            return "correction"
        return last_header

In [103]:
CODES = {
    "P": "point",
    "S": "structural",
    "J": "subjective",
    "E": "external",
    "N": "none",
}
LABELS: dict[str, dict[int, str]] = {
    "decisions/0006-staleness-from-okf-fields#decision": {1: "P"},
    "decisions/0007-skills-are-made-with-skill-creator#decision": {1: "J", 2: "S"},
    "decisions/0026-the-kmctl-on-path-runs-the-release-a-project-s-lock-pins#decision": {
        1: "P",
        2: "P",
        3: "P",
        4: "P",
        6: "P",
        7: "P",
        8: "P",
        9: "P",
        10: "P",
    },
    "decisions/0029-the-session-briefing-is-kmctl-s-only-hook-again#decision": {
        1: "S",
        2: "P",
        3: "P",
        4: "S",
        5: "PS",
        6: "SN",
    },
    "decisions/0033-kmctl-check-fails-on-a-missing-link-target-inside-the-bundle#decision": {
        1: "P",
        2: "P",
        3: "P",
        4: "P",
        5: "PN",
        9: "SN",
    },
    "decisions/0035-kmctl-s-repository-answers-and-triages-with-hermes#decision": {
        1: "P",
        2: "P",
        3: "P",
        4: "P",
        5: "P",
        6: "P",
        7: "P",
        8: "P",
        9: "PS",
    },
    "decisions/0008-handoffs-are-okf-concepts#context": {1: "EN", 2: "EN", 3: "EPN"},
    "decisions/0020-apm-installs-the-hooks-mcp-server-and-rules-kmctl-patches-the-rest#context": {
        5: "E",
        6: "E",
        7: "EP",
        8: "E",
        9: "ES",
        10: "E",
        11: "ES",
        12: "EN",
    },
    "decisions/0026-the-kmctl-on-path-runs-the-release-a-project-s-lock-pins#context": {
        1: "PS",
        2: "PS",
        5: "EN",
        6: "EN",
    },
    "decisions/0027-the-bundle-opens-with-a-summary-that-kmctl-keeps-in-step#context": {
        4: "JN"
    },
    "decisions/0008-handoffs-are-okf-concepts#correction-2026-10-15-consequences": {
        1: "P",
        2: "PS",
        3: "P",
        4: "PN",
    },
    "decisions/0014-apm-is-the-only-install-route#consequences": {
        1: "S",
        2: "PS",
        3: "P",
        4: "E",
        5: "PN",
        6: "P",
    },
    "decisions/0020-apm-installs-the-hooks-mcp-server-and-rules-kmctl-patches-the-rest#consequences": {
        1: "SE",
        2: "PN",
        3: "PS",
    },
    "decisions/0035-kmctl-s-repository-answers-and-triages-with-hermes#consequences": {
        1: "S",
        2: "JN",
        3: "P",
        4: "PJ",
        5: "PE",
        6: "P",
        7: "JN",
        8: "PS",
    },
    "decisions/0012-releases-come-from-ci#correction-2026-10-17": {1: "PS"},
    "decisions/0017-the-ci-audit-is-a-standalone-pydanticai-agent-script#correction-2026-10-17": {
        1: "P",
        2: "P",
        3: "EN",
        4: "P",
        5: "EN",
        6: "P",
        7: "P",
        8: "P",
        9: "P",
        11: "P",
        12: "P",
        13: "P",
        14: "P",
        15: "PN",
        16: "P",
        17: "PN",
        18: "P",
        19: "P",
        20: "P",
        21: "P",
        22: "P",
        24: "P",
        25: "JN",
    },
    "decisions/0027-the-bundle-opens-with-a-summary-that-kmctl-keeps-in-step#correction-2026-10-19": {
        3: "P",
        5: "P",
        6: "P",
    },
    "manifesto#invariants": {
        1: "SJ",
        2: "P",
        3: "P",
        4: "P",
        5: "S",
        7: "P",
        8: "P",
        9: "S",
        10: "S",
        11: "S",
        12: "P",
    },
    "manifesto#manifesto-kmctl": {1: "JN", 2: "JN", 4: "PS", 5: "JN", 6: "PS", 7: "SJ"},
    "manifesto#values": {1: "J", 2: "J", 3: "J", 4: "J", 5: "J"},
    "manifesto#where-knowledge-lives": {1: "S", 2: "S", 3: "JN"},
    "architecture/overview#architecture-overview": {
        1: "SN",
        2: "S",
        3: "S",
        4: "PS",
        5: "S",
        6: "PS",
        7: "P",
        8: "P",
        9: "PS",
        10: "SP",
    },
    "architecture/overview#releases": {
        1: "P",
        2: "P",
        3: "P",
        4: "P",
        5: "P",
        6: "P",
        7: "P",
        8: "SP",
        9: "P",
        10: "P",
    },
    "architecture/overview#the-package-and-its-install": {
        1: "PS",
        2: "P",
        3: "P",
        4: "SN",
        5: "PS",
        6: "P",
        7: "EN",
        8: "SP",
        9: "P",
        10: "P",
        11: "S",
        12: "P",
        13: "P",
        14: "P",
        15: "P",
        16: "P",
        17: "SP",
        18: "P",
        19: "P",
        20: "PS",
    },
    "summary#how-it-works": {
        1: "SN",
        2: "S",
        3: "S",
        4: "PS",
        5: "P",
        6: "S",
        7: "P",
        8: "P",
        9: "PS",
    },
    "summary#where-it-stands": {1: "SJN", 2: "PS", 3: "S", 4: "SP"},
}

In [104]:
class GoldLabeler:
    """Turn the hand labels into gold records keyed by each chunk's current key.

    Parameters
    ----------
    labels :
        Per chunk id, the type codes of each labelled sentence number.
    codes :
        The claim type each one-letter code stands for.
    """

    def __init__(
        self, labels: dict[str, dict[int, str]], codes: dict[str, str]
    ) -> None:
        self._labels = labels
        self._codes = codes

    def records(self, chunks: dict[str, Chunk]) -> Generator[GoldRecord, None, None]:
        """Yield one gold record per labelled chunk, in label order.

        Parameters
        ----------
        chunks :
            Every chunk, by id.

        Yields
        ------
        GoldRecord
            The labels of one chunk, keyed so a changed chunk is noticed.

        Raises
        ------
        KeyError
            If a labelled chunk id is not among ``chunks``.
        AssertionError
            If a label names a sentence number the chunk does not have.
        """
        for chunk_id, labels in self._labels.items():
            chunk = chunks[chunk_id]
            self._check_sentence_numbers(chunk, labels)
            yield GoldRecord(id=chunk_id, key=chunk.key, labels=self._expand(labels))

    @staticmethod
    def _check_sentence_numbers(chunk: Chunk, labels: dict[int, str]) -> None:
        """Fail when a label names a sentence number outside the chunk."""
        count = len(chunk.sentences)
        for number in labels:
            assert 1 <= number <= count, (chunk.id, count)

    def _expand(self, labels: dict[int, str]) -> dict[str, list[str]]:
        """Turn each sentence's codes into its sorted type names, by sentence number."""
        expanded = {}
        for number, codes in sorted(labels.items()):
            types = []
            for code in codes:
                types.append(self._codes[code])
            expanded[str(number)] = sorted(types)
        return expanded

In [105]:
sampled = list(GoldSampler(seed=SEED, quota=QUOTA).sample([Chunk.model_validate(c) for c in all_chunks]))
plb.Path("gold_ids.txt").write_text("".join(i + "\n" for i in sampled))
assert set(sampled) == set(LABELS), "the sample changed: relabel"

WRITE_GOLD = False
if WRITE_GOLD:
    labeler = GoldLabeler(labels=LABELS, codes=CODES)
    records = labeler.records({c["id"]: Chunk.model_validate(c) for c in all_chunks})
    plb.Path("gold.jsonl").write_text("".join(json.dumps(r.model_dump()) + "\n" for r in records))

In [ ]:
gold = [json.loads(line) for line in open("gold.jsonl")]
gold_ids = {g["id"] for g in gold}
gold_chunks = [c for c in all_chunks if c["id"] in gold_ids]
stale = [g["id"] for g in gold if g["key"] != next(c["key"] for c in gold_chunks if c["id"] == g["id"])]
len(gold_chunks), stale  # a stale chunk changed since it was labelled: relabel it

## Extract claims: a verdict per sentence

The model goes through every numbered sentence and gives it a kind (`point`, `structural`, `subjective`, `external`, `none`), filling `how_to_check` before the kind. Results are cached per chunk at `results/cache/<model>/<prompt hash>/<chunk key>.json`, so rerunning only calls the model for chunks that changed, and a changed prompt starts an empty cache.

Needs `BIFROST_CC_VIRTUAL_KEY`.

In [ ]:
MAX_CONCURRENT_CALLS = 12
MODEL = "gemini/gemini-3.5-flash-lite"
SYSTEM_PROMPT = """
You read one section of a design document from a software repository and
decide, sentence by sentence, which claims it makes about that repository.

Why: an auditor checks each claim against the code and reports the ones that
no longer hold. A claim it cannot check wastes its time; a claim you miss is
drift nobody sees.

Go through every numbered sentence in order and give each one a verdict,
including sentences that hold no claim. Do not stop early.

Kinds, by how a claim is checked:
- point: look at one named thing in this repository (function, command,
  flag, config key, file, workflow, test).
- structural: check many files for a pattern ("every X does Y", "nothing
  imports Z", "only the CLI writes").
- subjective: a principle or value a person must judge; no file settles it.
- external: about a tool or service outside this repository.
- broad: about this repository, but too sweeping for any one file to settle
  ("the tool covers every task a user needs", "the system has three
  layers"). A later step decides whether code can settle it.
- none: no claim. Status, dates, authors; history ("came out of", "used
  to", "was"); measurements from a past run; the reasons for a choice; the
  alternatives a decision rejected; what an operator asked for; headings.

Give every point or structural claim a quantifier, by how many places in
this repository settle it:
- exists: one place proves it; find the function, file, flag, key or step
  that does it.
- all: every instance of something must comply ("every concept has a type",
  "each workflow pins its actions"); you must find every instance.
- none: no instance may exist anywhere ("never writes outside .okf/",
  "nothing imports the CLI", "only the CLI writes X"); you must search
  everywhere.
Judge the meaning, not the words: "the hook stays silent when the bundle is
clean" is exists, because one hook settles it.

Fill how_to_check before you choose the kind: if you cannot name what to open
or run in this repository, the kind is subjective, external or none.

A sentence can hold several claims; list each one. Write each so it stands
alone: replace "it", "this", "the code", "both commits" and "the tool" with
the names they refer to, from this section or the earlier sections shown. Use
the present tense. In refers_to, list the labels of the sentences a claim
needs to be understood: [s.n] for an earlier section, [n] for this one; leave
it empty when the claim stands alone.

Earlier sections of the document are shown for context only: give no
verdicts for them.

If the record has later corrections, they are shown for context only: give
no verdicts for them, and mark a sentence they overturn as none.
""".strip()

In [ ]:
class ClaimText(BaseModel):
    """One claim and how many places settle it."""

    claim: str
    quantifier: Literal["exists", "all", "none"] = Field(
        description="exists: one place proves it; all: every instance must comply; none: no instance may exist."
    )
    refers_to: list[str] = Field(
        default=[], description="Labels of the sentences this claim needs to be understood, like [2.4] or [3]."
    )


class SentenceVerdict(BaseModel):
    """The claims one numbered sentence makes, with the check reasoned out before the kind."""

    n: int = Field(description="The sentence number.")
    how_to_check: str = Field(
        description="What to open or run in this repository to confirm it; empty if nothing would."
    )
    kind: Literal["point", "structural", "broad", "subjective", "external", "none"]
    claims: list[ClaimText] = Field(
        description="Each claim the sentence makes, standalone and in the present tense; empty for none."
    )

In [109]:
class Verdicts(BaseModel):
    """One verdict per numbered sentence, in order."""

    sentences: list[SentenceVerdict]

In [110]:
class TokenCounts(BaseModel):
    """Tokens one model call used."""

    model_config = ConfigDict(populate_by_name=True)

    input: int = Field(alias="in", description="Tokens sent to the model.")
    output: int = Field(alias="out", description="Tokens the model returned.")

In [ ]:
class Claim(BaseModel):
    """One claim from a sentence verdict.

    Attributes
    ----------
    evidence : list[int]
        The number of the sentence the verdict is for.
    claim_type : str
        The verdict's kind; never ``none``.
    claim : str
        The claim; empty when the verdict listed none.
    how_to_check : str
        What the model would open or run to confirm it.
    quantifier : str
        ``exists``, ``all`` or ``none``: how many places settle the claim.
    refers_to : list[str]
        Other sentences the claim needs, as ``chunk id|n``.
    """

    evidence: list[int]
    claim_type: str
    claim: str
    how_to_check: str
    quantifier: Literal["exists", "all", "none"] = "exists"
    refers_to: list[str] = []

In [112]:
class ChunkResult(BaseModel):
    """One chunk's extraction: what is cached, plus whether it came from the cache.

    Attributes
    ----------
    id : str
        The chunk id.
    key : str
        The chunk's content key.
    claims : list[Claim]
        The claims.
    error : str | None
        Why the model's answer could not be read, if it could not.
    tokens : TokenCounts
        Tokens the call used.
    cached : bool
        Whether the result came from the cache; not stored in it.
    """

    id: str
    key: str
    claims: list[Claim]
    error: str | None
    tokens: TokenCounts
    cached: bool = False

    def predicted_types(self) -> dict[int, set[str]]:
        """Return the claim types each cited sentence number was cited for."""
        predicted: dict[int, set[str]] = {}
        for claim in self.claims:
            for number in claim.evidence:
                predicted.setdefault(number, set()).add(claim.claim_type)
        return predicted

    def to_json(self) -> str:
        """Return the result as one JSON line, as a results file holds it."""
        return json.dumps(self.model_dump(mode="json", by_alias=True))

    def to_cache_json(self) -> str:
        """Return the result as the cache stores it, without the ``cached`` flag."""
        return json.dumps(self.model_dump(mode="json", by_alias=True, exclude={"cached"}))

In [ ]:
# Flash-Lite sometimes skips a run of sentences despite the prompt; the sentences it skipped are asked again.
FILL_NOTE = "You gave no verdict for some sentences. Give a verdict for each of these sentences, and only these: {}"


class ExtractionState(TypedDict):
    chunk: Chunk
    output: dict[str, Any]
    result: ChunkResult


class ClaimExtractor:
    """A two-node graph: ask the model for a verdict per sentence, then turn verdicts into claims.

    Parameters
    ----------
    llm :
        The chat model to send each chunk to.
    """

    def __init__(self, llm: ChatOpenAI) -> None:
        self._structured = llm.with_structured_output(
            Verdicts, method="function_calling", include_raw=True
        )
        self.graph = self._build()

    def _build(self):
        builder = StateGraph(ExtractionState)
        builder.add_node("verdicts", self._verdicts)
        builder.add_node("claims", self._claims)
        builder.add_edge(START, "verdicts")
        builder.add_edge("verdicts", "claims")
        builder.add_edge("claims", END)
        return builder.compile()

    @staticmethod
    def user_message(chunk: Chunk) -> str:
        """Build the user message with the concept's context, numbered sentences and any corrections."""
        status = chunk.status or "n/a"
        parts = [
            f"Concept: {chunk.concept} (type: {chunk.type}, status: {status})",
            f"Title: {chunk.title}",
            f"Section: {chunk.section}",
        ]
        if chunk.earlier:
            parts += ["", "Earlier sections of this document (context only):", chunk.earlier]
        parts += ["", "This section:", chunk.numbered_sentences()]
        if chunk.corrections:
            parts.append("")
            parts.append("Later corrections in this record (context only):")
            parts.append(chunk.corrections)
        return "\n".join(parts)

    async def _verdicts(self, state: ExtractionState) -> dict:
        """Node: ask the model for a verdict on every numbered sentence."""
        chunk = state["chunk"]
        message = self.user_message(chunk)
        output = await self._structured.ainvoke([SystemMessage(SYSTEM_PROMPT), HumanMessage(message)])
        parsed = output["parsed"]
        if parsed is None:
            return {"output": output}
        missing = sorted(set(range(1, len(chunk.sentences) + 1)) - {v.n for v in parsed.sentences})
        if missing:
            ask = HumanMessage(message + "\n\n" + FILL_NOTE.format(", ".join(f"[{n}]" for n in missing)))
            more = await self._structured.ainvoke([SystemMessage(SYSTEM_PROMPT), ask])
            if more["parsed"] is not None:
                filled = [v for v in more["parsed"].sentences if v.n in missing]
                output = {**output, "parsed": Verdicts(sentences=parsed.sentences + filled), "filled": len(missing)}
        return {"output": output}

    def _claims(self, state: ExtractionState) -> dict:
        """Node: one claim per claim in each verdict that is not none and names a sentence in the chunk."""
        chunk, output = state["chunk"], state["output"]
        parsed: Verdicts | None = output["parsed"]
        claims = []
        if parsed is not None:
            for verdict in parsed.sentences:
                if self._is_claim(verdict, len(chunk.sentences)):
                    claims.extend(self._verdict_claims(verdict, chunk))
        result = ChunkResult(
            id=chunk.id,
            key=chunk.key,
            claims=claims,
            error=self._parsing_error(output),
            tokens=self._tokens(output["raw"]),
        )
        return {"result": result}

    async def extract(self, chunk: Chunk) -> ChunkResult:
        """Run one chunk through the graph."""
        state = await self.graph.ainvoke({"chunk": chunk})
        return state["result"]

    @staticmethod
    def _is_claim(verdict: SentenceVerdict, count: int) -> bool:
        """Whether the verdict holds a claim about a sentence from 1 to ``count``."""
        return verdict.kind != "none" and 1 <= verdict.n <= count

    @staticmethod
    def _refers_to(labels: list[str], chunk: Chunk, own: int) -> list[str]:
        """Turn [s.n] and [n] labels into 'chunk id|n', dropping the claim's own sentence and labels that name nothing."""
        found = []
        for label in labels:
            match = re.fullmatch(r"\[?\s*(?:(\d+)\.)?(\d+)\s*\]?", label.strip())
            if not match:
                continue
            section, n = match.group(1), int(match.group(2))
            if section is None:
                if 1 <= n <= len(chunk.sentences) and n != own:
                    found.append(f"{chunk.id}|{n}")
            elif 1 <= int(section) <= len(chunk.earlier_ids):
                found.append(f"{chunk.earlier_ids[int(section) - 1]}|{n}")
        return found

    @staticmethod
    def _verdict_claims(verdict: SentenceVerdict, chunk: Chunk) -> list[Claim]:
        """Turn one verdict into one claim per listed claim, or one empty claim."""
        claims = []
        for text in verdict.claims or [ClaimText(claim="", quantifier="exists")]:
            claims.append(
                Claim(
                    evidence=[verdict.n],
                    claim_type=verdict.kind,
                    claim=text.claim,
                    how_to_check=verdict.how_to_check,
                    quantifier=text.quantifier,
                    refers_to=ClaimExtractor._refers_to(text.refers_to, chunk, verdict.n),
                )
            )
        return claims

    @staticmethod
    def _parsing_error(output: dict[str, Any]) -> str | None:
        """Return the structured output's parsing error as text, or None."""
        if output["parsing_error"]:
            return str(output["parsing_error"])
        return None

    @staticmethod
    def _tokens(message: BaseMessage) -> TokenCounts:
        """Read the tokens a model call used from its message; 0 when unknown."""
        usage = getattr(message, "usage_metadata", None) or {}
        return TokenCounts.model_validate(
            {"in": usage.get("input_tokens", 0), "out": usage.get("output_tokens", 0)}
        )

In [ ]:
class ExtractionCache:
    """Store results per chunk key on disk.

    Entries live at ``<root>/<model>/<prompt hash>/<key>.json``, with ``/`` in
    the model name replaced by ``_`` and the prompt hash the first 8 hex
    characters of the system prompt's sha256. A changed prompt therefore
    starts an empty cache.

    Parameters
    ----------
    root :
        The cache's root folder.
    model :
        The model name.
    prompt :
        The system prompt.
    """

    def __init__(self, root: plb.Path, model: str, prompt: str) -> None:
        prompt_hash = hashlib.sha256(prompt.encode()).hexdigest()[:8]
        self._directory = root / model.replace("/", "_") / prompt_hash

    def _entry_path(self, key: str) -> plb.Path:
        """Return the file that holds one chunk key's entry."""
        return self._directory / f"{key}.json"

    def load(self, chunk: Chunk) -> ChunkResult | None:
        """Return the chunk's cached result, marked as cached, or None on a miss."""
        path = self._entry_path(chunk.cache_key)
        if not path.exists():
            return None
        result = ChunkResult.model_validate_json(path.read_text())
        result.cached = True
        return result

    def save(self, result: ChunkResult, chunk: Chunk) -> None:
        """Store a fresh result under its chunk's cache key."""
        self._directory.mkdir(parents=True, exist_ok=True)
        self._entry_path(chunk.cache_key).write_text(result.to_cache_json())

In [ ]:
class ExtractionRunner:
    """Extract many chunks, reading the cache first.

    Parameters
    ----------
    extractor :
        Sends a chunk to the model.
    cache :
        Where results are read from and written to.
    max_concurrent :
        The most model calls in flight at once.
    """

    def __init__(
        self,
        extractor: ClaimExtractor,
        cache: ExtractionCache,
        max_concurrent: int = MAX_CONCURRENT_CALLS,
    ) -> None:
        self._extractor = extractor
        self._cache = cache
        self._max_concurrent = max_concurrent

    async def run(self, chunks: list[Chunk]) -> list[ChunkResult]:
        """Extract every chunk not yet cached, a bounded number at a time.

        Parameters
        ----------
        chunks :
            The chunks to extract.

        Returns
        -------
        list[ChunkResult]
            One result per chunk, in the same order.
        """
        limit = asyncio.Semaphore(self._max_concurrent)
        tasks = []
        for chunk in chunks:
            tasks.append(self._load_or_extract(chunk, limit))
        return list(await asyncio.gather(*tasks))

    async def _load_or_extract(
        self, chunk: Chunk, limit: asyncio.Semaphore
    ) -> ChunkResult:
        """Return the cached result, or extract the chunk and cache it."""
        cached = self._cache.load(chunk)
        if cached is not None:
            return cached
        async with limit:
            result = await self._extractor.extract(chunk)
        self._cache.save(result, chunk)
        return result

In [ ]:
def build_runner(model: str = MODEL, run: str = "") -> ExtractionRunner:
    """Build a runner for one model, caching under ``results/cache/``.

    Raises
    ------
    KeyError
        If ``BIFROST_CC_VIRTUAL_KEY`` is not set.
    """
    llm = ChatOpenAI(
        base_url="https://bifrost.lab.orangecluster.nl/v1",
        api_key=os.environ["BIFROST_CC_VIRTUAL_KEY"],  # type: ignore[arg-type]
        model=model,
        temperature=0,
        timeout=120,
        max_retries=3,
    )
    cache = ExtractionCache(root=plb.Path("results") / "cache", model=model, prompt=SYSTEM_PROMPT + FILL_NOTE + run)
    return ExtractionRunner(extractor=ClaimExtractor(llm=llm), cache=cache)

In [117]:
print(build_runner(MODEL)._extractor.graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	verdicts(verdicts)
	claims(claims)
	__end__([<p>__end__</p>]):::last
	__start__ --> verdicts;
	verdicts --> claims;
	claims --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [118]:
print(SYSTEM_PROMPT)

You read one section of a design document from a software repository and
decide, sentence by sentence, which claims it makes about that repository.

Why: an auditor checks each claim against the code and reports the ones that
no longer hold. A claim it cannot check wastes its time; a claim you miss is
drift nobody sees.

Go through every numbered sentence in order and give each one a verdict,
including sentences that hold no claim. Do not stop early.

Kinds, by how a claim is checked:
- point: look at one named thing in this repository (function, command,
  flag, config key, file, workflow, test).
- structural: check many files for a pattern ("every X does Y", "nothing
  imports Z", "only the CLI writes").
- subjective: a principle or value a person must judge; no file settles it.
- external: about a tool or service outside this repository.
- none: no claim. Status, dates, authors; history ("came out of", "used
  to", "was"); measurements from a past run; the reasons for a choice; the


In [119]:
example = next(c for c in all_chunks if c["id"] == "manifesto#invariants")
print(ClaimExtractor.user_message(Chunk.model_validate(example)))

Concept: manifesto (type: manifesto, status: stable)
Title: Manifesto: kmctl
Section: Manifesto: kmctl > Invariants

[1] Each is `planned` until the test named for it passes, then `enforced`.
[2] - **I1. One hook of its own, and it never breaks a session.** kmctl's own hook is SessionStart, and it exits 0 even when `kmctl` is missing or fails.
[3] In Antigravity kmctl also relays okf's opt-in Stop check, which APM's copy cannot run there (ADR 0018); it exits 0 and prints `{}` when okf's script is missing or fails.
[4] Status: enforced by `tests/test_plugin.py` (the package ships one SessionStart hook; it exits 0 without `kmctl` and when it fails) and `tests/test_antigravity.py` (Antigravity's SessionStart hook prints `{}` and exits 0 in both cases; the okf Stop relay prints `{}` and exits 0 without a working script) and `tests/test_opencode.py` (OpenCode's briefing plugin adds nothing and throws nothing without a working `kmctl`).
[5] - **I2. Audits never write knowledge.** The audit a

In [120]:
runner = build_runner(MODEL)
extracted = await runner.run([Chunk.model_validate(c) for c in gold_chunks])
out = plb.Path("results") / f"{MODEL.replace('/', '_')}.jsonl"
out.write_text("".join(r.to_json() + "\n" for r in extracted))
sum(not r.cached for r in extracted), "sent to the model"

(0, 'sent to the model')

In [121]:
by_id = {r.id: r for r in extracted}
for c in by_id["manifesto#invariants"].claims:
    print(c.evidence, c.claim_type, "|", c.claim)

[2] point | kmctl's own hook is SessionStart
[2] point | kmctl exits 0 even when kmctl is missing or fails
[3] point | In Antigravity kmctl relays okf's opt-in Stop check
[3] point | Antigravity kmctl exits 0 and prints {} when okf's script is missing or fails
[4] point | tests/test_plugin.py enforces that the package ships one SessionStart hook that exits 0 without kmctl and when it fails
[4] point | tests/test_antigravity.py enforces that Antigravity's SessionStart hook prints {} and exits 0 in both cases
[4] point | tests/test_antigravity.py enforces that the okf Stop relay prints {} and exits 0 without a working script
[4] point | tests/test_opencode.py enforces that OpenCode's briefing plugin adds nothing and throws nothing without a working kmctl
[5] structural | The audit agent, its skills and the CI action report findings
[5] structural | The audit agent, its skills and the CI action never edit .okf/ or docs/ and never write a verified stamp
[7] point | The hook prints what kmc

## Score

Per sentence, so the score does not depend on how finely the model splits claims. For each gold sentence the model either cites it as evidence for a claim of some type, or does not.

- recall: sentences that must be cited and were.
- precision: cited sentences that may be cited (required or optional).
- type ok: cited, labelled sentences where one of the model's types is allowed.
- checkable precision: sentences cited by a point or structural claim whose label allows point or structural; what the auditor would waste time on.

In [122]:
CHECKABLE = {"point", "structural"}
NOT_A_CLAIM = {"none"}

In [123]:
class ScoreRow(BaseModel):
    """The scores of one run, in the column order they are printed."""

    run: str = Field(description="A name for the run, such as the model.")
    recall: float = Field(description="Percent of required sentences cited.")
    precision: float = Field(
        description="Percent of cited sentences that may be cited."
    )
    type_ok: float = Field(
        description="Percent of cited, labelled sentences with an allowed type."
    )
    checkable_precision: float = Field(
        description="Percent of sentences cited as point or structural that allow either."
    )
    claims: int = Field(description="Claims extracted.")
    errors: int = Field(description="Chunks whose answer could not be read.")
    stale: int = Field(description="Gold chunks with no result for their current key.")
    tokens_in: int = Field(description="Tokens sent to the model.")
    tokens_out: int = Field(description="Tokens the model returned.")

In [124]:
class Scorer:
    """Score extraction runs against the gold set, one method per measure.

    Each measure except ``stale`` takes the pairs of a gold record and its
    fresh result: the result for the same chunk id with the same key.

    Parameters
    ----------
    gold :
        The gold records to score against.
    """

    def __init__(self, gold: list[GoldRecord]) -> None:
        self._gold = gold

    def score(self, run: str, extracted: list[ChunkResult]) -> ScoreRow:
        """Compute the sentence-level scores of one run against the gold set.

        Parameters
        ----------
        run :
            A name for the run, such as the model.
        extracted :
            One result per chunk; a later result for the same chunk wins.

        Returns
        -------
        ScoreRow
            Every measure for that run.
        """
        results = {result.id: result for result in extracted}
        pairs = self._fresh_pairs(results)
        return ScoreRow(
            run=run,
            recall=self.recall(pairs),
            precision=self.precision(pairs),
            type_ok=self.type_ok(pairs),
            checkable_precision=self.checkable_precision(pairs),
            claims=self.claims(pairs),
            errors=self.errors(pairs),
            stale=self.stale(results),
            tokens_in=self.tokens_in(pairs),
            tokens_out=self.tokens_out(pairs),
        )

    def _fresh_pairs(
        self, results: dict[str, ChunkResult]
    ) -> list[tuple[GoldRecord, ChunkResult]]:
        """Pair each gold record with its result, leaving out stale ones."""
        pairs = []
        for gold in self._gold:
            result = results.get(gold.id)
            if result is not None and result.key == gold.key:
                pairs.append((gold, result))
        return pairs

    @staticmethod
    def _percent(part: int, whole: int) -> float:
        """Return ``part`` as a percentage of ``whole``, to one decimal; 0.0 when ``whole`` is 0."""
        if not whole:
            return 0.0
        return round(100 * part / whole, 1)

    def stale(self, results: dict[str, ChunkResult]) -> int:
        """Count the gold chunks with no result, or a result for an older key."""
        fresh = self._fresh_pairs(results)
        return len(self._gold) - len(fresh)

    def recall(self, pairs: list[tuple[GoldRecord, ChunkResult]]) -> float:
        """Return the percent of sentences that must be cited and were."""
        required = 0
        cited_required = 0
        for gold, result in pairs:
            required_here = gold.required_sentences()
            required += len(required_here)
            cited_required += len(required_here & result.predicted_types().keys())
        return self._percent(cited_required, required)

    def precision(self, pairs: list[tuple[GoldRecord, ChunkResult]]) -> float:
        """Return the percent of cited sentences that may be cited."""
        cited = 0
        cited_allowed = 0
        for gold, result in pairs:
            labels = gold.sentence_labels()
            for number in result.predicted_types():
                cited += 1
                if labels.get(number, NOT_A_CLAIM) != NOT_A_CLAIM:
                    cited_allowed += 1
        return self._percent(cited_allowed, cited)

    def type_ok(self, pairs: list[tuple[GoldRecord, ChunkResult]]) -> float:
        """Return the percent of cited, labelled sentences where one of the model's types is allowed."""
        typed = 0
        typed_ok = 0
        for gold, result in pairs:
            labels = gold.sentence_labels()
            for number, types in result.predicted_types().items():
                allowed = labels.get(number, NOT_A_CLAIM)
                if allowed == NOT_A_CLAIM:
                    continue
                typed += 1
                if types & allowed:
                    typed_ok += 1
        return self._percent(typed_ok, typed)

    def checkable_precision(
        self, pairs: list[tuple[GoldRecord, ChunkResult]]
    ) -> float:
        """Return the percent of sentences cited as point or structural whose label allows either."""
        checkable_cited = 0
        checkable_ok = 0
        for gold, result in pairs:
            labels = gold.sentence_labels()
            for number, types in result.predicted_types().items():
                if not types & CHECKABLE:
                    continue
                checkable_cited += 1
                if labels.get(number, NOT_A_CLAIM) & CHECKABLE:
                    checkable_ok += 1
        return self._percent(checkable_ok, checkable_cited)

    @staticmethod
    def claims(pairs: list[tuple[GoldRecord, ChunkResult]]) -> int:
        """Count the claims extracted."""
        total = 0
        for _, result in pairs:
            total += len(result.claims)
        return total

    @staticmethod
    def errors(pairs: list[tuple[GoldRecord, ChunkResult]]) -> int:
        """Count the chunks whose answer could not be read."""
        total = 0
        for _, result in pairs:
            if result.error:
                total += 1
        return total

    @staticmethod
    def tokens_in(pairs: list[tuple[GoldRecord, ChunkResult]]) -> int:
        """Sum the tokens sent to the model."""
        total = 0
        for _, result in pairs:
            total += result.tokens.input
        return total

    @staticmethod
    def tokens_out(pairs: list[tuple[GoldRecord, ChunkResult]]) -> int:
        """Sum the tokens the model returned."""
        total = 0
        for _, result in pairs:
            total += result.tokens.output
        return total

In [ ]:
scorer = Scorer(gold=[GoldRecord.model_validate(g) for g in gold])
row = scorer.score(MODEL, extracted).model_dump()
# A second run in its own cache: the same prompt gives different verdicts from run to run.
again = await build_runner(MODEL, run="@2").run([Chunk.model_validate(c) for c in gold_chunks])
row2 = scorer.score(MODEL + "@2", again).model_dump()
for column, value in row.items():
    print(f"{column:20} {value}   {row2[column]}")

### Quantifiers: one place, every instance, or none

`gold_quantifiers.json` labels the 148 gold sentences that must or may hold a
checkable claim with the quantifiers of their claims. A subagent
(claude-opus/5.5) labelled them from the sentences alone, with the
definitions the prompt uses; no person has checked them. What matters for
routing is whether a sentence holds a universal (`all` or `none`) claim.

In [ ]:
gold_quantifiers = json.load(open("gold_quantifiers.json"))
by_id = {r.id: r for r in extracted}
UNIVERSAL = {"all", "none"}
pairs = []
for g in gold_quantifiers:
    predicted = {c.quantifier for c in by_id[g["id"]].claims if c.evidence == [g["n"]] and c.claim_type in CHECKABLE}
    pairs.append((g, set(g["quantifiers"]), predicted))
found = sum(1 for _, gq, p in pairs if gq & UNIVERSAL and p & UNIVERSAL)
missed = sum(1 for _, gq, p in pairs if gq & UNIVERSAL and not p & UNIVERSAL)
extra = sum(1 for _, gq, p in pairs if not gq & UNIVERSAL and p & UNIVERSAL)
agree = sum(1 for _, gq, p in pairs if bool(gq & UNIVERSAL) == bool(p & UNIVERSAL))
print(f"universal or not: agree on {agree} of {len(pairs)}; gold universal {found + missed}, model found {found}, "
      f"missed {missed}, marked {extra} more")
for g, gq, p in pairs:
    if bool(gq & UNIVERSAL) != bool(p & UNIVERSAL):
        print(f"  gold {sorted(gq)} model {sorted(p) or '-'}: {chunks_by_id[g['id']]['sentences'][g['n'] - 1][:140]}")

## Anchors: what a claim names, looked up in the code

An anchor is something a sentence names that should exist in the code: a path, or a name (function, class, constant, command, flag, environment variable, config key). Nothing here knows which kind it is or which language the code is in.

1. **Extract** anchors without an LLM: every backticked span, and every link to a file that is not markdown.
2. **Resolve** each one, cheapest first: a tracked path, then a definition from tree-sitter, then a mention in code, then a mention in prose.
3. An anchor that resolves nowhere is a candidate finding: the claim names something that does not exist.

In [128]:
import re
from typing import Literal


class Anchor(BaseModel):
    """Something a sentence names that should exist in the code."""

    text: str
    source: Literal["code_span", "link", "bare", "sources"]


class AnchorExtractor:
    """Pull anchors out of a sentence without an LLM: backticked spans and links to non-markdown files."""

    def __init__(self):
        self._code_span = re.compile(r"`([^`]+)`")
        self._link = re.compile(r"\]\(([^)\s]+)\)")

    @staticmethod
    def _is_anchor(text: str) -> bool:
        # A code block is not one anchor, and a span with no letters names nothing.
        return "\n" not in text and sum(char.isalnum() for char in text) >= 2

    def _code_spans(self, sentence: str) -> Generator[Anchor, None, None]:
        for match in self._code_span.finditer(sentence):
            text = match.group(1).strip()
            if self._is_anchor(text):
                yield Anchor(text=text, source="code_span")

    def _links(self, sentence: str) -> Generator[Anchor, None, None]:
        for match in self._link.finditer(sentence):
            target = match.group(1).split("#")[0]
            if target and "://" not in target and not target.endswith(".md"):
                yield Anchor(text=target, source="link")

    def extract(self, sentence: str) -> list[Anchor]:
        return [*self._code_spans(sentence), *self._links(sentence)]

In [129]:
from fnmatch import fnmatch


class TrackedPaths:
    """Every path git tracks, matched exactly, as a folder, as a glob or by its ending."""

    def __init__(self, root: plb.Path):
        self._root = root
        self._paths = self._ls_files()
        self._path_set = set(self._paths)

    def _ls_files(self) -> list[str]:
        result = subprocess.run(
            ["git", "-C", str(self._root), "ls-files"],
            capture_output=True, text=True, check=True
        )
        return result.stdout.splitlines()

    @staticmethod
    def _normalise(text: str) -> str:
        return text.strip().removeprefix("./").rstrip("/")

    def _exact(self, candidate: str) -> list[str]:
        return [candidate] if candidate in self._path_set else []

    def _folder(self, candidate: str) -> list[str]:
        return [p for p in self._paths if p.startswith(candidate + "/")]

    def _glob(self, candidate: str) -> list[str]:
        if not any(char in candidate for char in "*?["):
            return []
        return [p for p in self._paths if fnmatch(p, candidate)]

    def _ending(self, candidate: str) -> list[str]:
        if len(candidate) < 3 or " " in candidate:
            return []
        return [p for p in self._paths if p.endswith("/" + candidate)]

    def find(self, text: str) -> tuple[str, list[str]]:
        candidate = self._normalise(text)
        if not candidate:
            return "", []
        for how, match in (("exact", self._exact), ("folder", self._folder), ("glob", self._glob), ("ending", self._ending)):
            found = match(candidate)
            if found:
                return how, found
        return "", []

In [ ]:
class Definition(BaseModel):
    """One named definition, or config key, found by tree-sitter."""

    name: str
    path: str
    node_type: str
    line: int
    end_line: int = 0


class DefinitionIndex:
    """Definitions and config keys across every parsed file, by name.

    A node counts when its type reads like a definition (definition, declaration,
    spec, item, assignment, pair, ...) and it has a name, key or left field.
    Grammars without fields (TOML, HCL) fall back to the first key-like child.
    """

    _keep = re.compile(r"definition|declaration|declarator|spec|item|assignment|pair|table|attribute|block")
    _skip = re.compile(r"parameter|argument|call|command")
    _key_children = {"bare_key", "quoted_key", "dotted_key", "identifier"}

    def __init__(self, root: plb.Path, files: list[FileWithMedatadataModel], parsers: dict):
        self._root = root
        self._by_name: dict[str, list[Definition]] = {}
        self._build(files, parsers)

    @staticmethod
    def _walk(node) -> Generator:
        stack = [node]
        while stack:
            current = stack.pop()
            stack.extend(current.children)
            yield current

    def _is_definition(self, node) -> bool:
        return bool(self._keep.search(node.type)) and not self._skip.search(node.type)

    def _name_node(self, node):
        for field in ("name", "key", "left"):
            child = node.child_by_field_name(field)
            if child is not None:
                return child
        for child in node.named_children:
            if child.type in self._key_children:
                return child
        return None

    @staticmethod
    def _clean(text: bytes) -> str:
        return text.decode("utf-8", "replace").strip().strip("\"'")

    def _definitions(self, file: FileWithMedatadataModel, parser) -> Generator[Definition, None, None]:
        tree = parser.parse(file.content)
        for node in self._walk(tree.root_node):
            if not self._is_definition(node):
                continue
            name_node = self._name_node(node)
            if name_node is None:
                continue
            name = self._clean(name_node.text)
            if name:
                yield Definition(
                    name=name,
                    path=str(file.path.relative_to(self._root)),
                    node_type=node.type,
                    line=node.start_point[0] + 1,
                    end_line=node.end_point[0] + 1,
                )

    def _build(self, files: list[FileWithMedatadataModel], parsers: dict) -> None:
        for file in files:
            parser = parsers.get(file.metadata.lang_name)
            if parser is None:
                continue
            for definition in self._definitions(file, parser):
                self._by_name.setdefault(definition.name, []).append(definition)

    def _dotted(self, name: str) -> list[Definition]:
        parts = name.split(".")
        if len(parts) < 2:
            return []
        paths = None
        for part in parts:
            found = {d.path for d in self._by_name.get(part, [])}
            paths = found if paths is None else paths & found
        return [d for d in self._by_name.get(parts[-1], []) if d.path in paths]

    def find(self, name: str) -> tuple[str, list[Definition]]:
        exact = self._by_name.get(name, [])
        if exact:
            return "exact", exact
        dotted = self._dotted(name)
        if dotted:
            return "dotted", dotted
        return "", []

    def in_file(self, path: str) -> list[Definition]:
        found = [d for definitions in self._by_name.values() for d in definitions if d.path == path]
        return sorted(found, key=lambda d: d.line)

    def __len__(self) -> int:
        return sum(len(found) for found in self._by_name.values())

In [131]:
class MentionSearch:
    """Whole-word search over file contents, split into code and prose (markdown)."""

    _token = re.compile(r"[A-Za-z_][A-Za-z0-9_\-]{2,}")

    def __init__(self, root: plb.Path, files: list[FileWithMedatadataModel]):
        self._documents = [
            (str(f.path.relative_to(root)), f.content_str, f.path.suffix == ".md")
            for f in files
        ]

    @staticmethod
    def _word(text: str) -> re.Pattern:
        return re.compile(r"(?<![\w-])" + re.escape(text) + r"(?![\w-])")

    def _phrase(self, text: str, prose: bool) -> list[str]:
        pattern = self._word(text)
        return [path for path, content, is_prose in self._documents if is_prose == prose and pattern.search(content)]

    def _tokens(self, text: str, prose: bool) -> list[str]:
        tokens = self._token.findall(text)
        if len(tokens) < 2:
            return []
        patterns = [self._word(token) for token in tokens]
        return [
            path for path, content, is_prose in self._documents
            if is_prose == prose and all(p.search(content) for p in patterns)
        ]

    def find(self, text: str, prose: bool) -> tuple[str, list[str]]:
        found = self._phrase(text, prose)
        if found:
            return "phrase", found
        found = self._tokens(text, prose)
        if found:
            return "tokens", found
        return "", []

In [132]:
class Resolution(BaseModel):
    """Where an anchor was found, if anywhere."""

    anchor: Anchor
    status: Literal["path", "defined", "mentioned_in_code", "mentioned_in_prose", "unresolved"]
    via: str = ""
    locations: list[str] = []


class AnchorResolver:
    """Resolve an anchor through paths, definitions, code mentions and prose mentions, cheapest first."""

    def __init__(self, paths: TrackedPaths, definitions: DefinitionIndex, mentions: MentionSearch):
        self._paths = paths
        self._definitions = definitions
        self._mentions = mentions

    def _qualified(self, anchor: Anchor) -> Resolution | None:
        if "::" not in anchor.text:
            return None
        path_part, _, name_part = anchor.text.partition("::")
        _, paths = self._paths.find(path_part)
        _, definitions = self._definitions.find(name_part.split("::")[-1])
        inside = [f"{d.path}:{d.line}" for d in definitions if d.path in paths]
        if inside:
            return Resolution(anchor=anchor, status="defined", via="qualified", locations=inside)
        return Resolution(anchor=anchor, status="unresolved", via="qualified")

    def _path(self, anchor: Anchor) -> Resolution | None:
        how, found = self._paths.find(anchor.text)
        if found:
            return Resolution(anchor=anchor, status="path", via=how, locations=found[:5])
        return None

    def _definition(self, anchor: Anchor) -> Resolution | None:
        how, found = self._definitions.find(anchor.text)
        if found:
            locations = [f"{d.path}:{d.line}" for d in found[:5]]
            return Resolution(anchor=anchor, status="defined", via=how, locations=locations)
        return None

    def _mention(self, anchor: Anchor, prose: bool) -> Resolution | None:
        how, found = self._mentions.find(anchor.text, prose=prose)
        if found:
            status = "mentioned_in_prose" if prose else "mentioned_in_code"
            return Resolution(anchor=anchor, status=status, via=how, locations=found[:5])
        return None

    def resolve(self, anchor: Anchor) -> Resolution:
        qualified = self._qualified(anchor)
        if qualified is not None:
            return qualified
        for step in (self._path, self._definition):
            resolution = step(anchor)
            if resolution is not None:
                return resolution
        for prose in (False, True):
            resolution = self._mention(anchor, prose=prose)
            if resolution is not None:
                return resolution
        return Resolution(anchor=anchor, status="unresolved")

### Build the lookups

The index and the mention search read `files`, the code the `ExclusionFileReader` returned at the top (so not `.okf/` or `docs/`, which would find the claim's own text). Paths use everything git tracks, since a claim may name a docs or `.okf/` path.

In [133]:
tracked = TrackedPaths(root_path)
definitions = DefinitionIndex(root_path, files, parsers)
mentions = MentionSearch(root_path, files)
resolver = AnchorResolver(tracked, definitions, mentions)
anchor_extractor = AnchorExtractor()

len(definitions), "definitions indexed"

(15392, 'definitions indexed')

### Resolve the anchors of every gold sentence that must hold a checkable claim

Checkable means the gold label allows only `point` or `structural`.

In [ ]:
class SentenceAnchors(BaseModel):
    """The anchors of one gold sentence and where each resolved."""

    chunk_id: str
    number: int
    sentence: str
    resolutions: list[Resolution]


def checkable_sentences(gold: list[dict], chunks_by_id: dict) -> Generator[tuple[str, int, str], None, None]:
    for record in gold:
        sentences = chunks_by_id[record["id"]]["sentences"]
        for number, labels in record["labels"].items():
            if "none" in labels or not set(labels) <= {"point", "structural"}:
                continue
            yield record["id"], int(number), sentences[int(number) - 1]


sentence_anchors = [
    SentenceAnchors(
        chunk_id=chunk_id,
        number=number,
        sentence=sentence,
        resolutions=[resolver.resolve(a) for a in anchor_extractor.extract(sentence)],
    )
    for chunk_id, number, sentence in checkable_sentences(gold, chunks_by_id)
]
len(sentence_anchors), "checkable sentences"

In [135]:
from collections import Counter

all_resolutions = [r for s in sentence_anchors for r in s.resolutions]
without_anchors = [s for s in sentence_anchors if not s.resolutions]
with_unresolved = [s for s in sentence_anchors if any(r.status == "unresolved" for r in s.resolutions)]

print(f"{len(sentence_anchors)} sentences, {len(all_resolutions)} anchors")
print(f"{len(without_anchors)} sentences have no anchor; {len(with_unresolved)} have at least one unresolved anchor")
for status, count in Counter(r.status for r in all_resolutions).most_common():
    print(f"  {status:20} {count:4}  {100 * count / len(all_resolutions):5.1f}%")

127 sentences, 280 anchors
31 sentences have no anchor; 2 have at least one unresolved anchor
  mentioned_in_code     117   41.8%
  path                   99   35.4%
  defined                60   21.4%
  mentioned_in_prose      2    0.7%
  unresolved              2    0.7%


### Every unresolved anchor, to judge by hand: real drift or a lookup miss?

In [136]:
for s in with_unresolved:
    for r in s.resolutions:
        if r.status == "unresolved":
            print(f"{s.chunk_id}[{s.number}]  `{r.anchor.text}`")

decisions/0017-the-ci-audit-is-a-standalone-pydanticai-agent-script#correction-2026-10-17[18]  `agent-report-NAME`
architecture/overview#the-package-and-its-install[2]  `{git: SOURCE, path: apm, ref: v<cli version>}`


### Anchors that only resolved weakly (a mention, not a path or definition)

In [137]:
for r in all_resolutions:
    if r.status.startswith("mentioned"):
        print(f"{r.status:20} {r.via:7} `{r.anchor.text[:60]}` -> {r.locations[:2]}")

mentioned_in_code    phrase  `--path` -> ['src/kmctl/__init__.py', 'src/kmctl/briefing.py']
mentioned_in_code    phrase  `.git` -> ['ov-sync.toml', 'action/agent.py']
mentioned_in_code    phrase  `.agents/` -> ['src/kmctl/__init__.py', 'src/kmctl/antigravity.py']
mentioned_in_code    tokens  `uvx --from git+https://github.com/JasperHG90/kmctl@<resolved` -> ['src/kmctl/pin.py']
mentioned_in_code    phrase  `KMCTL_PINNED=1` -> ['src/kmctl/__init__.py', 'src/kmctl/pin.py']
mentioned_in_code    phrase  `kmctl briefing` -> ['apm/antigravity/.apm/hooks/hooks.json', 'apm/claude/.apm/hooks/hooks.json']
mentioned_in_code    tokens  `uvx --offline ... kmctl --version` -> ['src/kmctl/doctor.py', 'src/kmctl/pin.py']
mentioned_in_code    phrase  `just install` -> ['src/kmctl/__init__.py', 'src/kmctl/pin.py']
mentioned_in_code    phrase  `newer_release` -> ['src/kmctl/pin.py', 'tests/test_briefing.py']
mentioned_in_code    phrase  `--json` -> ['.github/workflows/hermes-issue-triage.yaml', '.github/w

## Tying claims to code: the building blocks

Every point or structural claim the extractor finds is tied to code through the anchors of the sentence it cites. A claim's tie is its strongest anchor:

- `strong`: an anchor is a tracked path or a tree-sitter definition;
- `weak`: anchors only turn up as mentions;
- `unresolved`: it has anchors and none resolve;
- `none`: the sentence names nothing, so there is nothing to look up.

`how_to_check` is the model's own text and often names a file or command, so it is counted separately: does it tie claims the sentence alone cannot?

In [ ]:
class ClaimTie(BaseModel):
    """One extracted claim and the anchors that tie it to code."""

    chunk_id: str
    claim: Claim
    sentence: str
    from_sentence: list[Resolution]
    from_check: list[Resolution]

    @staticmethod
    def _tie(resolutions: list[Resolution]) -> str:
        statuses = {r.status for r in resolutions}
        if statuses & {"path", "defined"}:
            return "strong"
        if statuses & {"mentioned_in_code", "mentioned_in_prose"}:
            return "weak"
        if statuses:
            return "unresolved"
        return "none"

    @property
    def tie(self) -> str:
        return self._tie(self.from_sentence)

    @property
    def tie_with_check(self) -> str:
        return self._tie(self.from_sentence + self.from_check)

    def files(self) -> set[str]:
        """The files a strong anchor points to."""
        found = set()
        for r in self.from_sentence:
            if r.status in {"path", "defined"}:
                found.update(location.split(":")[0] for location in r.locations)
        return found


# Claims the audit ties to code. Whether code can settle one is a separate step (Checkability);
# the extractor's broad is a hint to that step, so broad claims go there too.
AUDITED = {"point", "structural", "broad"}


class ClaimTier:
    """Ties extracted claims to code, resolving each distinct anchor once."""

    def __init__(self, extractor: AnchorExtractor, resolver: AnchorResolver):
        self._extractor = extractor
        self._resolver = resolver
        self._seen: dict[str, Resolution] = {}

    def _resolve(self, text: str) -> list[Resolution]:
        resolutions = []
        for anchor in self._extractor.extract(text):
            if anchor.text not in self._seen:
                self._seen[anchor.text] = self._resolver.resolve(anchor)
            resolutions.append(self._seen[anchor.text])
        return resolutions

    def tie(self, chunk_id: str, claim: Claim, sentences: list[str]) -> ClaimTie | None:
        """Tie one point or structural claim through the anchors of the sentence it cites; None for any other claim."""
        if claim.claim_type not in AUDITED or not claim.evidence:
            return None
        sentence = " ".join(sentences[n - 1] for n in claim.evidence if 0 < n <= len(sentences))
        return ClaimTie(
            chunk_id=chunk_id,
            claim=claim,
            sentence=sentence,
            from_sentence=self._resolve(sentence),
            from_check=self._resolve(claim.how_to_check),
        )

In [139]:
DOC_FOLDERS = (".okf/", "docs/")


def precision_of(resolution: Resolution) -> str:
    locations = [location.split(":")[0] for location in resolution.locations]
    if all(location.startswith(DOC_FOLDERS) for location in locations):
        return "docs only"
    if resolution.status == "defined" or (resolution.via in {"exact", "ending"} and len(locations) == 1):
        return "file or symbol"
    return "folder or glob"


def claim_precision(tie: ClaimTie) -> str:
    found = {precision_of(r) for r in tie.from_sentence if r.status in {"path", "defined"}}
    for level in ["file or symbol", "folder or glob", "docs only"]:
        if level in found:
            return level
    return tie.tie

## Closing the gap without a model: bare names and context

Two more places to look for anchors, for claims whose cited sentence has none that tie precisely:

1. **Bare names**: words outside backticks shaped like identifiers (a `/`, `.`, `_` or `-` inside, camelCase, or a leading `/`). Most such words are English, so a bare name only counts when it resolves to a path or a definition, never as a mention.
2. **Context**: the sentence just before, if it ties precisely ("The exit code passes through" follows the sentence that names the launcher), then the paths in the concept's `sources` front matter.

Context ties are inferred: the claim's subject is probably there, but nothing in the sentence says so.

In [140]:
class BareNameExtractor:
    """Identifier-shaped words outside backticks and links."""

    _code_span = re.compile(r"`[^`]*`")
    _link = re.compile(r"\]\([^)]*\)")
    _word = re.compile(r"(?<![\w/.-])/?[A-Za-z_.][\w./-]*[\w/]")
    _camel = re.compile(r"[a-z][A-Z]")

    def _looks_like_identifier(self, word: str) -> bool:
        if len(word) < 3 or "://" in word:
            return False
        return any(char in word for char in "/_-") or "." in word.strip(".") or bool(self._camel.search(word))

    def extract(self, sentence: str) -> list[Anchor]:
        text = self._link.sub(" ", self._code_span.sub(" ", sentence))
        words = sorted({word.rstrip(".") for word in self._word.findall(text)})
        return [Anchor(text=word, source="bare") for word in words if self._looks_like_identifier(word)]


class ConceptSources:
    """The tracked paths a concept's ``sources`` front matter lists."""

    def __init__(self, bundle: plb.Path, paths: TrackedPaths):
        self._bundle = bundle
        self._paths = paths

    @staticmethod
    def _resource(item) -> str:
        return item.get("resource", "") if isinstance(item, dict) else str(item)

    def resolutions(self, concept: str) -> list[Resolution]:
        post = frontmatter.load(self._bundle / f"{concept}.md")
        found = []
        for item in post.get("sources") or []:
            resource = self._resource(item)
            how, paths = self._paths.find(resource)
            if paths:
                anchor = Anchor(text=resource, source="sources")
                found.append(Resolution(anchor=anchor, status="path", via=how, locations=paths[:5]))
        return found

In [141]:
class ClaimContext(BaseModel):
    """A claim's tie, plus the anchors found in bare names and context."""

    tie: ClaimTie
    bare: list[Resolution]
    previous: list[Resolution]
    sources: list[Resolution]

    @staticmethod
    def _precise(resolutions: list[Resolution]) -> bool:
        return any(r.status in {"path", "defined"} and precision_of(r) == "file or symbol" for r in resolutions)

    @property
    def level(self) -> str:
        """The first source that ties the claim precisely, else the tie it had."""
        if self._precise(self.tie.from_sentence):
            return "sentence"
        for name in ["bare", "previous", "sources"]:
            if self._precise(getattr(self, name)):
                return name
        return claim_precision(self.tie)

    def precise_resolutions(self) -> list[Resolution]:
        """The resolutions that make the level, for the drift test."""
        resolutions = {"sentence": self.tie.from_sentence, "bare": self.bare, "previous": self.previous, "sources": self.sources}
        return [r for r in resolutions.get(self.level, []) if r.status in {"path", "defined"} and precision_of(r) == "file or symbol"]


class ContextTier:
    """Adds bare-name and context anchors to each claim's tie, resolving each anchor once."""

    def __init__(self, anchors: AnchorExtractor, bare: BareNameExtractor, resolver: AnchorResolver, sources: ConceptSources):
        self._anchors = anchors
        self._bare = bare
        self._resolver = resolver
        self._sources = sources
        self._seen: dict[str, Resolution] = {}

    def _resolve(self, anchor: Anchor) -> Resolution:
        if anchor.text not in self._seen:
            self._seen[anchor.text] = self._resolver.resolve(anchor)
        return self._seen[anchor.text].model_copy(update={"anchor": anchor})

    def _bare_strong(self, sentence: str) -> list[Resolution]:
        resolved = [self._resolve(a) for a in self._bare.extract(sentence)]
        return [r for r in resolved if r.status in {"path", "defined"}]

    def _sentence(self, sentence: str) -> list[Resolution]:
        return [self._resolve(a) for a in self._anchors.extract(sentence)] + self._bare_strong(sentence)

    def _previous(self, sentences: list[str], number: int) -> list[Resolution]:
        """The anchors of the sentence just before, if it ties precisely."""
        if number < 2:
            return []
        resolutions = self._sentence(sentences[number - 2])
        return resolutions if ClaimContext._precise(resolutions) else []

    def context(self, tie: ClaimTie, chunk: dict) -> ClaimContext:
        number = tie.claim.evidence[0]
        return ClaimContext(
            tie=tie,
            bare=self._bare_strong(tie.sentence),
            previous=self._previous(chunk["sentences"], number),
            sources=self._sources.resolutions(chunk["concept"]),
        )

## Triage agent: tracing what is left with tree-sitter tools

The claims no lookup ties precisely go to two agents.

1. **The tracer** has the lookups as tools, plus reading code: `find_path`, `find_definition`, `search_code`, `list_definitions`, `read_lines`. It reports `found`, `not_found` or `not_code`, and every location carries a **quote**: the lines that make the claim true. Given where the claim was traced last time, it checks there first.
2. **The reviewer** is adversarial and never sees the search. It gets the claim and the code around each location, read from the repository by us rather than taken from the tracer, and its job is to reject: `supported`, `partly` or `not_supported`.

Deterministic checks sit between the two:

- every location must exist: a tracked path, its symbol in that file, its line inside it, and its quote within a few lines of that line;
- a test, fixture, eval or doc only counts as where a claim lives when the claim names that file. Otherwise it supports at most, and a claim found only there is `only tests or docs`.

Results are cached per claim, earlier location, model, prompts and repository copy.

In [ ]:
class CodeTools:
    """The lookups, as tools an agent can call, over one copy of a repository."""

    MAX_LINES = 80
    MAX_HITS = 25

    def __init__(self, root: plb.Path, paths: TrackedPaths, definitions: DefinitionIndex, files: list[FileWithMedatadataModel]):
        self._root = root
        self._paths = paths
        self._definitions = definitions
        self._texts = {str(f.path.relative_to(root)): f.content_str for f in files if f.metadata.is_text}

    def find_path(self, text: str) -> str:
        """Find tracked files by exact path, folder, glob or path ending."""
        how, found = self._paths.find(text)
        return f"{how}: " + "\n".join(found[: self.MAX_HITS]) if found else "no tracked path matches"

    def find_definition(self, name: str) -> str:
        """Find where a function, class, variable or config key is defined, by its name."""
        _, found = self._definitions.find(name)
        lines = [f"{d.path}:{d.line} {d.node_type}" for d in found[: self.MAX_HITS]]
        return "\n".join(lines) or "no definition with that name"

    def search_code(self, text: str) -> str:
        """Search every code file for a whole word or phrase; returns path:line: text."""
        pattern = MentionSearch._word(text)
        hits = []
        for path, content in self._texts.items():
            for number, line in enumerate(content.splitlines(), 1):
                if pattern.search(line):
                    hits.append(f"{path}:{number}: {line.strip()[:160]}")
        if not hits:
            return "no matches"
        return "\n".join(hits[: self.MAX_HITS]) + (f"\n... {len(hits) - self.MAX_HITS} more" if len(hits) > self.MAX_HITS else "")

    def list_definitions(self, path: str) -> str:
        """List what one file defines, with line numbers."""
        found = self._definitions.in_file(path)
        return "\n".join(f"{d.line} {d.node_type} {d.name}" for d in found[:150]) or "nothing defined, or not a parsed file"

    def _lines(self, path: str) -> list[str] | None:
        if path in self._texts:
            return self._texts[path].splitlines()
        file = self._root / path
        if self._paths.find(path)[0] == "exact" and file.is_file():
            return file.read_text(errors="replace").splitlines()
        return None

    def read_lines(self, path: str, start: int, end: int) -> str:
        """Read lines start to end (1-based, at most 80) of a tracked file."""
        lines = self._lines(path)
        if lines is None:
            return "not a tracked text file"
        end = min(end, start + self.MAX_LINES - 1, len(lines))
        return "\n".join(f"{n}: {lines[n - 1]}" for n in range(max(start, 1), end + 1)) or "no such lines"

    MAX_FILE_LINES = 300
    MAX_GIT_LINES = 120

    def grep(self, pattern: str, path_glob: str = "") -> str:
        """Search every tracked text file, docs included, for a Python regex; optionally only paths matching a glob. Returns path:line: text."""
        try:
            compiled = re.compile(pattern)
        except re.error as error:
            return f"bad regex: {error}"
        hits = []
        for path in self._paths._paths:
            if path_glob and not fnmatch(path, path_glob):
                continue
            lines = self._lines(path)
            for number, line in enumerate(lines or [], 1):
                if compiled.search(line):
                    hits.append(f"{path}:{number}: {line.strip()[:160]}")
        if not hits:
            return "no matches"
        more = f"\n... {len(hits) - 40} more" if len(hits) > 40 else ""
        return "\n".join(hits[:40]) + more

    def read_file(self, path: str, start: int = 1, end: int = 0) -> str:
        """Read a tracked file, whole or from start to end (1-based), at most 300 lines at a time."""
        lines = self._lines(path)
        if lines is None:
            return "not a tracked text file"
        end = min(end or len(lines), start + self.MAX_FILE_LINES - 1, len(lines))
        return "\n".join(f"{n}: {lines[n - 1]}" for n in range(max(start, 1), end + 1)) or "no such lines"

    def list_files(self, glob: str) -> str:
        """List tracked files whose path matches a glob, e.g. 'src/**/*.py' or '*hook*'."""
        found = [p for p in self._paths._paths if fnmatch(p, glob) or fnmatch(plb.PurePosixPath(p).name, glob)]
        return "\n".join(found[:100]) + (f"\n... {len(found) - 100} more" if len(found) > 100 else "") if found else "no tracked file matches"

    def _git(self, *args: str) -> str:
        result = subprocess.run(["git", "-C", str(self._root), *args], capture_output=True, text=True)
        lines = (result.stdout or result.stderr).splitlines()
        return "\n".join(lines[: self.MAX_GIT_LINES]) + (f"\n... {len(lines) - self.MAX_GIT_LINES} more lines" if len(lines) > self.MAX_GIT_LINES else "")

    def git_log(self, text: str, path: str = "") -> str:
        """Commits that added or removed this exact text (git log -S), newest first, optionally only for one path."""
        return self._git("log", "-S", text, "--oneline", "-n", "15", *(["--", path] if path else [])) or "no commits"

    def git_show(self, commit: str, path: str = "") -> str:
        """What one commit changed, optionally only in one path (git show)."""
        if not re.fullmatch(r"[0-9a-fA-F]{4,40}", commit):
            return "give a commit hash"
        return self._git("show", "--stat", "-p", commit, *(["--", path] if path else []))

    def tools(self, extended: bool = False) -> list[StructuredTool]:
        """The tracer's tools; extended adds what the judges had: regex grep, whole files, file globs and git history."""
        functions = [self.find_path, self.find_definition, self.search_code, self.list_definitions, self.read_lines]
        if extended:
            functions += [self.grep, self.read_file, self.list_files, self.git_log, self.git_show]
        return [StructuredTool.from_function(f) for f in functions]

    def run(self, name: str, args: dict) -> str:
        try:
            return getattr(self, name)(**args)
        except Exception as error:  # a bad argument from the model is its problem to fix
            return f"error: {error}"

    def check(self, location: "Location") -> bool:
        """Whether a reported location exists: tracked path, symbol in that file, line in range."""
        lines = self._lines(location.path)
        if lines is None:
            return False
        if location.line and not 1 <= location.line <= len(lines):
            return False
        if not self._quote_found(location, lines):
            return False
        if not location.symbol:
            return True
        defined = any(d.path == location.path for d in self._definitions.find(location.symbol)[1])
        return defined or bool(MentionSearch._word(location.symbol).search("\n".join(lines)))

    QUOTE_SLACK = 10

    def _quote_found(self, location: "Location", lines: list[str]) -> bool:
        """Whether every quoted line is in the file, near the reported line when there is one."""
        wanted = [re.sub(r"^\d+:\s?", "", line).strip() for line in location.quote.splitlines()]
        wanted = [line for line in wanted if line]
        if not wanted:
            return False
        window = lines
        if location.line:
            start = max(0, location.line - 1 - self.QUOTE_SLACK)
            window = lines[start : location.line + len(wanted) + self.QUOTE_SLACK]
        return all(any(line in have for have in window) for line in wanted)

    MAX_VIEW = 150

    def _quote_line(self, location: "Location", lines: list[str]) -> int:
        """The first line of the file that holds the quote's first line; 0 if none does."""
        wanted = [re.sub(r"^\d+:\s?", "", line).strip() for line in location.quote.splitlines()]
        first = next((w for w in wanted if w), "")
        return next((n for n, have in enumerate(lines, 1) if first and first in have), 0)

    def span(self, location: "Location", radius: int = 20) -> tuple[int, int]:
        """The lines the reviewer sees: the whole definition of the symbol (tree-sitter), else around the line or the quote."""
        lines = self._lines(location.path) or []
        line = location.line or self._quote_line(location, lines)
        if location.symbol:
            defined = [d for d in self._definitions.find(location.symbol)[1] if d.path == location.path and d.end_line]
            around = [d for d in defined if not line or d.line <= line <= d.end_line] or defined
            if around:
                d = min(around, key=lambda d: d.end_line - d.line)
                if d.end_line - d.line < self.MAX_VIEW:
                    return d.line, d.end_line
                line = line or d.line
        line = line or 1
        return max(1, line - radius), line + radius

    def context(self, location: "Location") -> str:
        """The code at a location, read from this copy, for the reviewer."""
        start, end = self.span(location)
        lines = self._lines(location.path)
        if lines is None:
            return "not a tracked text file"
        end = min(end, len(lines))
        shown = "\n".join(f"{n}: {lines[n - 1]}" for n in range(start, end + 1)) or "no such lines"
        names = self.names_used(location.path, start, end, "\n".join(lines[start - 1 : end]))
        return shown + ("\n\nNames used above, defined elsewhere:\n" + "\n".join(names) if names else "")

    MAX_NAMES = 8

    def names_used(self, path: str, start: int, end: int, text: str) -> list[str]:
        """Definitions (from tree-sitter) of names the shown code uses but does not define, constants first."""
        found = []
        for name in sorted(set(re.findall(r"\b[A-Za-z_][A-Za-z0-9_]{2,}\b", text)), key=lambda n: (not n.isupper(), n)):
            defined = [d for d in self._definitions.find(name)[1] if not (d.path == path and start <= d.line <= end)]
            if not defined or len(defined) > 3:
                continue
            d = next((d for d in defined if d.path == path), defined[0])
            lines = self._lines(d.path) or []
            body = lines[d.line - 1 : min(d.end_line or d.line, d.line + 4)]
            found.append(f"{d.path}:{d.line}: " + "\n    ".join(l.rstrip() for l in body))
            if len(found) >= self.MAX_NAMES:
                break
        return found

    def absent(self, name: str) -> bool:
        """Whether a path or name is gone: no tracked path matches it and nothing defines it."""
        name = name.strip().strip("`")
        if not name or self._paths.find(name)[1]:
            return False
        path_like = "/" in name or bool(re.search(r"\.\w{1,5}$", name)) and " " not in name
        if path_like:
            return True
        if self._definitions.find(name)[1]:
            return False
        mention = re.compile(r"(?<![\w.-])" + re.escape(name) + r"(?![\w-])")
        return not any(mention.search(text) for path, text in self._texts.items() if not SUPPORTING.search(path))

    def why_not(self, location: "Location") -> str:
        """Why a location fails the check, in words for the tracer; empty when it passes."""
        lines = self._lines(location.path)
        if lines is None:
            return "not a tracked text file"
        if location.line and not 1 <= location.line <= len(lines):
            return f"the file has {len(lines)} lines"
        if not self._quote_found(location, lines):
            return "the quote is not in the file near that line"
        if location.symbol and not self.check(location):
            return "the symbol is not in that file"
        return ""

    def verify(self, report: "Report | None") -> bool:
        """Whether a found report names at least one location and every one exists."""
        if report is not None and report.verdict == "absent":
            return bool(report.absent_names) and all(self.absent(n) for n in report.absent_names)
        if report is None or report.verdict not in {"found", "contradicted"} or not report.locations:
            return False
        return all(self.check(location) for location in report.locations)

In [ ]:
SUPPORTING = re.compile(r"(^|/)(tests?|fixtures?|evals?|examples?)/|(^|/)test_[^/]*$|^docs/|^\.okf/")
# A claim that something is gone; only those may be settled by the absence check.
ABSENCE = re.compile(r"\b(gone|absent|no longer|removed|deleted|dropped|retired|does not exist|do not exist|is not there)\b", re.I)


class Location(BaseModel):
    path: str = Field(description="A tracked file path, as the tools printed it.")
    symbol: str = Field(default="", description="The function, class, key or step name, if there is one.")
    line: int = Field(default=0, description="The line the claim lives at; 0 if none.")
    quote: str = Field(description="The lines that make the claim true, copied exactly from tool output, without line numbers.")


class Report(BaseModel):
    """Finish: say whether and where the claim lives in this repository."""

    verdict: Literal["found", "contradicted", "not_found", "not_code", "absent"]
    locations: list[Location] = Field(default=[], description="Where the claim lives, or the code that goes against it; only places seen in tool output.")
    absent_names: list[str] = Field(default=[], description="For absent: the paths or names the claim says are gone.")
    reason: str = Field(description="One sentence: why these locations, which part of the claim they show, or why nothing.")


class LocationReview(BaseModel):
    path: str
    verdict: Literal["supports", "partly", "contradicts", "unrelated"]
    reason: str = Field(description="What in the code shows it, or what is missing.")


class Review(BaseModel):
    """Your verdict: does the code shown do what the claim says?"""

    locations: list[LocationReview]
    verdict: Literal["supported", "partly", "contradicted", "not_supported"]
    reason: str = Field(description="One sentence.")


class Triage(BaseModel):
    """One claim's trace and review, with the checks between them."""

    chunk_id: str
    claim: str
    sentence: str = ""
    repo: str
    previous: list[str] = []
    report: Report | None
    tool_calls: list[str]
    verified: bool
    review: Review | None = None
    review_calls: int = 0
    error: str | None = None

    def _named(self, path: str) -> bool:
        text = self.claim + " " + self.sentence
        return path in text or plb.PurePosixPath(path).name in text

    def primary_locations(self) -> list[Location]:
        """Locations that can be where the claim lives: not a test or doc, unless the claim names it."""
        if self.report is None:
            return []
        return [l for l in self.report.locations if not SUPPORTING.search(l.path) or self._named(l.path)]

    @property
    def outcome(self) -> str:
        if self.report is None:
            return "gave up"
        if self.report.verdict == "absent":
            if not self.report.absent_names:
                return "made up"
            if not ABSENCE.search(self.claim):
                return "not_found"  # the claim says it exists; the tracer says it is gone
            return "found" if self.verified else "still there"
        if self.report.verdict not in {"found", "contradicted"}:
            return self.report.verdict
        if not self.verified:
            return "made up"
        if not self.primary_locations():
            return "only tests or docs"
        if self.review is None:
            return "not reviewed"
        return {"supported": "found", "partly": "partly", "contradicted": "contradicted", "not_supported": "rejected"}[self.review.verdict]

In [ ]:
TRIAGE_PROMPT = """
You trace one claim from a repository's design documents to the code that
makes it true, using the tools.

Find the code a maintainer would open to check the claim: the function,
class, config key, workflow step or script that does what the claim says.
Start from the names and words in the claim and its sentence, follow what you
find, and read the code before you report it.

Unless the claim is about it, none of these is where a claim lives: a test
that asserts it, a document that describes it, a file that only mentions a
name. If that is all you find, the answer is not_found.

not_found is a normal, useful answer: the claim may be out of date, and a
loosely related place hides that. When an earlier run's location is given,
check it first; if the code is no longer there, look for where it went, and
answer not_found if it is gone.

Finish by calling report:
- found: each location as a tracked path, the symbol when there is one, the
  line, and a quote: the lines that make the claim true, copied exactly from
  tool output without the line numbers. If only part of the claim holds,
  report that part and say in reason what is missing.
- contradicted: you found the code the claim is about, and it does
  something else: another value, another order, another name, a step the
  claim says is skipped. Report that code with quotes that show the
  difference, and say in reason what the claim says and what the code does.
  Missing code is not_found, not contradicted.
- not_found: nothing in this repository does what the claim says.
- absent: the claim says something is gone or does not exist. Always list in
  absent_names each path, name or identifier it says is gone, exactly as the
  claim writes it; the check confirms that no tracked path matches it, nothing
  defines it, and no code outside tests and docs mentions it.
- not_code: the claim is about people, process or outside tools, or it is
  history: it says what happened, was true or was measured in the past ("a
  fresh init wrote", "ADR 0010 chose", "both commits held"). History is not
  checked against today's code.

Use at most 8 tool calls.
""".strip()


REVIEW_PROMPT = """
Another agent says the code below is where a claim from a repository's design
documents comes true. Your job is to find out whether it is wrong.

For each location, read the code itself and decide:
- supports: this code does what the claim says, or the part of the claim
  that is about it.
- partly: it does some of that; say what is missing.
- contradicts: this is the code the claim is about, and it does something
  else (another value, order or name); say what differs.
- unrelated: it does not. A test that asserts the behaviour, a comment or
  string that names it, a call into code you cannot see, or code on a nearby
  topic are unrelated, unless the claim is about that very file.

Two kinds of claim are not_supported whatever the code shows:
- A claim about what a tool outside this repository does (a package
  manager, git, an editor, a spec reader). Only that tool's code would show
  it, and it is not here. A file here that configures the tool shows the
  configuration, not what the tool does with it; say so in the reason.
- A claim that refers to something it does not name ("both commits", "the
  earlier run", "it", "the two tiers"), when nothing shown says what it is.
  It cannot be checked as written; say what is unresolved.

Then the claim as a whole: supported when the locations together show all of
it, partly when they show some of it and you can say what is missing,
contradicted when the code the claim is about does something else,
not_supported when none supports it. Partly is a common, useful answer: a
claim with several parts is rarely shown whole by one place. Do not assume
that code you cannot see does the rest. When unsure, pick the weaker verdict.
""".strip()


# The split tracer (FEVER's pipeline): the tracer only gathers candidates; a judge reads each one alone.
SEARCH_NOTE = """
In this run another model judges each location you report, one at a time.
So do not decide whether the claim holds: report found with every place a
maintainer would open to check it, up to 5, each with its quote, including a
place that does only one part of it. Report not_found only if you found no
candidate at all.
""".strip()

JUDGE_PROMPT = """
You see one claim from a repository's design documents and one piece of that
repository's code, with the definitions of the names it uses. Decide what
this code shows about the claim:
- supports: this code does what the claim says.
- partly: it does one part of it; say which part, and what is missing.
- contradicts: this is the code the claim is about, and it does something
  else (another value, order or name); say what differs.
- unrelated: it does not show the claim. A test that asserts it, a comment or
  string that names it, a call into code you cannot see, or code on a nearby
  topic are unrelated, unless the claim is about that very file.

A claim about what a tool outside this repository does is unrelated to code
that only configures that tool. A claim that refers to something it does not
name ("both commits", "it") is unrelated unless this code says what it is.
""".strip()

In [ ]:
class VerificationPlan(BaseModel):
    """What code would prove the claim, where it probably is, and what to search for."""

    what_proves_it: str = Field(description="The code that would make the claim true: a function, step, key or check, and what it does.")
    likely_places: list[str] = Field(description="Folders or files from the map or the sentence where it probably is.")
    search_terms: list[str] = Field(description="Words, names and regexes to search for, including ones the sentence does not use.")


def chunk_text(chunk: dict) -> str:
    return "\n".join(f"[{n}] {s}" for n, s in enumerate(chunk["sentences"], 1))


class TriageState(TypedDict):
    context: ClaimContext
    previous: list[str]
    messages: Annotated[list[AnyMessage], operator.add]
    tool_calls: Annotated[list[str], operator.add]
    llm_calls: int
    report: Report | None
    verified: bool
    review: Review | None
    retries: int
    review_messages: list[AnyMessage]
    review_calls: int


class TriageGraph:
    """Trace a claim to code, check what the tracer reports, then have a reviewer try to reject it.

    trace <-> run_tools, until the tracer calls report or runs out of steps
    (force_report); then check (deterministic); then review when the check passes.
    """

    MAX_STEPS = 8
    RETRY_STEPS = 3

    def __init__(self, llm: ChatOpenAI, tools: CodeTools, repo: str, repo_map: "RepoMap | None" = None, quote_retry: bool = False,
                 extended_tools: bool = False, section: bool = False, max_steps: int = 8, plan: bool = False, support_retry: bool = False,
                 review_tools: bool = False, refs: bool = False, split: bool = False):
        self._tools = tools
        self._repo = repo
        self._map = repo_map
        self._quote_retry = quote_retry
        self._support_retry = support_retry
        self._section = section
        self._plan = plan
        self.MAX_STEPS = max_steps
        self._tracer = llm.bind_tools([*tools.tools(extended_tools), Report])
        self._planner = llm.with_structured_output(VerificationPlan, method="function_calling")
        self._final = llm.with_structured_output(Report, method="function_calling")
        self._reviewer = llm.with_structured_output(Review, method="function_calling")
        self._review_tools = review_tools
        self._refs = refs
        self._split = split
        self._judge = llm.with_structured_output(LocationReview, method="function_calling")
        reading = [tools.find_definition, tools.list_definitions, tools.read_lines, tools.read_file, tools.grep]
        self._review_agent = llm.bind_tools([*(StructuredTool.from_function(f) for f in reading), Review])
        self.graph = self._build()

    def _build(self):
        builder = StateGraph(TriageState)
        builder.add_node("trace", self._trace)
        builder.add_node("run_tools", self._run_tools)
        builder.add_node("force_report", self._force_report)
        builder.add_node("check", self._check)
        builder.add_node("review", self._review)
        builder.add_node("review_tools", self._run_review_tools)
        builder.add_node("retry", self._retry)
        if self._plan:
            builder.add_node("plan", self._make_plan)
            builder.add_edge(START, "plan")
            builder.add_edge("plan", "trace")
        else:
            builder.add_edge(START, "trace")
        builder.add_conditional_edges("trace", self._after_trace, ["run_tools", "trace", "force_report", "check"])
        builder.add_edge("run_tools", "trace")
        builder.add_edge("force_report", "check")
        builder.add_conditional_edges("check", self._after_check, ["review", "retry", END])
        builder.add_edge("retry", "trace")
        builder.add_conditional_edges("review", self._after_review, ["review_tools", "review", END])
        builder.add_edge("review_tools", "review")
        return builder.compile()

    def user_message(self, context: ClaimContext, previous: list[str]) -> str:
        chunk = chunks_by_id[context.tie.chunk_id]
        lines = [
            f"Claim: {context.tie.claim.claim}",
            f"Sentence it comes from: {context.tie.sentence}",
            f"Document: {chunk['concept']} > {chunk['section']}",
            f"Hint from the extractor (may be wrong): {context.tie.claim.how_to_check}",
        ]
        if previous:
            lines.append("An earlier run traced this claim to: " + "; ".join(previous))
        if context.level in {"previous", "sources"}:
            hints = sorted({loc for r in context.precise_resolutions() for loc in r.locations[:2]})
            lines.append("The document's context points at (it may be unrelated to this claim): " + ", ".join(hints))
        if self._refs:
            lines += self._relied_on(context)
        if self._section:
            lines.append("")
            lines.append("The whole section, for what the sentence refers to:")
            lines.append(chunk_text(chunk))
        if self.MAX_STEPS != 8:
            lines.append(f"You may use up to {self.MAX_STEPS} turns of tool calls, not 8.")
        if self._map is not None:
            lines.append("")
            lines.append("A map of the repository, to choose where to look (not evidence; read the code before you report it):")
            lines.append(self._map.for_claim(context.tie.claim.claim + " " + context.tie.sentence))
        return "\n".join(lines)

    @staticmethod
    def _relied_on(context: ClaimContext) -> list[str]:
        """The other sentences the extractor said this claim needs, by text."""
        texts = []
        for ref in context.tie.claim.refers_to:
            chunk_id, _, n = ref.rpartition("|")
            chunk = chunks_by_id.get(chunk_id)
            if chunk and n.isdigit() and 1 <= int(n) <= len(chunk["sentences"]):
                texts.append(f"- ({chunk['section'].split(' > ')[-1]}) {chunk['sentences'][int(n) - 1]}")
        return ["Sentences this claim relies on:", *texts] if texts else []

    async def _make_plan(self, state: TriageState) -> dict:
        """Node: before searching, say what code would prove the claim and what to search for."""
        plan = await self._planner.ainvoke(state["messages"] + [HumanMessage("Before you search, make a plan.")])
        text = (f"Your plan. Expect: {plan.what_proves_it}. Look in: {', '.join(plan.likely_places)}. "
                f"Search for: {', '.join(plan.search_terms)}. Now search, read the code, and report.")
        return {"messages": [HumanMessage(text)]}

    async def _trace(self, state: TriageState) -> dict:
        """Node: the tracer picks its next tool calls, or reports."""
        response = await self._tracer.ainvoke(state["messages"])
        update = {"messages": [response], "llm_calls": state["llm_calls"] + 1}
        for call in response.tool_calls:
            if call["name"] == "Report":
                update["report"] = Report.model_validate(call["args"])
        if not response.tool_calls:
            update["messages"].append(HumanMessage("Call report now."))
        return update

    def _after_trace(self, state: TriageState) -> str:
        if state["report"] is not None:
            return "check"
        if state["llm_calls"] >= self.MAX_STEPS + self.RETRY_STEPS * state["retries"]:
            return "force_report"
        if getattr(state["messages"][-1], "tool_calls", None):
            return "run_tools"
        return "trace"

    def _run_tools(self, state: TriageState) -> dict:
        """Node: run the tracer's tool calls against this copy of the repository."""
        calls = state["messages"][-1].tool_calls
        results = [ToolMessage(self._tools.run(c["name"], c["args"]), tool_call_id=c["id"]) for c in calls]
        return {"messages": results, "tool_calls": [c["name"] for c in calls]}

    async def _force_report(self, state: TriageState) -> dict:
        """Node: out of steps, so ask for a report from what the tracer has seen."""
        budget = HumanMessage("Your tool budget is spent. Report now, from what you saw.")
        return {"messages": [budget], "report": await self._final.ainvoke(state["messages"] + [budget])}

    def _check(self, state: TriageState) -> dict:
        """Node: every location must exist and its quote must be there."""
        return {"verified": self.verify(state["report"])}

    def _after_check(self, state: TriageState) -> str:
        triage = self._triage(state)
        if triage.report is not None and triage.report.verdict == "absent":
            return END
        if triage.verified and triage.primary_locations():
            return "review"
        if self._support_retry and state["retries"] < 1 and triage.verified and triage.report.verdict == "found":
            return "retry"
        if self._quote_retry and state["retries"] < 1 and triage.report is not None and triage.report.verdict == "found" and not triage.verified:
            return "retry"
        return END

    def _retry(self, state: TriageState) -> dict:
        """Node: the check failed, so tell the tracer which locations or quotes are not there and let it try again."""
        problems = [f"{l.path}:{l.line} {l.symbol}: {self._tools.why_not(l)}" for l in state["report"].locations if self._tools.why_not(l)]
        if problems:
            feedback = ("Your report did not pass the check: " + "; ".join(problems) +
                        ". Read those lines with read_lines and copy the quote exactly from the output, or report not_found.")
        else:
            feedback = ("You cited only tests or docs, and those are not where a claim lives. Use them to find the code "
                        "they test or describe (follow the names, imports and commands they use), read it, and report that "
                        "code; or report not_found.")
        last = state["messages"][-1]
        replies = [ToolMessage(feedback if c["name"] == "Report" else self._tools.run(c["name"], c["args"]), tool_call_id=c["id"])
                   for c in getattr(last, "tool_calls", None) or []]
        messages = replies if any(c["name"] == "Report" for c in getattr(last, "tool_calls", None) or []) else replies + [HumanMessage(feedback)]
        return {"messages": messages, "report": None, "verified": False, "retries": state["retries"] + 1}

    REVIEW_STEPS = 6
    REVIEW_TOOLS_NOTE = ("\n\nYou can read more of the repository with the tools: the code around a location, the "
                         "definitions of the names it uses, the functions it calls. Look before you call something "
                         "unrelated because it is out of view. Finish by calling Review.")

    def _review_message(self, context: ClaimContext, locations: list[Location]) -> str:
        parts = [f"Claim: {context.tie.claim.claim}", f"Sentence it comes from: {context.tie.sentence}"]
        if self._refs:
            parts += self._relied_on(context)
        for location in locations:
            parts.append(f"\n--- {location.path}:{location.line} {location.symbol}\n{self._tools.context(location)}")
        return "\n".join(parts)

    async def _judge_each(self, context: ClaimContext, locations: list[Location]) -> Review:
        """The split route: judge each location on its own, then combine (any support wins, then partly, then contradicts)."""
        head = f"Claim: {context.tie.claim.claim}\nSentence it comes from: {context.tie.sentence}"
        reviews = await asyncio.gather(*[
            self._judge.ainvoke([SystemMessage(JUDGE_PROMPT), HumanMessage(
                f"{head}\n\n--- {l.path}:{l.line} {l.symbol}\n{self._tools.context(l)}")]) for l in locations])
        reviews = [r.model_copy(update={"path": l.path}) for r, l in zip(reviews, locations)]
        verdicts = {r.verdict for r in reviews}
        verdict = ("supported" if "supports" in verdicts else "partly" if "partly" in verdicts
                   else "contradicted" if "contradicts" in verdicts else "not_supported")
        best = next(r for r in reviews if r.verdict == {"supported": "supports", "partly": "partly",
                                                          "contradicted": "contradicts", "not_supported": "unrelated"}[verdict])
        return Review(locations=reviews, verdict=verdict, reason=best.reason)

    async def _review(self, state: TriageState) -> dict:
        """Node: the reviewer sees only the claim and the code at each location, and tries to reject it."""
        locations = self._triage(state).primary_locations()
        if self._split:
            return {"review": await self._judge_each(state["context"], locations)}
        if not self._review_tools:
            messages = [SystemMessage(REVIEW_PROMPT), HumanMessage(self._review_message(state["context"], locations))]
            return {"review": await self._reviewer.ainvoke(messages)}
        messages = state["review_messages"] or [SystemMessage(REVIEW_PROMPT + self.REVIEW_TOOLS_NOTE),
                                                HumanMessage(self._review_message(state["context"], locations))]
        new = [] if state["review_messages"] else messages
        if state["review_calls"] >= self.REVIEW_STEPS:
            done = HumanMessage("Your reading budget is spent. Give your verdict now.")
            return {"review": await self._reviewer.ainvoke(messages + [done])}
        response = await self._review_agent.ainvoke(messages)
        for call in response.tool_calls:
            if call["name"] == "Review":
                return {"review": Review.model_validate(call["args"]), "review_messages": state["review_messages"] + new + [response]}
        nudge = [] if response.tool_calls else [HumanMessage("Call Review now.")]
        return {"review_messages": state["review_messages"] + new + [response] + nudge, "review_calls": state["review_calls"] + 1}

    def _after_review(self, state: TriageState) -> str:
        if state["review"] is not None:
            return END
        if getattr(state["review_messages"][-1], "tool_calls", None):
            return "review_tools"
        return "review"

    def _run_review_tools(self, state: TriageState) -> dict:
        """Node: run the reviewer's reads against this copy of the repository."""
        calls = state["review_messages"][-1].tool_calls
        results = [ToolMessage(self._tools.run(c["name"], c["args"]), tool_call_id=c["id"]) for c in calls]
        return {"review_messages": state["review_messages"] + results}

    def _triage(self, state: TriageState) -> Triage:
        context = state["context"]
        return Triage(
            chunk_id=context.tie.chunk_id,
            claim=context.tie.claim.claim,
            sentence=context.tie.sentence,
            repo=self._repo,
            previous=state["previous"],
            report=state["report"],
            tool_calls=state["tool_calls"],
            verified=state["verified"],
            review=state["review"],
            review_calls=state.get("review_calls", 0),
        )

    def verify(self, report: Report | None) -> bool:
        return self._tools.verify(report)

    async def triage(self, context: ClaimContext, previous: list[str]) -> Triage:
        system = TRIAGE_PROMPT + ("\n\n" + SEARCH_NOTE if self._split else "")
        messages = [SystemMessage(system), HumanMessage(self.user_message(context, previous))]
        start = {"context": context, "previous": previous, "messages": messages, "tool_calls": [],
                 "llm_calls": 0, "report": None, "verified": False, "review": None, "retries": 0,
                 "review_messages": [], "review_calls": 0}
        state = await self.graph.ainvoke(start, {"recursion_limit": 4 * (self.MAX_STEPS + self.RETRY_STEPS) + 2 * self.REVIEW_STEPS + 4})
        return self._triage(state)

In [ ]:
class TriageRunner:
    """Run the triage graph over many claims, a bounded number at a time, cached per claim and earlier location."""

    def __init__(self, graph: TriageGraph, root: plb.Path, model: str, repo: str, variant: str = ""):
        prompts = hashlib.sha256((TRIAGE_PROMPT + REVIEW_PROMPT + SEARCH_NOTE + JUDGE_PROMPT).encode()).hexdigest()[:8]
        self._directory = root / model.replace("/", "_") / (f"{prompts}-{variant}" if variant else prompts) / repo
        self._graph = graph

    def _path(self, context: ClaimContext, previous: list[str]) -> plb.Path:
        content = [context.tie.chunk_id, context.tie.claim.claim, context.tie.sentence, previous]
        return self._directory / (hashlib.sha256(json.dumps(content).encode()).hexdigest()[:16] + ".json")

    async def cached(self, context: ClaimContext, previous: list[str]) -> Triage:
        """Triage one claim, from the cache when this claim and earlier location were done before."""
        path = self._path(context, previous)
        if path.exists():
            cached = Triage.model_validate_json(path.read_text())
            if cached.report is not None:  # a claim that gave up is tried again
                return cached.model_copy(update={"verified": self._graph.verify(cached.report)})
        try:
            result = await self._graph.triage(context, previous)
        except Exception as error:  # a failed call is a result to count, not a crash
            return Triage(chunk_id=context.tie.chunk_id, claim=context.tie.claim.claim, repo="", report=None, tool_calls=[], verified=False, error=str(error)[:300])
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(result.model_dump_json())
        return result

    async def run(self, contexts: list[ClaimContext], previous: list[list[str]] | None = None) -> list[Triage]:
        """Triage many claims outside the audit graph, a bounded number at a time (the drift test uses this)."""
        limit = asyncio.Semaphore(MAX_CONCURRENT_CALLS)
        previous = previous or [[] for _ in contexts]

        async def one(context: ClaimContext, earlier: list[str]) -> Triage:
            async with limit:
                return await self.cached(context, earlier)

        return list(await asyncio.gather(*[one(c, p) for c, p in zip(contexts, previous)]))


def triage_llm(model: str = MODEL) -> ChatOpenAI:
    return ChatOpenAI(
        base_url="https://bifrost.lab.orangecluster.nl/v1",
        api_key=os.environ["BIFROST_CC_VIRTUAL_KEY"],  # type: ignore[arg-type]
        model=model,
        temperature=0,
        timeout=120,
        max_retries=3,
    )


FLASH = "gemini/gemini-3.8-flash"
_BASE = {"use_map": True, "quote_retry": True}
TRIAGE_VARIANTS = {
    "": {},
    "map": {"use_map": True},
    "map-retry": _BASE,
    # Each adds one lever to map-retry; the last adds them all.
    "flash": {**_BASE, "model": FLASH},
    "tools": {**_BASE, "extended_tools": True},
    "section": {**_BASE, "section": True},
    "turns20": {**_BASE, "max_steps": 20},
    "plan": {**_BASE, "plan": True},
    "all-lite": {**_BASE, "extended_tools": True, "section": True, "max_steps": 20, "plan": True},
    "all-flash": {**_BASE, "extended_tools": True, "section": True, "max_steps": 20, "plan": True, "model": FLASH},
    # all-lite plus a retry when it cites only tests or docs.
    "all-lite-retry": {**_BASE, "extended_tools": True, "section": True, "max_steps": 20, "plan": True, "support_retry": True},
    # all-lite again, to measure how much two runs of the same tracer differ.
    "all-lite-again": {**_BASE, "extended_tools": True, "section": True, "max_steps": 20, "plan": True},
    # all-lite plus the sentences a claim relies on; plus a reviewer that can read; plus both.
    "all-lite-refs": {**_BASE, "extended_tools": True, "section": True, "max_steps": 20, "plan": True, "refs": True},
    "all-lite-review": {**_BASE, "extended_tools": True, "section": True, "max_steps": 20, "plan": True, "review_tools": True},
    "all-new": {**_BASE, "extended_tools": True, "section": True, "max_steps": 20, "plan": True, "refs": True, "review_tools": True},
    # all-lite, but the tracer only gathers candidates and a judge reads each one alone (FEVER's split).
    "all-lite-split": {**_BASE, "extended_tools": True, "section": True, "max_steps": 20, "plan": True, "split": True},
}


def triage_runner_for(tools: CodeTools, repo: str, variant: str = "", repo_map: "RepoMap | None" = None) -> TriageRunner:
    """A runner for one variant of TRIAGE_VARIANTS: '' is the plain tracer, 'map-retry' the audit's default, the rest add one lever each."""
    options = dict(TRIAGE_VARIANTS[variant.split("@")[0]])
    model = options.pop("model", MODEL)
    use_map = options.pop("use_map", False)
    graph = TriageGraph(triage_llm(model), tools, repo, repo_map=repo_map if use_map else None, **options)
    return TriageRunner(graph, plb.Path("results") / "triage", model, repo, variant)

## Can code settle the claim? A typed question per claim

The extractor splits a sentence into claims; this step decides, for each
point or structural claim, whether code can settle it. Asking the extractor
to do both made it worse at both: a stricter "broad" rule in its prompt cost
four points of gold recall. This is CheckThat!'s Task 1 (is a claim worth
checking) as its own step.

It asks four typed yes-or-no questions (is it about the past, is it
sweeping, does it refer to something unnamed, is there nothing to look for)
and a fixed rule combines them, so a decision
model such as Clef (state plus typed questions in, probabilities out) can
answer them instead. One question asking for the kind directly let almost
everything through as checkable. The kinds:

- **checkable**: the tracer, lookup or universal check goes on.
- **broad**, **vague**, **unclear** (refers to something it does not name):
  traced anyway; a person sees the claim only if the trace does not settle
  it (found, partly, contradicted). Cutting these claims before the trace
  sent 74 claims to a person, and in a sample of 12 only about 5 belonged
  there: Flash reads "only", "one" and "each" as sweeping.
- **history**: not code. Outside tools are left to the extractor's
  external kind and to the reviewer.

Scored on the labels below (checkable judged claims kept of 38; not
checkable caught of 11; your ten; gold point or structural claims kept):

| Setup | Kept | Caught | Ten | Gold kept |
|---|---|---|---|---|
| Flash-Lite, one question for the kind | 37 | 4 | 5 | 163/165 |
| Flash-Lite, five questions | 32 | 5 | 6 | 156/173 |
| Flash, five questions | 25 | 9 | 7 | 154/173 |
| Flash-Lite, four (no "outside") | 32 | 3 | 5 | 169/173 |
| **Flash, four (no "outside")**, the default | 34 | 5 | 7 | 169/173 |

Your ten were looked at while choosing among these five, so the judged
sample and the gold set are the fairer test.

The examples in the prompt are made up; none comes from the scored claims.

In [ ]:
CHECK_PROMPT = """
You answer four yes-or-no questions about one claim from a repository's
design documents, so an auditor knows whether the repository's code can
settle it. The sentence the claim comes from and its section are shown. The
claim may have been rewritten in the present tense, so judge tense and
references by the sentence. Answer each question on its own; the auditor
combines the answers.

Examples (made up):
- past: "the first release shipped two commands", "the check skipped those
  links", "a fresh install wrote four files".
- sweeping: "the tool covers every task a user needs", "the system has three
  layers", "every part works on each supported platform". Not sweeping:
  "every workflow pins its actions", because the workflows are a list to go
  through.
- unnamed: "both changes keep the entry", "it reads the file first", when
  the section does not say what.
- nothing to look for: "the design stays simple", "the code is easy to
  change".
""".strip()

# Outside tools are left to the extractor's external kind and the reviewer: asking "is the actor outside the
# repository?" here threw out checkable claims on both models (scored below), since almost every claim
# involves an outside tool somewhere (a CI service runs the workflows, a package manager installs the package).
CHECK_KINDS = ["checkable", "broad", "vague", "history", "unclear"]
TO_A_PERSON = {"broad", "vague", "unclear"}
NOT_CODE = {"history"}
CHECK_MODEL = FLASH  # Flash-Lite kept fewer checkable claims and caught fewer of the rest (scored below)


class Checkability(BaseModel):
    """Four typed questions about one claim; kind combines the answers."""

    past: bool = Field(description="Does the sentence say what happened, was done, decided or measured in the past, rather than what is true now?")
    sweeping: bool = Field(description="Would settling it mean judging the whole repository, or confirming that a set is complete (a count of parts, every step, every platform) with no list in the repository to go through?")
    unnamed: bool = Field(description="Does it refer to something it does not name, and the section does not say what?")
    nothing_to_look_for: bool = Field(description="Does no reading of it name anything to open, run or search for?")
    reason: str = Field(description="One sentence.")
    error: str | None = None

    @property
    def kind(self) -> str:
        """The first answer that rules code out wins: history, unclear, vague, broad; else checkable."""
        for flag, kind in [(self.past, "history"), (self.unnamed, "unclear"),
                           (self.nothing_to_look_for, "vague"), (self.sweeping, "broad")]:
            if flag:
                return kind
        return "checkable"


class CheckabilityRunner:
    """Ask the checkability question for one claim, cached per prompt, claim and section."""

    def __init__(self, llm: ChatOpenAI, root: plb.Path, model: str):
        prompt = hashlib.sha256(CHECK_PROMPT.encode()).hexdigest()[:8]
        self._directory = root / model.replace("/", "_") / prompt
        self._llm = llm.with_structured_output(Checkability, method="function_calling")

    @staticmethod
    def message(claim: str, chunk: dict, n: int) -> str:
        return "\n".join([f"Claim: {claim}", f"From sentence [{n}] of {chunk['concept']} > {chunk['section']}:", chunk_text(chunk)])

    async def cached(self, claim: str, chunk: dict, n: int) -> Checkability:
        content = [chunk["id"], claim, chunk["sentences"]]
        path = self._directory / (hashlib.sha256(json.dumps(content).encode()).hexdigest()[:16] + ".json")
        if path.exists():
            return Checkability.model_validate_json(path.read_text())
        try:
            result = await self._llm.ainvoke([SystemMessage(CHECK_PROMPT), HumanMessage(self.message(claim, chunk, n))])
            if result is None:
                raise ValueError("the answer could not be read")
        except Exception as error:  # a failed call leaves the claim checkable, so nothing is dropped
            return Checkability(past=False, sweeping=False, unnamed=False, nothing_to_look_for=False, reason="", error=str(error)[:300])
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(result.model_dump_json())
        return result

    async def run(self, items: list[tuple[str, dict, int]]) -> list[Checkability]:
        limit = asyncio.Semaphore(MAX_CONCURRENT_CALLS)

        async def one(item):
            async with limit:
                return await self.cached(*item)

        return list(await asyncio.gather(*[one(i) for i in items]))

## Repo map: where to look, for a model

A model that has to find code for a claim that names nothing needs to know
what the repository holds. The map lists the top-level folders, then the
files whose path or definitions share words with the claim, each with the
names it defines. It is ranked and cut to a budget, so it fits any size of
repository. It tells a model where to look; it is not evidence.

In [ ]:
class RepoMap:
    """A short, ranked map of a repository for one claim: folders, then matching files and what they define."""

    _word = re.compile(r"[a-z][a-z0-9]{2,}")
    _split = re.compile(r"[_\-./`:()\[\],]+|(?<=[a-z])(?=[A-Z])")
    _stop = {"the", "and", "for", "with", "that", "this", "from", "its", "are", "not", "has", "each", "every",
             "all", "never", "only", "one", "into", "when", "which", "their", "them", "but", "any", "own", "does"}

    def __init__(self, paths: list[str], definitions: DefinitionIndex, max_files: int = 25, max_names: int = 12):
        self._paths = sorted(p for p in paths if not SUPPORTING.search(p))
        self._max_files = max_files
        self._max_names = max_names
        self._by_file: dict[str, list[Definition]] = {}
        for found in definitions._by_name.values():
            for d in found:
                self._by_file.setdefault(d.path, []).append(d)
        self._folders = Counter(p.split("/")[0] if "/" in p else "(root)" for p in self._paths)
        self._file_words = {p: self._words(p) for p in self._paths}
        self._name_words = {p: self._words(" ".join(d.name for d in ds)) for p, ds in self._by_file.items()}

    def _words(self, text: str) -> set[str]:
        words = {w.rstrip("s") for w in self._word.findall(self._split.sub(" ", text).lower())}
        return words - self._stop

    def _score(self, path: str, words: set[str]) -> int:
        return 3 * len(words & self._file_words[path]) + len(words & self._name_words.get(path, set()))

    def _names(self, path: str) -> list[str]:
        found = self._by_file.get(path, [])
        code_first = sorted(found, key=lambda d: (not re.search(r"function|class|method", d.node_type), d.line))
        return list(dict.fromkeys(d.name for d in code_first))[: self._max_names]

    def for_claim(self, text: str) -> str:
        """The map for one claim, as plain lines."""
        words = self._words(text)
        scored = sorted(((self._score(p, words), p) for p in self._paths), key=lambda x: (-x[0], x[1]))
        top = [p for score, p in scored[: self._max_files] if score > 0]
        lines = ["Top-level folders (files): " + ", ".join(f"{f} ({n})" for f, n in self._folders.most_common(25))]
        lines.append("Files whose path or definitions share words with the claim, best first, with what they define:")
        for p in top:
            names = self._names(p)
            lines.append(f"- {p}: {', '.join(names)}" if names else f"- {p}")
        return "\n".join(lines)

## Universal claims: every instance, or none

"Every workflow pins its actions" or "kmctl never writes outside `.okf/`"
cannot be proven by finding one place; every instance has to be checked.
The claim branch sends claims the extractor marked `all` or `none` here
instead of to the tracer.

1. **plan**: the model writes broad searches to learn how this repository
   does the thing (a shared helper, a wrapper, the library used directly).
2. **probe**: code runs those searches and summarises the hits per file.
3. **candidates**: from the probe, the model writes the searches that find
   every instance, and says what an instance must show to comply (for
   `none`: what makes it break the claim).
4. **harvest**: code runs them over every tracked text file. Tests and docs
   are left out unless the claim names them.
5. Few instances (up to `UNIVERSAL_THRESHOLD`): `Send` one check per batch of
   instances, in parallel. Many: the model writes an **ast-grep** rule pair
   (every instance, and the ones that break), code runs it, and the model
   checks a sample of what it flagged and what it let through. A rule that
   does not parse, matches nothing, or disagrees with the sample goes back
   with the reason, at most `MAX_RULE_ROUNDS` times; after that the claim is
   "could not check".
6. **gather**: holds, broken (with every breaking file and line), no
   instances, or could not check. The searches and rule are kept, so a later
   run can repeat them without the model.

In [ ]:
import shutil
import sys
import tempfile

AST_GREP = shutil.which("ast-grep") or str(plb.Path(sys.executable).parent / "ast-grep")
UNIVERSAL_THRESHOLD = 30
SITE_BATCH = 10
MAX_RULE_ROUNDS = 3
RULE_SAMPLE = 3
AST_GREP_LANGUAGES = {".py": "python", ".sh": "bash", ".bash": "bash", ".yaml": "yaml", ".yml": "yaml",
                      ".json": "json", ".js": "javascript", ".ts": "typescript", ".go": "go", ".rs": "rust",
                      ".java": "java", ".rb": "ruby", ".tf": "hcl", ".html": "html", ".css": "css"}


class SearchPlan(BaseModel):
    """First look: broad searches that show how this repository does what the claim is about."""

    subject: str = Field(description="What one instance is, e.g. 'a place that writes a file', 'a workflow step that uses an action'.")
    probes: list[str] = Field(description="2 to 5 broad Python regexes to find where the subject shows up; case-sensitive.")


class Candidates(BaseModel):
    """The searches that find every instance, and what an instance must show."""

    idiom: str = Field(description="How this repository does the subject, from the probe hits: the helper, wrapper or call it uses.")
    patterns: list[str] = Field(description="Python regexes matched against each line of file content (not file names); every matching line is one instance to check. Prefer too many matches to too few.")
    complies: str = Field(description="What an instance must show for the claim to hold. For a 'none' claim: what makes an instance break it.")


class Site(BaseModel):
    path: str
    line: int
    text: str


class SiteCheck(BaseModel):
    index: int = Field(description="The instance's number as shown.")
    verdict: Literal["complies", "breaks", "not_an_instance"]
    reason: str = Field(description="What in the code shows it; short.")


class SiteChecks(BaseModel):
    """A verdict for every instance shown."""

    checks: list[SiteCheck]


class RuleDraft(BaseModel):
    """An ast-grep rule pair for one language."""

    language: str = Field(description="One ast-grep language, e.g. python, bash, yaml.")
    instances: str = Field(description="YAML for ast-grep's `rule:` mapping (without the `rule:` key) that matches every instance.")
    breaks: str = Field(description="YAML for the `rule:` mapping that matches only the instances that break the claim.")
    ignores: list[str] = Field(default=[], description="Globs of files the breaks rule skips, e.g. the one module allowed to do it.")


class UniversalResult(BaseModel):
    """One universal claim's check, with everything a later run needs to repeat it."""

    chunk_id: str
    claim: str
    quantifier: str
    repo: str
    plan: SearchPlan | None = None
    candidates: Candidates | None = None
    sites: list[Site] = []
    route: Literal["", "per instance", "rule"] = ""
    checks: list[SiteCheck] = []
    rule: RuleDraft | None = None
    rule_rounds: int = 0
    rule_notes: list[str] = []
    breaking: list[Site] = []
    verdict: Literal["holds", "broken", "no instances", "could not check"] = "could not check"
    llm_calls: int = 0
    error: str | None = None

In [ ]:
PLAN_PROMPT = """
You check a universal claim from a repository's design documents: a claim
about every instance of something ("all", "each") or about no instance
("never", "nothing", "only X"). Proving it needs every instance, so start by
learning how this repository does the thing.

Say what one instance is, then write 2 to 5 broad Python regexes that find
where it shows up. A map of the repository is given; use it to pick words
and names. Broad is right: the next step narrows it.
""".strip()

CANDIDATES_PROMPT = """
You see a universal claim and where broad searches hit in the repository.
First say how this repository does the subject (a helper, wrapper or call
everyone uses, or the library directly). Then write the regexes that find
every instance, so nothing escapes the check: if code goes through a helper,
search for the helper's callers and for direct uses that skip it. Then say
what an instance must show for the claim to hold; for a 'none' claim, what
makes an instance break it.
""".strip()

SITE_PROMPT = """
You check instances of a universal claim from a repository's design
documents, one verdict per instance:
- complies: this instance does what the claim requires.
- breaks: this instance goes against the claim. Be sure: quote the reason.
- not_an_instance: the search caught something that is not what the claim is
  about (a comment, a string, a different thing with the same name).
Judge only the code shown. Say an instance breaks a rule that something must
be present only when the code shown covers the whole instance; if it may be
further down, say complies and note that you could not see it all.
""".strip()

RULE_PROMPT = """
Too many instances to check one by one, so write an ast-grep rule pair for
the claim: `instances` matches every instance of the subject, `breaks`
matches only the instances that break the claim. Each is the YAML body of
ast-grep's `rule:` key, e.g.

  pattern: httpx.Client($$$ARGS)

or

  all:
    - pattern: httpx.Client($$$ARGS)
    - not:
        has:
          pattern: limits=$L
          stopBy: end

Use only ast-grep rule keys (pattern, kind, regex, has, inside, follows,
precedes, all, any, not, stopBy), as valid YAML. When some files may do what
the claim forbids (the one module allowed to write, say), list them in
ignores instead of encoding paths in the rule. Pick the language most
instances are in.
If you get feedback on an earlier rule, fix what it names.
""".strip()

In [ ]:
class UniversalTools:
    """Deterministic search for the universal route: regex harvests over text files, and ast-grep runs."""

    MAX_SITES = 2000

    def __init__(self, tools: CodeTools):
        self._tools = tools

    @staticmethod
    def _compile(patterns: list[str]) -> tuple[list[re.Pattern], list[str]]:
        compiled, errors = [], []
        for p in patterns:
            try:
                compiled.append(re.compile(p))
            except re.error as error:
                errors.append(f"{p!r}: {error}")
        return compiled, errors

    def harvest(self, patterns: list[str], named: str = "") -> tuple[list[Site], list[str]]:
        """Every line any pattern matches; tests and docs only when ``named`` mentions them."""
        compiled, errors = self._compile(patterns)
        sites = []
        for path, content in self._tools._texts.items():
            if SUPPORTING.search(path) and path not in named and plb.PurePosixPath(path).name not in named:
                continue
            for number, line in enumerate(content.splitlines(), 1):
                if any(p.search(line) for p in compiled):
                    sites.append(Site(path=path, line=number, text=line.strip()[:200]))
                    if len(sites) >= self.MAX_SITES:
                        return sites, errors
        return sites, errors

    def probe(self, patterns: list[str]) -> str:
        """A summary of where broad patterns hit: counts per file, then a few lines."""
        sites, errors = self.harvest(patterns)
        per_file = Counter(s.path for s in sites)
        lines = [f"{len(sites)} matching lines in {len(per_file)} files."] + [f"bad regex {e}" for e in errors]
        lines += [f"  {n:4} {p}" for p, n in per_file.most_common(30)]
        lines.append("First lines:")
        lines += [f"  {s.path}:{s.line}: {s.text[:140]}" for s in sites[:40]]
        return "\n".join(lines)

    BLOCK_LINES = 60

    def show(self, site: Site, radius: int = 8) -> str:
        """The code around an instance: the whole block it opens, up to 60 lines, else a few lines either side."""
        lines = self._tools._lines(site.path) or []
        start = max(1, site.line - 3)
        end = min(len(lines), site.line + radius)
        if 0 < site.line <= len(lines) and lines[site.line - 1].rstrip().endswith(("{", "(", "[", ":")):
            indent = len(lines[site.line - 1]) - len(lines[site.line - 1].lstrip())
            end = site.line
            while end < min(len(lines), site.line + self.BLOCK_LINES):
                end += 1
                text = lines[end - 1]
                if text.strip() and len(text) - len(text.lstrip()) <= indent:
                    break
        return self._tools.read_lines(site.path, start, end)

    def ast_grep(self, language: str, rule_yaml: str, ignores: list[str] | None = None) -> tuple[list[Site], str]:
        """Run one rule over the repository; the matches, or an error to hand back to the model."""
        import yaml

        try:
            rule = yaml.safe_load(rule_yaml)
        except yaml.YAMLError as error:
            return [], f"the rule is not valid YAML: {error}"
        if not isinstance(rule, dict):
            return [], "the rule must be a YAML mapping"
        document = {"id": "claim", "language": language, "severity": "info", "message": "m", "rule": rule}
        if ignores:
            document["ignores"] = ignores
        with tempfile.NamedTemporaryFile("w", suffix=".yml", delete=False) as handle:
            yaml.safe_dump(document, handle)
        result = subprocess.run(
            [AST_GREP, "scan", "--rule", handle.name, "--json=stream", str(self._tools._root)],
            capture_output=True, text=True,
        )
        os.unlink(handle.name)
        if result.returncode not in (0, 1) or (result.stderr.strip() and not result.stdout.strip()):
            return [], "ast-grep refused the rule: " + result.stderr.strip()[:500]
        sites = []
        for line in result.stdout.splitlines():
            match = json.loads(line)
            path = plb.Path(match["file"]).resolve().relative_to(self._tools._root.resolve()).as_posix()
            if path in self._tools._texts and not SUPPORTING.search(path):
                number = match["range"]["start"]["line"] + 1
                sites.append(Site(path=path, line=number, text=match["lines"].strip()[:200]))
        return sites, ""

In [ ]:
class UniversalState(TypedDict):
    context: ClaimContext
    plan: SearchPlan | None
    probe: str
    candidates: Candidates | None
    sites: list[Site]
    checks: Annotated[list[SiteCheck], operator.add]
    llm_calls: Annotated[int, operator.add]
    rule: RuleDraft | None
    rule_rounds: int
    rule_notes: list[str]
    breaking: list[Site]
    rule_done: bool
    error: str | None


class UniversalGraph:
    """Check a claim about every instance, or about none: plan, probe, candidates, harvest, then per-instance checks or an ast-grep rule."""

    def __init__(self, llm: ChatOpenAI, tools: CodeTools, repo_map: RepoMap, repo: str, threshold: int = UNIVERSAL_THRESHOLD):
        self._search = UniversalTools(tools)
        self._map = repo_map
        self._repo = repo
        self._threshold = threshold
        self._planner = llm.with_structured_output(SearchPlan, method="function_calling")
        self._candidate_writer = llm.with_structured_output(Candidates, method="function_calling")
        self._checker = llm.with_structured_output(SiteChecks, method="function_calling")
        self._rule_writer = llm.with_structured_output(RuleDraft, method="function_calling")
        self.graph = self._build()

    def _build(self):
        builder = StateGraph(UniversalState)
        for name in ["plan", "probe", "candidates", "harvest", "check_sites", "write_rule", "run_rule", "gather"]:
            builder.add_node(name, getattr(self, "_" + name))
        builder.add_edge(START, "plan")
        builder.add_edge("plan", "probe")
        builder.add_edge("probe", "candidates")
        builder.add_edge("candidates", "harvest")
        builder.add_conditional_edges("harvest", self._after_harvest, ["check_sites", "write_rule", "gather"])
        builder.add_edge("check_sites", "gather")
        builder.add_edge("write_rule", "run_rule")
        builder.add_conditional_edges("run_rule", self._after_rule, ["write_rule", "gather"])
        builder.add_edge("gather", END)
        return builder.compile()

    @staticmethod
    def _claim_lines(context: ClaimContext) -> list[str]:
        return [
            f"Claim ({context.tie.claim.quantifier}): {context.tie.claim.claim}",
            f"Sentence it comes from: {context.tie.sentence}",
        ]

    async def _plan(self, state: UniversalState) -> dict:
        """Node: what one instance is, and broad searches to see how the repository does it."""
        context = state["context"]
        text = "\n".join(self._claim_lines(context) + ["", self._map.for_claim(context.tie.claim.claim + " " + context.tie.sentence)])
        plan = await self._planner.ainvoke([SystemMessage(PLAN_PROMPT), HumanMessage(text)])
        return {"plan": plan, "llm_calls": 1}

    def _probe(self, state: UniversalState) -> dict:
        """Node: run the broad searches and summarise where they hit."""
        return {"probe": self._search.probe(state["plan"].probes)}

    async def _candidates(self, state: UniversalState) -> dict:
        """Node: from the probe, the searches that find every instance, and what complying looks like."""
        text = "\n".join(self._claim_lines(state["context"]) + [f"One instance: {state['plan'].subject}", "", "Probe hits:", state["probe"]])
        return {"candidates": await self._candidate_writer.ainvoke([SystemMessage(CANDIDATES_PROMPT), HumanMessage(text)]), "llm_calls": 1}

    def _harvest(self, state: UniversalState) -> dict:
        """Node: every line the candidate searches match, outside tests and docs unless the claim names them."""
        context = state["context"]
        sites, errors = self._search.harvest(state["candidates"].patterns, named=context.tie.claim.claim + " " + context.tie.sentence)
        return {"sites": sites, "error": "; ".join(errors) or None}

    def _after_harvest(self, state: UniversalState) -> list[Send] | str:
        sites = state["sites"]
        if not sites:
            return "gather"
        if len(sites) > self._threshold:
            return "write_rule"
        batches = [list(enumerate(sites))[i : i + SITE_BATCH] for i in range(0, len(sites), SITE_BATCH)]
        return [Send("check_sites", {"context": state["context"], "candidates": state["candidates"], "batch": b}) for b in batches]

    async def _check_batch(self, context: ClaimContext, candidates: Candidates, batch: list[tuple[int, Site]]) -> list[SiteCheck]:
        parts = self._claim_lines(context) + [f"To comply (or, for a 'none' claim, to break): {candidates.complies}"]
        for index, site in batch:
            parts.append(f"\n--- instance {index}: {site.path}:{site.line}\n{self._search.show(site)}")
        result = await self._checker.ainvoke([SystemMessage(SITE_PROMPT), HumanMessage("\n".join(parts))])
        wanted = {index for index, _ in batch}
        return [c for c in result.checks if c.index in wanted]

    async def _check_sites(self, task: dict) -> dict:
        """Node (one per batch, in parallel): a verdict for each instance in the batch."""
        checks = await self._check_batch(task["context"], task["candidates"], task["batch"])
        return {"checks": checks, "llm_calls": 1}

    def _languages(self, sites: list[Site]) -> Counter:
        return Counter(AST_GREP_LANGUAGES.get(plb.PurePosixPath(s.path).suffix, "") for s in sites) - Counter({"": 10**9})

    async def _write_rule(self, state: UniversalState) -> dict:
        """Node: an ast-grep rule pair, with the feedback from earlier rounds."""
        context, candidates = state["context"], state["candidates"]
        languages = self._languages(state["sites"])
        parts = self._claim_lines(context) + [
            f"How the repository does it: {candidates.idiom}",
            f"To comply (or, for a 'none' claim, to break): {candidates.complies}",
            f"{len(state['sites'])} instances found by regex, by ast-grep language: {dict(languages) or 'none supported'}",
            "Some of them:",
        ] + [f"  {s.path}:{s.line}: {s.text[:140]}" for s in state["sites"][:15]]
        if state["rule_notes"]:
            parts += ["", "Feedback on your earlier rules:"] + [f"- {note}" for note in state["rule_notes"]]
        rule = await self._rule_writer.ainvoke([SystemMessage(RULE_PROMPT), HumanMessage("\n".join(parts))])
        return {"rule": rule, "rule_rounds": state["rule_rounds"] + 1, "llm_calls": 1}

    async def _run_rule(self, state: UniversalState) -> dict:
        """Node: run the rule pair, test it against the regex harvest and a checked sample, and keep or reject it."""
        rule, context, candidates = state["rule"], state["context"], state["candidates"]
        instances, error = self._search.ast_grep(rule.language, rule.instances)
        if error:
            return {"rule_notes": state["rule_notes"] + [f"instances rule: {error}"]}
        breaking, error = self._search.ast_grep(rule.language, rule.breaks, rule.ignores)
        if error:
            return {"rule_notes": state["rule_notes"] + [f"breaks rule: {error}"]}
        if not instances:
            return {"rule_notes": state["rule_notes"] + ["the instances rule matched nothing; look at the instances listed and match them"]}
        flagged = random.Random(0).sample(breaking, min(RULE_SAMPLE, len(breaking)))
        breaking_keys = {(s.path, s.line) for s in breaking}
        passed = [s for s in instances if (s.path, s.line) not in breaking_keys]
        passed = random.Random(1).sample(passed, min(RULE_SAMPLE, len(passed)))
        sample = list(enumerate(flagged + passed))
        checks = await self._check_batch(context, candidates, sample)
        by_index = {c.index: c for c in checks}
        wrong = []
        for index, site in sample:
            check = by_index.get(index)
            expected = "breaks" if index < len(flagged) else "complies"
            if check is not None and check.verdict != expected and check.verdict != "not_an_instance":
                wrong.append(f"{site.path}:{site.line} was {'flagged' if expected == 'breaks' else 'let through'}, but a check says it {check.verdict}: {check.reason}")
        if wrong:
            return {"rule_notes": state["rule_notes"] + wrong, "llm_calls": 1}
        return {"breaking": breaking, "rule_done": True, "checks": checks, "llm_calls": 1}

    def _after_rule(self, state: UniversalState) -> str:
        if state["rule_done"] or state["rule_rounds"] >= MAX_RULE_ROUNDS:
            return "gather"
        return "write_rule"

    def _gather(self, state: UniversalState) -> dict:
        """Node: the verdict is gathered in ``result``; nothing to update."""
        return {}

    def result(self, state: UniversalState) -> UniversalResult:
        """Turn the final state into a result: holds, broken, no instances or could not check."""
        context = state["context"]
        sites = state["sites"]
        checks = state["checks"]
        route = "" if not sites else ("rule" if len(sites) > self._threshold else "per instance")
        if route == "per instance":
            breaking = [sites[c.index] for c in checks if c.verdict == "breaks" and 0 <= c.index < len(sites)]
            instances = [c for c in checks if c.verdict != "not_an_instance"]
            verdict = "broken" if breaking else ("holds" if instances else "no instances")
        elif route == "rule":
            breaking = state["breaking"]
            verdict = ("broken" if breaking else "holds") if state["rule_done"] else "could not check"
        else:
            breaking, verdict = [], "no instances"
        return UniversalResult(
            chunk_id=context.tie.chunk_id, claim=context.tie.claim.claim, quantifier=context.tie.claim.quantifier,
            repo=self._repo, plan=state["plan"], candidates=state["candidates"], sites=sites, route=route,
            checks=checks, rule=state["rule"], rule_rounds=state["rule_rounds"], rule_notes=state["rule_notes"],
            breaking=breaking, verdict=verdict, llm_calls=state["llm_calls"], error=state["error"],
        )

    async def check(self, context: ClaimContext) -> UniversalResult:
        start = {"context": context, "plan": None, "probe": "", "candidates": None, "sites": [], "checks": [],
                 "llm_calls": 0, "rule": None, "rule_rounds": 0, "rule_notes": [], "breaking": [], "rule_done": False, "error": None}
        state = await self.graph.ainvoke(start, {"recursion_limit": 40})
        return self.result(state)

In [ ]:
class UniversalRunner:
    """Run the universal graph, cached per claim, prompts, repository and threshold."""

    def __init__(self, graph: UniversalGraph, root: plb.Path, model: str, repo: str):
        prompts = hashlib.sha256((PLAN_PROMPT + CANDIDATES_PROMPT + SITE_PROMPT + RULE_PROMPT).encode()).hexdigest()[:8]
        self._directory = root / model.replace("/", "_") / f"{prompts}-t{graph._threshold}" / repo
        self._graph = graph

    def _path(self, context: ClaimContext) -> plb.Path:
        content = [context.tie.chunk_id, context.tie.claim.claim, context.tie.sentence, context.tie.claim.quantifier]
        return self._directory / (hashlib.sha256(json.dumps(content).encode()).hexdigest()[:16] + ".json")

    async def cached(self, context: ClaimContext) -> UniversalResult:
        path = self._path(context)
        if path.exists():
            return UniversalResult.model_validate_json(path.read_text())
        try:
            result = await self._graph.check(context)
        except Exception as error:  # a failed call is a result to count, not a crash
            return UniversalResult(chunk_id=context.tie.chunk_id, claim=context.tie.claim.claim,
                                   quantifier=context.tie.claim.quantifier, repo=self._graph._repo, error=str(error)[:300])
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(result.model_dump_json())
        return result

    async def run(self, contexts: list[ClaimContext]) -> list[UniversalResult]:
        limit = asyncio.Semaphore(MAX_CONCURRENT_CALLS)

        async def one(context: ClaimContext) -> UniversalResult:
            async with limit:
                return await self.cached(context)

        return list(await asyncio.gather(*[one(c) for c in contexts]))


def universal_runner_for(tools: CodeTools, repo_map: RepoMap, repo: str, threshold: int = UNIVERSAL_THRESHOLD) -> UniversalRunner:
    return UniversalRunner(UniversalGraph(triage_llm(), tools, repo_map, repo, threshold), plb.Path("results") / "universal", MODEL, repo)

## The audit graph

One LangGraph graph from chunks to triaged claims:

- `START` sends one `extract` per chunk (cached by chunk key);
- `gather` waits for every chunk, then sends one `claim` branch per point or structural claim;
- a claim branch is its own graph: `lookup` ties it without a model, and `triage` (the trace, check and review graph above) runs only when the lookup does not tie it precisely (cached by claim).

`max_concurrency` bounds how many branches run at once.

In [ ]:
UNTIED = {"folder or glob", "docs only", "weak", "unresolved", "none"}
# A tie through context (the sentence before, the concept's sources) is a guess, so the agent checks it too.
TO_TRIAGE = UNTIED | {"previous", "sources"}


class ClaimResult(BaseModel):
    """One claim's way through the graph: its ties and, when they fell short, its triage."""

    order: tuple[int, int]
    context: ClaimContext
    triage: Triage | None = None
    universal: UniversalResult | None = None
    check: Checkability | None = None


class ClaimState(TypedDict):
    order: tuple[int, int]
    chunk_id: str
    claim: Claim
    context: ClaimContext | None
    triage: Triage | None
    universal: UniversalResult | None
    check: Checkability | None


class AuditState(TypedDict):
    chunks: list[Chunk]
    extracted: Annotated[list[ChunkResult], operator.add]
    claims: Annotated[list[ClaimResult], operator.add]


class AuditGraph:
    """The whole audit as one graph.

    START sends one extract per chunk; gather waits for them all; then one claim
    branch per claim. A claim branch is its own graph: lookup ties the claim
    without a model; checkable asks whether code can settle it at all (a history
    claim stops; a broad, vague or unclear one is traced, and waits for a
    person only if the trace does not settle it); a claim about every instance or none goes to the
    universal graph; any other claim the lookup does not tie precisely goes to
    triage (the trace, check and review graph).
    """

    def __init__(self, extraction: ExtractionRunner, ties: ClaimTier, context: ContextTier, triage: TriageRunner, universal: UniversalRunner,
                 checkability: CheckabilityRunner):
        self._checkability = checkability
        self._extraction = extraction
        self._ties = ties
        self._context = context
        self._triage = triage
        self._universal = universal
        self.claim_graph = self._build_claim_graph()
        self.graph = self._build()

    def _build(self):
        builder = StateGraph(AuditState)
        builder.add_node("extract", self._extract)
        builder.add_node("gather", self._gather)
        builder.add_node("claim", self._claim)
        builder.add_conditional_edges(START, self._per_chunk, ["extract"])
        builder.add_edge("extract", "gather")
        builder.add_conditional_edges("gather", self._per_claim, ["claim"])
        builder.add_edge("claim", END)
        return builder.compile()

    def _build_claim_graph(self):
        builder = StateGraph(ClaimState)
        builder.add_node("lookup", self._lookup)
        builder.add_node("checkable", self._checkable)
        builder.add_node("triage", self._run_triage)
        builder.add_node("universal", self._run_universal)
        builder.add_edge(START, "lookup")
        builder.add_conditional_edges("lookup", lambda s: END if s["context"] is None else "checkable", ["checkable", END])
        builder.add_conditional_edges("checkable", self._after_check, ["universal", "triage", END])
        builder.add_edge("triage", END)
        builder.add_edge("universal", END)
        return builder.compile()

    @staticmethod
    def _per_chunk(state: AuditState) -> list[Send]:
        return [Send("extract", {"chunk": chunk}) for chunk in state["chunks"]]

    async def _extract(self, task: dict) -> dict:
        """Node: one chunk's claims, from the cache when its key is unchanged."""
        return {"extracted": await self._extraction.run([task["chunk"]])}

    async def _gather(self, state: AuditState) -> dict:
        """Node: a barrier, so the claims fan out once every chunk is extracted."""
        return {}

    @staticmethod
    def _per_claim(state: AuditState) -> list[Send]:
        order = {chunk.id: index for index, chunk in enumerate(state["chunks"])}
        sends = []
        for result in state["extracted"]:
            for index, claim in enumerate(result.claims):
                if claim.claim_type in AUDITED and claim.evidence:
                    task = {"order": (order[result.id], index), "chunk_id": result.id, "claim": claim}
                    sends.append(Send("claim", task))
        return sends

    async def _claim(self, task: dict) -> dict:
        """Node: run one claim's branch."""
        state = await self.claim_graph.ainvoke({**task, "context": None, "triage": None, "universal": None, "check": None})
        if state["context"] is None:
            return {"claims": []}
        return {"claims": [ClaimResult(order=state["order"], context=state["context"], triage=state["triage"], universal=state["universal"],
                                       check=state["check"])]}

    async def _lookup(self, state: ClaimState) -> dict:
        """Claim node: tie the claim through its sentence, bare names and context, without a model."""
        chunk = chunks_by_id[state["chunk_id"]]
        tie = self._ties.tie(state["chunk_id"], state["claim"], chunk["sentences"])
        if tie is None:
            return {"context": None}
        return {"context": self._context.context(tie, chunk)}

    async def _checkable(self, state: ClaimState) -> dict:
        """Claim node: can code settle this claim at all?"""
        claim = state["claim"]
        return {"check": await self._checkability.cached(claim.claim, chunks_by_id[state["chunk_id"]], claim.evidence[0])}

    @staticmethod
    def _after_check(state: ClaimState) -> str:
        context, kind = state["context"], state["check"].kind
        if kind in NOT_CODE:
            return END
        if kind in TO_A_PERSON:  # traced anyway: a person sees it only if the tracer cannot settle it
            return "triage"
        if context.tie.claim.quantifier in {"all", "none"}:
            return "universal"
        return "triage" if context.level in TO_TRIAGE else END

    async def _run_triage(self, state: ClaimState) -> dict:
        """Claim node: trace, check and review a claim the lookup could not tie."""
        return {"triage": await self._triage.cached(state["context"], [])}

    async def _run_universal(self, state: ClaimState) -> dict:
        """Claim node: check a claim about every instance, or none, across the repository."""
        return {"universal": await self._universal.cached(state["context"])}

In [ ]:
TRIAGE_VARIANT = "all-lite-review"  # linked sentences help the extractor, not the tracer (judged sample)
REPO_LABEL = "original" if REPO == "kmctl" else REPO
code_tools = CodeTools(root_path, tracked, definitions, files)
repo_map = RepoMap(list(code_tools._texts), definitions)
triage_runner = triage_runner_for(code_tools, REPO_LABEL, TRIAGE_VARIANT, repo_map)
universal_runner = universal_runner_for(code_tools, repo_map, REPO_LABEL)
audit = AuditGraph(
    extraction=build_runner(MODEL),
    ties=ClaimTier(anchor_extractor, resolver),
    context=ContextTier(anchor_extractor, BareNameExtractor(), resolver, ConceptSources(path, tracked)),
    triage=triage_runner,
    universal=universal_runner,
    checkability=CheckabilityRunner(triage_llm(CHECK_MODEL), plb.Path("results") / "checkability", CHECK_MODEL),
)
for name, graph in [("audit", audit.graph), ("claim branch", audit.claim_graph), ("triage", triage_runner._graph.graph),
                    ("universal", universal_runner._graph.graph)]:
    print(f"%% {name}\n{graph.get_graph().draw_mermaid()}")

In [ ]:
state = await audit.graph.ainvoke(
    {"chunks": [Chunk.model_validate(c) for c in all_chunks], "extracted": [], "claims": []},
    {"max_concurrency": MAX_CONCURRENT_CALLS},
)
chunk_order = {c["id"]: index for index, c in enumerate(all_chunks)}
bundle_results = sorted(state["extracted"], key=lambda r: chunk_order[r.id])
claim_results = sorted(state["claims"], key=lambda r: r.order)
claim_ties = [r.context.tie for r in claim_results]
contexts = [r.context for r in claim_results]
to_triage = [r.context for r in claim_results if r.triage is not None]
triaged = [r.triage for r in claim_results if r.triage is not None]
universal = [r.universal for r in claim_results if r.universal is not None]

print(f"{len(bundle_results)} chunks, {sum(len(r.claims) for r in bundle_results)} claims, "
      f"{len(claim_ties)} point or structural, {len(triaged)} triaged, {len(universal)} universal")
print("can code settle it:", dict(Counter(r.check.kind for r in claim_results).most_common()))

### Can code settle it? Scored

Three label sets, none written for this step:

- **the judged sample**: 50 claims Opus judges checked with a shell. Your
  verdicts on ten of them win: one broad, two outside tools, one vague, one
  unclear, one history, four checkable. The rest: supported, partly or
  drifted means checkable; not_code means not checkable.
- **the gold set**: claims extracted from sentences labelled only point or
  structural must stay checkable; from sentences labelled only external,
  they should come out external.

A wrong "not checkable" drops a claim the audit should check, so the rate
that checkable claims stay checkable matters most.

In [ ]:
judge_labels = json.load(open("judge_labels.json"))
# Your verdicts on the ten (judge_labels index -> the kinds that count as right).
PERSON_LABELS = {5: {"broad"}, 19: {"external"}, 24: {"external"}, 26: {"vague", "broad"}, 29: {"unclear"},
                 31: {"checkable"}, 33: {"checkable"}, 42: {"history"}, 45: {"checkable"}, 46: {"checkable"}}
NOT_CHECKABLE = set(CHECK_KINDS) - {"checkable"}


def expected_kinds(index: int, label: dict) -> set[str] | None:
    if index in PERSON_LABELS:
        return PERSON_LABELS[index]
    if label["verdict"] in {"supported", "partly", "drifted"}:
        return {"checkable"}
    if label["verdict"] == "not_code":
        return NOT_CHECKABLE
    return None  # unsure


checkability = audit._checkability
judged_checks = await checkability.run([(l["claim"], chunks_by_id[l["chunk_id"]], l["sentence_n"]) for l in judge_labels])
rows = [(i, l, expected_kinds(i, l), c) for i, (l, c) in enumerate(zip(judge_labels, judged_checks))]
rows = [r for r in rows if r[2] is not None]
kept = [r for r in rows if r[2] == {"checkable"}]
held = [r for r in rows if not r[2] & {"checkable"}]
print(f"judged sample: checkable kept {sum(c.kind == 'checkable' for *_, c in kept)}/{len(kept)}; "
      f"not checkable caught {sum(c.kind != 'checkable' for *_, c in held)}/{len(held)}, "
      f"right kind {sum(c.kind in e for _, _, e, c in held)}/{len(held)}")
print(f"your ten: right {sum(c.kind in e for i, _, e, c in rows if i in PERSON_LABELS)}/10")
for i, l, e, c in rows:
    if c.kind not in e:
        print(f"  [{i}] expected {'/'.join(sorted(e)) if len(e) < 4 else 'not checkable'}, got {c.kind}: {l['claim'][:80]}\n      {c.reason[:160]}")

gold_by_id = {g["id"]: GoldRecord.model_validate(g).sentence_labels() for g in gold}
on_gold = [(gold_by_id[r.context.tie.chunk_id].get(r.context.tie.claim.evidence[0], {"none"}), r) for r in claim_results
           if r.context.tie.chunk_id in gold_by_id]
must_keep = [r for labels, r in on_gold if labels <= CHECKABLE]
external = [r for labels, r in on_gold if labels == {"external"}]
print(f"gold: point or structural only kept {sum(r.check.kind == 'checkable' for r in must_keep)}/{len(must_keep)} "
      f"({Counter(r.check.kind for r in must_keep if r.check.kind != 'checkable')}); "
      f"external only caught {sum(r.check.kind == 'external' for r in external)}/{len(external)} "
      f"({Counter(r.check.kind for r in external)})")
for r in must_keep:
    if r.check.kind != "checkable":
        print(f"  gold P/S, got {r.check.kind}: {r.context.tie.claim.claim[:90]}\n      {r.check.reason[:160]}")

## Results: claims tied to code

In [150]:
def tie_table(ties: list[ClaimTie], attribute: str) -> None:
    counts = Counter(getattr(t, attribute) for t in ties)
    for tie in ["strong", "weak", "unresolved", "none"]:
        print(f"  {tie:11} {counts[tie]:4}  {100 * counts[tie] / len(ties):5.1f}%")


print("From the cited sentence:")
tie_table(claim_ties, "tie")
print("Adding the anchors in how_to_check:")
tie_table(claim_ties, "tie_with_check")
for kind in ["point", "structural"]:
    print(f"\n{kind} claims only, from the sentence:")
    tie_table([t for t in claim_ties if t.claim.claim_type == kind], "tie")

From the cited sentence:
  strong       520   54.6%
  weak         201   21.1%
  unresolved     1    0.1%
  none         231   24.2%
Adding the anchors in how_to_check:
  strong       524   55.0%
  weak         198   20.8%
  unresolved     1    0.1%
  none         230   24.1%

point claims only, from the sentence:
  strong       409   57.8%
  weak         159   22.5%
  unresolved     1    0.1%
  none         139   19.6%

structural claims only, from the sentence:
  strong       111   45.3%
  weak          42   17.1%
  unresolved     0    0.0%
  none          92   37.6%


### How precise is a strong tie?

A folder or glob anchor is a strong tie to many files at once, and an anchor that lands in `.okf/` or `docs/` ties the claim to more docs, not to code. A claim is tied **precisely** when one anchor is a single file or a definition outside `.okf/` and `docs/`.

In [151]:
levels = Counter(claim_precision(t) for t in claim_ties)
for level in ["file or symbol", "folder or glob", "docs only", "weak", "unresolved", "none"]:
    print(f"  {level:15} {levels[level]:4}  {100 * levels[level] / len(claim_ties):5.1f}%")
print(Counter(r.via for t in claim_ties for r in t.from_sentence if r.status == "path"))

  file or symbol   440   46.2%
  folder or glob    27    2.8%
  docs only         53    5.6%
  weak             201   21.1%
  unresolved         1    0.1%
  none             231   24.2%
Counter({'exact': 415, 'folder': 261, 'ending': 243, 'glob': 1})


### Which files the strong ties point to

By top folder, then the files most claims point to.

In [152]:
strong = [t for t in claim_ties if t.tie == "strong"]
by_folder = Counter(folder for t in strong for folder in {f.split("/")[0] for f in t.files()})
by_file = Counter(f for t in strong for f in t.files())
print(f"{len(strong)} strong claims point to {len(by_file)} files")
for folder, count in by_folder.most_common(12):
    print(f"  {folder:28} {count:4} claims")
print()
for file, count in by_file.most_common(15):
    print(f"  {file:60} {count:4} claims")

520 strong claims point to 135 files
  src                           171 claims
  tests                         157 claims
  action                        102 claims
  .github                        77 claims
  .okf                           77 claims
  docs                           62 claims
  apm                            61 claims
  scripts                        42 claims
  action.yml                     41 claims
  apm.yml                        36 claims
  CLAUDE.md                      36 claims
  pyproject.toml                 27 claims

  action/agent.py                                                93 claims
  src/kmctl/kinds.py                                             74 claims
  .okf/architecture/index.md                                     63 claims
  .okf/decisions/0002-briefing-and-rules-injected-by-the-cli.md   62 claims
  .okf/decisions/0001-standalone-cli-installs-its-plugin.md      62 claims
  .okf/decisions/0003-claude-code-plugin-first-depends-on-okf.md   62 

### Claims with no anchor in their sentence, by concept

What the lookup cannot reach: the claim names no file, symbol or command.

In [153]:
unanchored = [t for t in claim_ties if t.tie == "none"]
for concept, count in Counter(t.chunk_id.split("#")[0] for t in unanchored).most_common(10):
    print(f"  {count:3}  {concept}")
print()
for t in unanchored[:12]:
    print(f"- [{t.claim.claim_type}] {t.claim.claim}\n    check: {t.claim.how_to_check}")

   22  decisions/0005-audits-report-only
   17  decisions/0017-the-ci-audit-is-a-standalone-pydanticai-agent-script
   15  decisions/0011-kmctl-init-writes-knowledge-rules-and-a-claude-md-block-for-claude-code
   15  summary
   14  decisions/0023-kmctl-verify-records-a-person-s-check
   13  decisions/0008-handoffs-are-okf-concepts
   13  manifesto
   10  decisions/0015-antigravity-gets-skills-and-rules-from-apm-and-a-kmctl-plugin-folder
    9  decisions/0006-staleness-from-okf-fields
    9  decisions/0027-the-bundle-opens-with-a-summary-that-kmctl-keeps-in-step

- [point] kmctl has four parts (manifesto, Parts).
    check: Open src/kmctl/ and check the parts mentioned in the codebase and documentation.
- [structural] Both audits read and report only.
    check: Inspect the audit implementations in the repository to ensure they only read and report.
- [point] The Antigravity sub-package also ships okf's Stop check wrapper.
    check: ls apm/antigravity/
- [point] The auditor's tools are

### After bare names and context

In [154]:
LEVELS = ["sentence", "bare", "previous", "sources", "folder or glob", "docs only", "weak", "unresolved", "none"]


def level_table(contexts: list[ClaimContext]) -> None:
    counts = Counter(c.level for c in contexts)
    running = 0
    for level in LEVELS:
        running += counts[level]
        print(f"  {level:15} {counts[level]:4}  {100 * counts[level] / len(contexts):5.1f}%   cumulative {100 * running / len(contexts):5.1f}%")


level_table(contexts)
for kind in ["point", "structural"]:
    print(f"\n{kind}:")
    level_table([c for c in contexts if c.tie.claim.claim_type == kind])

  sentence         440   46.2%   cumulative  46.2%
  bare              36    3.8%   cumulative  49.9%
  previous         135   14.2%   cumulative  64.1%
  sources          146   15.3%   cumulative  79.4%
  folder or glob    18    1.9%   cumulative  81.3%
  docs only          7    0.7%   cumulative  82.1%
  weak              61    6.4%   cumulative  88.5%
  unresolved         1    0.1%   cumulative  88.6%
  none             109   11.4%   cumulative 100.0%

point:
  sentence         360   50.8%   cumulative  50.8%
  bare              29    4.1%   cumulative  54.9%
  previous         106   15.0%   cumulative  69.9%
  sources          106   15.0%   cumulative  84.9%
  folder or glob     1    0.1%   cumulative  85.0%
  docs only          4    0.6%   cumulative  85.6%
  weak              45    6.4%   cumulative  91.9%
  unresolved         1    0.1%   cumulative  92.1%
  none              56    7.9%   cumulative 100.0%

structural:
  sentence          80   32.7%   cumulative  32.7%
  bare    

### What the bare names found

To judge by hand: a bare name that ties to the wrong thing is a false tie.

In [155]:
bare_hits = Counter((r.anchor.text, r.status, r.locations[0]) for c in contexts if c.level == "bare" for r in c.bare)
for (text, status, location), count in bare_hits.most_common(40):
    print(f"{count:3}  {status:8} {text:40} -> {location}")

 18  defined  SessionStart                             -> apm/antigravity/.apm/hooks/hooks.json:3
 10  defined  stale_after                              -> src/kmctl/concepts.py:213
  4  defined  okf-bundle                               -> tests/test_apm_package.py:100
  3  defined  how-to                                   -> apm/.apm/skills/write-docs/evals/grade.py:21
  2  path     apm.yml                                  -> apm.yml
  2  path     hooks.json                               -> apm/antigravity/.apm/hooks/hooks.json
  1  defined  bundle-weaver                            -> tests/test_opencode.py:85
  1  defined  event-analyzer                           -> tests/test_opencode.py:85


### Triage of every claim the lookups could not tie precisely

In [ ]:
OUTCOMES = ["found", "partly", "contradicted", "rejected", "only tests or docs", "made up", "not_found", "not_code", "not reviewed", "gave up"]


def outcome_table(results: list[Triage]) -> None:
    if not results:
        print("  none")
        return
    counts = Counter(t.outcome for t in results)
    for outcome in OUTCOMES:
        print(f"  {outcome:18} {counts[outcome]:4}  {100 * counts[outcome] / len(results):5.1f}%")
    errors = sum(1 for t in results if t.error)
    calls = sum(len(t.tool_calls) for t in results)
    print(f"  {errors} errors, {calls / len(results):.1f} tool calls a claim")


outcome_table(triaged)
for level in ["none", "weak", "docs only", "folder or glob", "previous", "sources"]:
    print(f"\nwas {level}:")
    outcome_table([t for t, c in zip(triaged, to_triage) if c.level == level])

In [ ]:
# Every claim, after the lookups, the universal check and the agent
final = Counter()
for r in claim_results:
    if r.universal is not None:
        final["universal: " + r.universal.verdict] += 1
    elif r.triage is not None:
        final[("agent: " if r.context.level in UNTIED else "context, then agent: ") + r.triage.outcome] += 1
    else:
        final["lookup: in the sentence"] += 1
for name, count in final.most_common():
    print(f"  {name:36} {count:4}  {100 * count / len(claim_results):5.1f}%")

### Universal claims

In [ ]:
print(f"{len(universal)} universal claims ({Counter(u.quantifier for u in universal)})")
for (verdict, route), count in Counter((u.verdict, u.route or "-") for u in universal).most_common():
    print(f"  {verdict:16} {route:13} {count:4}")
print(f"  {sum(u.llm_calls for u in universal) / len(universal):.1f} model calls a claim, "
      f"{sum(1 for u in universal if u.error)} errors; rule route: "
      f"{Counter(u.rule_rounds for u in universal if u.route == 'rule')} rounds")
print("\nBroken, with where:")
for u in universal:
    if u.verdict == "broken":
        print(f"- [{u.quantifier}] {u.claim}")
        for site in u.breaking[:3]:
            reason = next((c.reason for c in u.checks if c.verdict == "breaks" and 0 <= c.index < len(u.sites) and u.sites[c.index] == site), "")
            print(f"    {site.path}:{site.line}: {site.text[:90]}  -- {reason[:120]}")

### The tracer with a repo map, and with a retry when the quote is not there

The same claims through three versions of the triage graph: the plain
tracer, the tracer with the repo map in its message, and the map plus one
retry when the check rejects a quote (the tracer is told which quote failed
and why). The audit graph uses `TRIAGE_VARIANT`.

In [ ]:
untied_claims = [c for c in to_triage if c.level in UNTIED]
variant_results = {}
for variant in ["", "map", "map-retry", "all-lite"]:  # the steps that built the tracer; the judged sample scores the rest
    variant_results[variant] = await triage_runner_for(code_tools, REPO_LABEL, variant, repo_map).run(untied_claims)
    print(f"{variant or 'plain'}:")
    outcome_table(variant_results[variant])

### Scored against judges: what makes the tracer find what it misses

`judge_labels.json`: 50 claims sampled from the 246 the audit left untied
(not found, unconfirmed, universal with no instances or no working rule,
not code), each judged by a claude-opus/5.5 subagent with a shell, whole
files and git history, without seeing the audit's verdict. The judges found
38 supported, 5 partly, 1 drifted, 5 not about code and 1 unsure: the audit
missed 43 of the 50. No person has checked these labels.

Each variant changes one thing on top of `map-retry` (the audit's tracer):
`flash` (gemini-3.8-flash), `tools` (the judges' tools: regex grep, whole
files, file globs, git log and show), `section` (the whole section as
context), `turns20` (20 turns, not 8), `plan` (a plan before searching);
`all-lite` and `all-flash` add them all. All 50 go through the tracer,
universal ones included. Recall: of the 43 the judges found supported or
partly, how many the variant finds or partly finds. Same file: how many of
those found point at a file the judges also named.

In [ ]:
judge_labels = json.load(open("judge_labels.json"))
# The judged claims are rebuilt from their labels, so a new extraction does not move them; the sentences
# they rely on come from the current extraction's claims on the same sentence.
current_refs: dict[tuple[str, int], set[str]] = {}
for r in claim_results:
    current_refs.setdefault((r.context.tie.chunk_id, r.context.tie.claim.evidence[0]), set()).update(r.context.tie.claim.refers_to)


def judged_context(label: dict) -> ClaimContext:
    chunk = chunks_by_id[label["chunk_id"]]
    claim = Claim(evidence=[label["sentence_n"]], claim_type=label["claim_type"], claim=label["claim"],
                  how_to_check=label["how_to_check"], quantifier=label["quantifier"],
                  refers_to=sorted(current_refs.get((label["chunk_id"], label["sentence_n"]), set())))
    return audit._context.context(audit._ties.tie(label["chunk_id"], claim, chunk["sentences"]), chunk)


judged = [(label, judged_context(label)) for label in judge_labels]
judged_contexts = [context for _, context in judged]
RESOLVABLE = {"supported", "partly"}


def judge_score(results: list[Triage]) -> dict:
    found = [(label, t) for (label, _), t in zip(judged, results) if t.outcome in {"found", "partly"}]
    hits = [(label, t) for label, t in found if label["verdict"] in RESOLVABLE]
    same_file = sum(1 for label, t in hits if t.report.verdict == "absent"
                    or {l.path for l in t.primary_locations()} & {loc.split(":")[0] for loc in label["locations"]})
    wanted = sum(1 for label in judge_labels if label["verdict"] in RESOLVABLE)
    return {"recall": f"{len(hits)}/{wanted}", "same file": f"{same_file}/{len(hits)}",
            "found where judges did not": len(found) - len(hits),
            "contradicted": sum(t.outcome == "contradicted" for t in results),
            "not_found": sum(t.outcome == "not_found" for t in results),
            "absent": sum(1 for t in results if t.report is not None and t.report.verdict == "absent"),
            "not_code (judges agree)": f"{sum(t.outcome == 'not_code' for t in results)} "
                                       f"({sum(t.outcome == 'not_code' and l['verdict'] == 'not_code' for (l, _), t in zip(judged, results))})",
            "tool calls": round(sum(len(t.tool_calls) for t in results) / len(results), 1),
            "errors": sum(1 for t in results if t.error)}


judge_results = {}
# The eight levers above were measured with the earlier prompts (see the README); these two run with the current ones.
for variant in ["all-lite-review", "all-lite-review@2", "all-lite-split", "all-lite-split@2"]:
    judge_results[variant] = await triage_runner_for(code_tools, REPO_LABEL, variant, repo_map).run(judged_contexts)
    print(f"{variant:10}", judge_score(judge_results[variant]))

## Ties: what the audit found, saved per commit

Every checkable claim gets one record: the claim, its sentence, how it was
tied and the evidence. For a lookup tie that is the anchor and where it
resolved; for the agent, each location with its quote; for a universal
claim, the searches, the rule and any instances that break it. Records are
keyed by a hash of the concept and claim text, and the file is named by the
commit, so the next run can recheck without a model:

- a lookup tie is resolved again: the path or definition must still be there;
- an agent tie needs each quote still in its file, near its line;
- a universal claim reruns its searches (or rule): new or vanished instances
  mean it must be checked again.

Only claims that fail the recheck, and new claims, need the model again.

In [ ]:
class Evidence(BaseModel):
    """Where a claim lives: a path, maybe a line and symbol, and the quote that shows it."""

    path: str
    line: int = 0
    symbol: str = ""
    quote: str = ""
    anchor: str = ""


class TieRecord(BaseModel):
    """One checkable claim's result, as saved and as the report shows it."""

    claim_id: str
    concept: str
    chunk_id: str
    section: str
    sentence_n: int
    sentence: str
    claim: str
    kind: str
    quantifier: str
    status: Literal["tied", "tied by context", "found", "partly", "contradicted", "not found", "not code", "unconfirmed",
                    "holds", "broken", "no instances", "could not check", "needs a person", "confirmed by a person"]
    how: str
    note: str = ""
    evidence: list[Evidence] = []
    breaking: list[Evidence] = []
    searches: list[str] = []
    rule: RuleDraft | None = None
    sites: list[str] = []


STATUS_OF_OUTCOME = {"found": "found", "partly": "partly", "rejected": "unconfirmed", "only tests or docs": "unconfirmed",
                     "made up": "unconfirmed", "not_found": "not found", "not_code": "not code", "not reviewed": "unconfirmed",
                     "gave up": "unconfirmed", "still there": "broken", "contradicted": "contradicted"}


def claim_id(concept: str, claim: str) -> str:
    return hashlib.sha256(json.dumps([concept, claim]).encode()).hexdigest()[:12]


class Confirmation(BaseModel):
    """A person's verdict on one claim, valid while its sentence reads the same."""

    sentence: str
    claim: str = ""
    verdict: Literal["holds", "does not hold"]
    by: str = ""
    note: str = ""


class Confirmations:
    """People's verdicts, one JSON file per repository: {claim id: Confirmation}. A person edits it by hand."""

    def __init__(self, path: plb.Path):
        self.path = path
        raw = json.loads(path.read_text()) if path.exists() else {}
        self._by_id = {k: Confirmation.model_validate(v) for k, v in raw.items()}

    def get(self, claim_id: str, sentence: str) -> Confirmation | None:
        """The verdict for this claim, unless the sentence has changed since the person gave it."""
        found = self._by_id.get(claim_id)
        return found if found is not None and found.sentence == sentence else None


confirmations = Confirmations(plb.Path("results") / "confirmations" / f"{REPO}.json")


def with_person(record: "TieRecord") -> "TieRecord":
    """A person's verdict wins; a claim code cannot settle waits for one, with any evidence the agent found."""
    found = confirmations.get(record.claim_id, record.sentence)
    if found is not None:
        status = "confirmed by a person" if found.verdict == "holds" else "broken"
        return record.model_copy(update={"status": status, "note": f"{found.by or 'a person'}: {found.note}".strip(": ")})
    return record


# A trace that ends here settles a claim even when the checkability step doubted code could:
# a wrong doubt then costs nothing, and only an unsettled doubtful claim goes to a person.
SETTLED = {"found", "partly", "contradicted", "broken", "holds"}


def tie_record(result: ClaimResult) -> TieRecord:
    """Turn one claim's way through the graph into a record, then the checkability answer, then any person's verdict."""
    record, check = _tie_record(result), result.check
    if check is not None and check.kind in TO_A_PERSON and record.status not in SETTLED:
        record = record.model_copy(update={"status": "needs a person", "how": f"check: {check.kind}, {record.how}",
                                           "note": f"{check.kind}: {check.reason}"})
    elif check is not None and check.kind in NOT_CODE:
        record = record.model_copy(update={"status": "not code", "how": f"check: {check.kind}", "evidence": [],
                                           "note": f"{check.kind}: {check.reason}"})
    return with_person(record)


def _tie_record(result: ClaimResult) -> TieRecord:
    context = result.context
    tie = context.tie
    chunk = chunks_by_id[tie.chunk_id]
    base = dict(claim_id=claim_id(chunk["concept"], tie.claim.claim), concept=chunk["concept"], chunk_id=tie.chunk_id,
                section=chunk["section"], sentence_n=tie.claim.evidence[0], sentence=tie.sentence, claim=tie.claim.claim,
                kind=tie.claim.claim_type, quantifier=tie.claim.quantifier)
    if result.universal is not None:
        u = result.universal
        rule_note = f"{len(u.sites)} instances, checked {u.route}" if u.route else "the searches found nothing"
        return TieRecord(**base, status=u.verdict, how="universal", note=(u.candidates.complies if u.candidates else u.error or "") ,
                         breaking=[Evidence(path=s.path, line=s.line, quote=s.text) for s in u.breaking],
                         searches=u.candidates.patterns if u.candidates else [], rule=u.rule if u.route == "rule" else None,
                         sites=[f"{s.path}: {s.text}" for s in u.sites], evidence=[]) .model_copy(update={"how": "universal: " + rule_note})
    if result.triage is not None:
        t = result.triage
        evidence = [Evidence(path=l.path, line=l.line, symbol=l.symbol, quote=l.quote) for l in t.primary_locations()] if t.outcome in {"found", "partly", "contradicted"} else []
        note = (t.review.reason if t.review else t.report.reason if t.report else t.error or "")
        if t.report is not None and t.report.verdict == "absent" and t.outcome in {"found", "still there"}:
            gone = [Evidence(path=n, quote="absent" if code_tools.absent(n) else "still there") for n in t.report.absent_names]
            return TieRecord(**base, status=STATUS_OF_OUTCOME[t.outcome], how="agent: absent", note=note, evidence=gone)
        return TieRecord(**base, status=STATUS_OF_OUTCOME[t.outcome], how=f"agent: {t.outcome}", note=note, evidence=evidence)
    level = context.level
    evidence = [Evidence(path=loc.split(":")[0], line=int(loc.split(":")[1]) if ":" in loc else 0, anchor=r.anchor.text)
                for r in context.precise_resolutions() for loc in r.locations[:1]]
    status = "tied" if level in {"sentence", "bare"} else "tied by context"
    return TieRecord(**base, status=status, how=f"lookup: {level}", evidence=evidence)


class TieStore:
    """Ties on disk, one JSON file per repository and commit."""

    def __init__(self, root: plb.Path):
        self._root = root

    @staticmethod
    def commit(repo_root: plb.Path) -> str:
        result = subprocess.run(["git", "-C", str(repo_root), "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
        dirty = subprocess.run(["git", "-C", str(repo_root), "status", "--porcelain"], capture_output=True, text=True).stdout.strip()
        return (result.stdout.strip() or "none") + ("-dirty" if dirty else "")

    def save(self, repo: str, commit: str, records: list[TieRecord]) -> plb.Path:
        path = self._root / repo / f"{commit}.json"
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps({"repo": repo, "commit": commit, "ties": [r.model_dump(mode="json") for r in records]}, indent=1))
        return path

    def load(self, path: plb.Path) -> list[TieRecord]:
        return [TieRecord.model_validate(r) for r in json.loads(path.read_text())["ties"]]

In [ ]:
class Rechecker:
    """Recheck saved ties against a copy of the repository, without a model."""

    def __init__(self, tools: CodeTools, resolver: AnchorResolver):
        self._tools = tools
        self._resolver = resolver
        self._search = UniversalTools(tools)

    def _anchor_still_there(self, evidence: Evidence) -> bool:
        resolution = self._resolver.resolve(Anchor(text=evidence.anchor, source="code_span"))
        return resolution.status in {"path", "defined"}

    def recheck(self, record: TieRecord) -> str:
        """still holds, drifted (the evidence is gone), changed (universal instances differ) or unchecked."""
        if record.status in {"tied", "tied by context"}:
            return "still holds" if all(self._anchor_still_there(e) for e in record.evidence) else "drifted"
        if record.status == "confirmed by a person":
            return "still holds"  # a person's verdict stands until the sentence changes, which the next audit sees
        if record.how == "agent: absent":
            return "still holds" if all(self._tools.absent(e.path) for e in record.evidence) else "drifted"
        if record.status in {"found", "partly", "contradicted"}:
            locations = [Location(path=e.path, line=e.line, symbol=e.symbol, quote=e.quote) for e in record.evidence]
            return "still holds" if all(self._tools.check(l) for l in locations) else "drifted"
        if record.how.startswith("universal") and record.searches:
            sites, _ = self._search.harvest(record.searches, named=record.claim + " " + record.sentence)
            # By path and text, so lines that only moved do not count as a change.
            before = set(record.sites)
            after = {f"{s.path}: {s.text}" for s in sites}
            return "still holds" if before == after else "changed"
        return "unchecked"

## The report: one page per concept

For a person: a summary at the top, then every section with its sentences,
each claim's status and the evidence (file, line, symbol, the quoted
lines), and what needs a look first. For an agent: the same records as
JSON, in the ties file. Statuses, worst first:

- **broken**: a universal claim with instances that go against it.
- **not found**: the agent searched and nothing in the code does it.
  Either the claim is out of date or the code is somewhere it could not find.
- **unconfirmed**: the agent pointed somewhere but the quote was not there,
  the reviewer rejected it, or it only found tests or docs.
- **could not check** / **no instances**: the universal route could not
  build a working search, or found nothing to check.
- **tied by context**: only the section's other sentences or the concept's
  sources point at code; a guess.
- **partly**, **found**, **tied**, **holds**: evidence below.

In [ ]:
STATUS_ORDER = ["contradicted", "broken", "not found", "unconfirmed", "needs a person", "could not check", "no instances", "tied by context",
                "partly", "found", "tied", "holds", "confirmed by a person", "not code"]


class ReportWriter:
    """Markdown pages, one per concept, plus an index, from tie records and the extraction."""

    def __init__(self, repo: str, commit: str):
        self._repo = repo
        self._commit = commit

    @staticmethod
    def _evidence(e: Evidence) -> str:
        where = f"`{e.path}{':' + str(e.line) if e.line else ''}`" + (f" `{e.symbol}`" if e.symbol else "")
        quote = e.quote.strip().splitlines()[0][:120] if e.quote.strip() else ""
        return where + (f": `{quote}`" if quote else (f" (from `{e.anchor}`)" if e.anchor else ""))

    def _claim_line(self, r: TieRecord) -> list[str]:
        q = "" if r.quantifier == "exists" else f" [{r.quantifier}]"
        lines = [f"  - **{r.status}**{q}: {r.claim}"]
        for e in r.evidence[:3]:
            lines.append(f"    - {self._evidence(e)}")
        for e in r.breaking[:5]:
            lines.append(f"    - breaks at {self._evidence(e)}")
        if len(r.breaking) > 5:
            lines.append(f"    - and {len(r.breaking) - 5} more")
        if r.note and r.status not in {"tied", "holds"}:
            lines.append(f"    - _{r.note[:240]}_")
        return lines

    def page(self, concept: str, chunks: list[dict], results: dict[str, ChunkResult], records: list[TieRecord]) -> str:
        by_sentence: dict[tuple[str, int], list[TieRecord]] = {}
        for r in records:
            by_sentence.setdefault((r.chunk_id, r.sentence_n), []).append(r)
        sentences = sum(len(c["sentences"]) for c in chunks)
        with_claims = {(c["id"], n) for c in chunks for claim in results[c["id"]].claims for n in claim.evidence}
        kinds = Counter(claim.claim_type for c in chunks for claim in results[c["id"]].claims)
        statuses = Counter(r.status for r in records)
        lines = [f"# {concept}", "", f"Repository `{self._repo}` at `{self._commit}`.", "",
                 f"**Coverage**: {sentences} sentences; {len(with_claims)} hold a claim, {sentences - len(with_claims)} hold none "
                 f"(status, history, reasons). Claims: " + ", ".join(f"{n} {k}" for k, n in kinds.most_common()) + ".", "",
                 "**Checkable claims**: " + (", ".join(f"{statuses[s]} {s}" for s in STATUS_ORDER if statuses[s]) or "none") + ".", ""]
        worst = [r for r in records if r.status in STATUS_ORDER[:5]]
        if worst:
            lines += ["**Look at first**:", ""] + [f"- **{r.status}**: {r.claim} (sentence {r.sentence_n}, {r.section.split(' > ')[-1]})" for r in sorted(worst, key=lambda r: STATUS_ORDER.index(r.status))] + [""]
        for chunk in chunks:
            lines += [f"## {chunk['section']}", ""]
            for n, sentence in enumerate(chunk["sentences"], 1):
                claims = results[chunk["id"]].claims
                cited = [c for c in claims if n in c.evidence]
                checkable = by_sentence.get((chunk["id"], n), [])
                if not cited:
                    lines.append(f"- [{n}] {sentence[:300]}  \n  _no claim_")
                    continue
                lines.append(f"- [{n}] {sentence[:300]}")
                for c in cited:
                    if c.claim_type not in {"point", "structural"}:
                        lines.append(f"  - {c.claim_type}" + (f": {c.claim}" if c.claim else ""))
                for r in checkable:
                    lines += self._claim_line(r)
            lines.append("")
        return "\n".join(lines)

    def write(self, out: plb.Path, chunks: list[dict], results: list[ChunkResult], records: list[TieRecord]) -> list[plb.Path]:
        by_id = {r.id: r for r in results}
        concepts: dict[str, list[dict]] = {}
        for c in chunks:
            concepts.setdefault(c["concept"], []).append(c)
        written, index = [], [f"# Audit of `{self._repo}` at `{self._commit}`", "", "| Concept | Checkable | Worst | " + " | ".join(STATUS_ORDER) + " |",
                              "|---|---|---|" + "---|" * len(STATUS_ORDER)]
        for concept, concept_chunks in concepts.items():
            mine = [r for r in records if r.concept == concept]
            path = out / f"{concept}.md"
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(self.page(concept, concept_chunks, by_id, mine))
            written.append(path)
            counts = Counter(r.status for r in mine)
            worst = next((s for s in STATUS_ORDER if counts[s]), "")
            index.append(f"| [{concept}]({concept}.md) | {len(mine)} | {worst} | " + " | ".join(str(counts[s] or "") for s in STATUS_ORDER) + " |")
        (out / "index.md").write_text("\n".join(index) + "\n")
        (out / "needs-a-person.md").write_text(self.person_page(records))
        return written

    def person_page(self, records: list[TieRecord]) -> str:
        """The claims that wait for a person, with the evidence and how to record a verdict."""
        waiting = [r for r in records if r.status == "needs a person"]
        lines = [f"# Needs a person: {len(waiting)} claims", "",
                 "The checkability step doubted code could settle these claims (too broad, too vague, or they refer to "
                 "something they do not name), and the tracer did not settle them either. "
                 "Read the evidence, then add a verdict to "
                 f"`{confirmations.path}` under the claim's id:", "",
                 '```json\n{"<claim id>": {"sentence": "<the sentence, exactly>", "claim": "...", "verdict": "holds", "by": "you", "note": ""}}\n```', "",
                 "A verdict stands until the sentence changes.", ""]
        for r in waiting:
            lines += [f"## {r.claim}", "", f"- id `{r.claim_id}`, `{r.chunk_id}` sentence {r.sentence_n}", f"- sentence: {r.sentence}"]
            lines += [f"- evidence: {self._evidence(e)}" for e in r.evidence[:5]] or ["- no evidence found"]
            lines += [f"- agent: _{r.note[:300]}_", ""] if r.note else [""]
        return "\n".join(lines)

In [ ]:
records = [tie_record(r) for r in claim_results]
commit = TieStore.commit(root_path)
ties_path = TieStore(plb.Path("results") / "ties").save(REPO, commit, records)
# Outside the bundle: the pages are not concepts, and OKF validation would reject them.
report_dir = plb.Path(os.environ.get("AUDIT_REPORT_DIR", plb.Path.home() / ".cache" / "audit-agent" / "report")) / REPO
pages = ReportWriter(REPO, commit).write(report_dir, all_chunks, bundle_results, records)
print(ties_path, len(records), "ties;", len(pages), "pages in", report_dir)
for status, count in Counter(r.status for r in records).most_common():
    print(f"  {status:16} {count:4}  {100 * count / len(records):5.1f}%")

In [ ]:
# A recheck against the same copy must say every tie still holds.
same = Counter(Rechecker(code_tools, resolver).recheck(r) for r in records)
same

### Are context ties right?

The agent traces a sample of claims tied only through context. A context tie holds when the agent's verified locations include a file the context pointed to.

In [158]:
context_sample = random.Random(7).sample([c for c in contexts if c.level in {"previous", "sources"}], 60)
context_triaged = await triage_runner.run(context_sample)

agreement = Counter()
for c, t in zip(context_sample, context_triaged):
    inferred = {location.split(":")[0] for r in c.precise_resolutions() for location in r.locations}
    if t.outcome != "found":
        agreement[(c.level, "agent: " + t.outcome)] += 1
    elif inferred & {l.path for l in t.report.locations}:
        agreement[(c.level, "agent agrees")] += 1
    else:
        agreement[(c.level, "agent found elsewhere")] += 1
for (level, verdict), count in sorted(agreement.items()):
    print(f"  {level:9} {verdict:24} {count:3}")

  previous  agent agrees               4
  previous  agent found elsewhere      6
  previous  agent: made up             9
  previous  agent: not_found           7
  previous  agent: only tests or docs   1
  previous  agent: partly              2
  previous  agent: rejected            4
  sources   agent agrees               2
  sources   agent found elsewhere      2
  sources   agent: made up             2
  sources   agent: not_code            2
  sources   agent: not_found           9
  sources   agent: partly              2
  sources   agent: rejected            8


### A sample of each outcome, to judge by hand

In [159]:
for outcome in ["found", "partly", "rejected", "only tests or docs", "made up", "not_found"]:
    sample = [t for t in triaged if t.outcome == outcome]
    print(f"\n=== {outcome} ({len(sample)})")
    for t in random.Random(1).sample(sample, min(6, len(sample))):
        where = ", ".join(f"{l.path}:{l.line} {l.symbol}" for l in t.report.locations) if t.report else ""
        why = t.review.reason if t.review else (t.report.reason if t.report else t.error)
        print(f"- {t.claim}\n    {where}\n    {why}")


=== found (57)
- The audit skill, its three review skills, and the kmctl-auditor agent write no knowledge.
    apm/.apm/skills/audit/SKILL.md:8 , apm/.apm/agents/kmctl-auditor.md:7 
    The documentation files for the audit skill and kmctl-auditor agent explicitly specify that they are read-only and write no knowledge.
- The keys are not in the shell tool's environment.
    action/agent.py:384 run_shell
    The implementation of run_shell correctly creates an environment that filters out unwanted variables according to the claim.
- Antigravity relays okf's Stop check.
    src/kmctl/antigravity.py:58 STOP_COMMAND
    The provided code defines STOP_COMMAND which executes okf's stop hook script and relays its check.
- kmctl plugin patch fills the gaps.
    src/kmctl/plugin.py:706 patch
    The code location defines the `patch` function described in the claim.
- The reusable GitHub Action runs the deterministic checks on every pull request
    .github/workflows/ci.yml:3 , .github/workflow

## Planted drift: does anyone notice when the code changes?

A copy of the repository with drift planted in it, against the same claims:

- **deleted files**: a sample of files that claims tie to precisely, deleted; whatever mentions them stays.
- **renamed definitions**: a sample of distinctive symbols renamed to `<name>_v2` in every code file; the docs keep the old name.

First the lookup: does each planted anchor still resolve? Then the agent, on the claims those anchors tied, in both the original and the drifted copy. A good tracer finds them in the original and reports `not_found` (or the renamed symbol) in the copy.

In [160]:
import shutil
import tempfile

SAMPLE_SIZE = 15
rng = random.Random(20261106)

precise = [(c, r) for c in contexts for r in c.precise_resolutions()]
deleted_candidates = sorted({r.locations[0] for _, r in precise if r.status == "path" and r.locations and not r.locations[0].endswith(".md")})
renamed_candidates = sorted({
    r.anchor.text for _, r in precise
    if r.status == "defined" and re.fullmatch(r"[A-Za-z_][\w]{5,}", r.anchor.text)
    and len(mentions.find(r.anchor.text, prose=False)[1]) <= 3
})
to_delete = set(rng.sample(deleted_candidates, min(SAMPLE_SIZE, len(deleted_candidates))))
to_rename = set(rng.sample(renamed_candidates, min(SAMPLE_SIZE, len(renamed_candidates))))
len(deleted_candidates), len(renamed_candidates), sorted(to_delete), sorted(to_rename)

(57,
 20,
 ['.claude/settings.json',
  '.github/workflows/hermes-interactive.yaml',
  '.github/workflows/knowledge-audit.yml',
  '.github/workflows/release.yml',
  'action.yml',
  'action/agent.py',
  'copier.yml',
  'scripts/check_wheel.py',
  'scripts/release.py',
  'src/kmctl/antigravity.py',
  'src/kmctl/review_due.py',
  'src/kmctl/summary.py',
  'src/kmctl/supersede.py',
  'tests/test_apm_package.py',
  'tests/test_starter.py'],
 ['OLD_RULES',
  'STATUS_VALUES',
  'ScaffoldResult',
  '_write_first_decision',
  'api_key_env',
  'commandExecutionPolicy',
  'harnesses_of_path',
  'issue_title',
  'list_files',
  'mainAgent',
  'min_severity',
  'read_file',
  'report_name',
  'run_shell',
  'schedule'])

In [ ]:
class DriftedCopy:
    """A copy of the tracked files, with files deleted and symbols renamed in code but not in docs."""

    DOCS = (".okf/", "docs/")

    def __init__(self, root: plb.Path):
        self.root = plb.Path(tempfile.mkdtemp(prefix="drift-"))
        for path in TrackedPaths(root)._paths:
            source = root / path
            if source.is_file():
                (self.root / path).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, self.root / path)

    def delete(self, paths: set[str]) -> None:
        for path in paths:
            (self.root / path).unlink()

    def rename(self, names: set[str]) -> None:
        patterns = {name: re.compile(r"\b" + re.escape(name) + r"\b") for name in names}
        for file in self.root.rglob("*"):
            relative = file.relative_to(self.root).as_posix()
            if not file.is_file() or relative.startswith(self.DOCS):
                continue
            try:
                text = file.read_text()
            except UnicodeDecodeError:
                continue
            changed = text
            for name, pattern in patterns.items():
                changed = pattern.sub(name + "_v2", changed)
            if changed != text:
                file.write_text(changed)

    def commit(self) -> None:
        """Make git track what is left, so ls-files matches the copy."""
        subprocess.run(["git", "init", "-q", str(self.root)], check=True)
        subprocess.run(["git", "-C", str(self.root), "add", "-A"], check=True)


# The cache label names what was planted, so a different sample never reuses another copy's results.
DRIFT_LABEL = "drift-" + hashlib.sha256(json.dumps([sorted(to_delete), sorted(to_rename)]).encode()).hexdigest()[:8]
drift = DriftedCopy(root_path)
drift.delete(to_delete)
drift.rename(to_rename)
drift.commit()

drift_files = list(reader.read_files(drift.root))
drift_tracked = TrackedPaths(drift.root)
drift_definitions = DefinitionIndex(drift.root, drift_files, parsers)
drift_resolver = AnchorResolver(drift_tracked, drift_definitions, MentionSearch(drift.root, drift_files))
drift.root, len(drift_files)

### The lookup on the drifted copy

In [ ]:
planted = [(c, r) for c, r in precise if (r.status == "path" and r.locations[0] in to_delete) or (r.status == "defined" and r.anchor.text in to_rename)]
lookup_after = Counter()
seen_anchors = {}
for _, r in planted:
    if r.anchor.text not in seen_anchors:
        seen_anchors[r.anchor.text] = (r.status, drift_resolver.resolve(r.anchor).status)
for (before, after) in seen_anchors.values():
    lookup_after[(before, after)] += 1
print(f"{len(seen_anchors)} planted anchors, {len({id(c) for c, _ in planted})} claims rest on them")
for (before, after), count in lookup_after.most_common():
    print(f"  {before:8} -> {after:20} {count:3}")

### The agent on the same claims, original and drifted

At most two claims per planted anchor.

In [ ]:
drift_claims = []
per_anchor = Counter()
for c, r in planted:
    if per_anchor[r.anchor.text] < 2 and c not in drift_claims:
        per_anchor[r.anchor.text] += 1
        drift_claims.append(c)

drift_tools = CodeTools(drift.root, drift_tracked, drift_definitions, drift_files)
on_original = await triage_runner_for(code_tools, "original").run(drift_claims)


def where(triage: Triage) -> list[str]:
    """A found claim's locations, as the earlier location for the next run."""
    if triage.outcome not in {"found", "partly"}:
        return []
    return [f"{l.path}:{l.line} {l.symbol}".strip() for l in triage.report.locations]


drifted_runner = triage_runner_for(drift_tools, DRIFT_LABEL)
on_drifted_new = await drifted_runner.run(drift_claims)
on_drifted_known = await drifted_runner.run(drift_claims, previous=[where(t) for t in on_original])

print(f"{len(drift_claims)} claims\nOriginal copy:")
outcome_table(on_original)
print("Drifted copy, as a new claim (no earlier location):")
outcome_table(on_drifted_new)
print("Drifted copy, given the location the original run found:")
outcome_table(on_drifted_known)

In [ ]:
def drift_outcome(triage: Triage) -> str:
    """What the agents made of a claim whose anchor was deleted or renamed."""
    if triage.outcome not in {"found", "partly"}:
        return triage.outcome
    names = {l.symbol for l in triage.report.locations} | {l.quote for l in triage.report.locations}
    if any(name + "_v2" in text for name in to_rename for text in names):
        return triage.outcome + ", the renamed symbol"
    return triage.outcome + ", elsewhere"


for label, drifted in [("new claim", on_drifted_new), ("earlier location given", on_drifted_known)]:
    print(f"\noriginal -> drifted ({label})")
    for (a, b), count in Counter((a.outcome, drift_outcome(b)) for a, b in zip(on_original, drifted)).most_common():
        print(f"  {a:18} -> {b:28} {count:3}")

print("\nStill found elsewhere, given the earlier location:")
for c, b in zip(drift_claims, on_drifted_known):
    if drift_outcome(b) in {"found, elsewhere", "partly, elsewhere"}:
        locations = ", ".join(f"{l.path}:{l.line} {l.symbol}" for l in b.report.locations)
        print(f"- [{b.outcome}] {c.tie.claim.claim}\n    {locations}\n    {b.review.reason}")

### The same drift with the repo map and quote retry

The three runs again with `TRIAGE_VARIANT`.

In [ ]:
variant_original = await triage_runner_for(code_tools, "original", TRIAGE_VARIANT, repo_map).run(drift_claims)
drift_map = RepoMap(list(drift_tools._texts), drift_definitions)
variant_drifted = triage_runner_for(drift_tools, DRIFT_LABEL, TRIAGE_VARIANT, drift_map)
variant_new = await variant_drifted.run(drift_claims)
variant_known = await variant_drifted.run(drift_claims, previous=[where(t) for t in variant_original])
for label, results in [("original", variant_original), ("drifted, new claim", variant_new), ("drifted, earlier location given", variant_known)]:
    print(label)
    outcome_table(results)
    if label != "original":
        print("  still found something else:", sum(drift_outcome(t) in {"found, elsewhere", "partly, elsewhere"} for t in results))

### Rechecking the saved ties on the drifted copy, without a model

Every tie from the original run, rechecked on the drifted copy. The planted
claims are the ones resting on a deleted file or renamed symbol; any other
claim that drifts rests on one too, through a different anchor or quote.

In [ ]:
drift_checker = Rechecker(drift_tools, drift_resolver)
drift_rechecks = [drift_checker.recheck(r) for r in records]
planted_ids = {claim_id(chunks_by_id[c.tie.chunk_id]["concept"], c.tie.claim.claim) for c, _ in planted}
table = Counter((r.claim_id in planted_ids, result) for r, result in zip(records, drift_rechecks))
for planted_claim in (True, False):
    print("planted" if planted_claim else "not planted", {k[1]: v for k, v in table.items() if k[0] == planted_claim})
deleted_or_renamed = re.compile("|".join([re.escape(p) for p in to_delete] + [r"\b" + re.escape(n) + r"\b" for n in to_rename]))
unplanted_drift = [(r, x) for r, x in zip(records, drift_rechecks) if x == "drifted" and r.claim_id not in planted_ids]
explained = sum(1 for r, _ in unplanted_drift if any(deleted_or_renamed.search(e.path + " " + e.anchor + " " + e.symbol + " " + e.quote) for e in r.evidence))
print(f"of {len(unplanted_drift)} drifted claims outside the planted set, {explained} rest on a deleted file or renamed symbol")

## Planted violations of universal claims

Seven universal claims that held on the original, each broken on purpose in
a copy, the way a real change would break it: a publish step, a command that
was removed, a call to APM, a rule file without `applyTo:`, a flag flipped,
a `git push`, an import. The universal graph runs on the copy as on the
original (instances up to the threshold are checked one by one), then again
with the threshold at 0, which forces the ast-grep rule route. The saved
searches are also rerun without a model.

In [ ]:
PLANTS = [
    ("Nothing goes to PyPI", ".github/workflows/release-tag.yml", "      # verdict: raised",
     "      - name: Publish to PyPI\n        run: uv publish dist/*\n      # verdict: raised"),
    ("kmctl plugin install, update and remove are gone", "src/kmctl/__init__.py", '@plugin_app.command("patch", epilog=_PATCH_EPILOG)',
     '@plugin_app.command("install")\ndef plugin_install() -> None:\n    """Install kmctl\'s plugin for a harness."""\n\n\n@plugin_app.command("patch", epilog=_PATCH_EPILOG)'),
    ("kmctl plugin patch never runs APM", "src/kmctl/__init__.py", '    """Add what APM cannot install yet for kmctl\'s package; run it after apm install."""\n',
     '    """Add what APM cannot install yet for kmctl\'s package; run it after apm install."""\n    import subprocess\n\n    subprocess.run(["apm", "install"], check=True)\n'),
    ("Each rule is apm/claude/.apm/instructions/NAME.instructions.md with an applyTo:", "apm/claude/.apm/instructions/release-notes.instructions.md", None,
     "---\ndescription: Every release has notes.\n---\n\nRead the notes after a release.\n"),
    ("each agent gets mainAgent false", "src/kmctl/antigravity.py", '"mainAgent": False,', '"mainAgent": True,'),
    ("Nothing the agent changes is committed or pushed.", "action/agent.py", "    return [read_file, search_files, list_files, run_shell]",
     '    subprocess.run(["git", "push"], check=False)\n    return [read_file, search_files, list_files, run_shell]'),
    ("action/agent.py does not import kmctl", "action/agent.py", "import argparse\n", "import argparse\n\nimport kmctl.checks\n"),
]

planted_copy = DriftedCopy(root_path)
plant_targets = []
for claim_text, file, old, new in PLANTS:
    target = planted_copy.root / file
    if old is None:
        target.write_text(new)
    else:
        text = target.read_text()
        assert text.count(old) == 1, (file, old)
        target.write_text(text.replace(old, new))
    matches = [r for r in claim_results if r.universal is not None and r.context.tie.claim.claim.startswith(claim_text)]
    assert matches, claim_text
    plant_targets.append((matches[0], file))
planted_copy.commit()

PLANT_LABEL = "planted-" + hashlib.sha256(json.dumps(PLANTS).encode()).hexdigest()[:8]
planted_files = list(reader.read_files(planted_copy.root))
planted_tracked = TrackedPaths(planted_copy.root)
planted_definitions = DefinitionIndex(planted_copy.root, planted_files, parsers)
planted_tools = CodeTools(planted_copy.root, planted_tracked, planted_definitions, planted_files)
planted_map = RepoMap(list(planted_tools._texts), planted_definitions)
plant_contexts = [r.context for r, _ in plant_targets]
on_planted = await universal_runner_for(planted_tools, planted_map, PLANT_LABEL).run(plant_contexts)
on_planted_rule = await universal_runner_for(planted_tools, planted_map, PLANT_LABEL, threshold=0).run(plant_contexts)
plant_rechecks = [Rechecker(planted_tools, resolver).recheck(tie_record(r)) for r, _ in plant_targets]


def caught(result: UniversalResult, file: str) -> str:
    if result.verdict != "broken":
        return result.verdict
    return "broken, at the plant" if any(s.path == file for s in result.breaking) else "broken, elsewhere only"


for (r, file), planted, ruled, recheck in zip(plant_targets, on_planted, on_planted_rule, plant_rechecks):
    print(f"- [{r.universal.quantifier}] {r.universal.claim[:90]}")
    print(f"    original: {r.universal.verdict} ({r.universal.route}); planted: {caught(planted, file)} ({planted.route}, "
          f"{len(planted.sites)} instances); rule forced: {caught(ruled, file)} ({ruled.rule_rounds} rounds); saved searches: {recheck}")

## Parsing HCL

Tree-sitter on a Terraform file. Needs `HCL_SAMPLE_PATH`.

In [ ]:
parser = get_parser("hcl")

In [ ]:
CODE_FP = os.environ["HCL_SAMPLE_PATH"]  # any .tf file

In [ ]:
b = plb.Path(CODE_FP).read_bytes()

In [ ]:
tree = parser.parse(b)

In [ ]:
def print_tree(node, indent=0):
    # Print the node type and its line numbers
    print("  " * indent + f"{node.type} [{node.start_point[0]}:{node.start_point[1]}]")

    # If it's a leaf node (has no children), print the actual code text
    if len(node.children) == 0:
        print("  " * (indent + 1) + f"-> {node.text.decode('utf-8')}")

    for child in node.children:
        print_tree(child, indent + 1)

In [ ]:
print_tree(tree.root_node)

In [ ]:
m = Magika()

mr = m.identify_bytes(b)

label = mr.prediction.dl.label

In [ ]:
parser = get_parser(label)

In [ ]:
mr